In [1]:
ONLY_INFER = True # True # False

if ONLY_INFER:
    # --- Early-exit guard for local runs (put this as the very first cell) ---
    import os
    import sys
    import pandas as pd
    
    flag = os.environ.get("KAGGLE_IS_COMPETITION_RERUN")
    
    # Kaggle sets this to a truthy value during the submission rerun environment.
    # If not in that environment, write an all-zeros submission and stop.
    if not flag:
        # Try to derive the correct row count / IDs from an available sequences file.
        # (adjust paths if your competition uses different names)
        candidate_seq_paths = [
            "/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv",
            "/kaggle/input/stanford-rna-3d-folding-2/validation_sequences.csv",
            "/kaggle/input/stanford-rna-3d-folding-2/train_sequences.csv",
        ]
    
        seq_df = None
        for p in candidate_seq_paths:
            if os.path.exists(p):
                seq_df = pd.read_csv(p)
                break
    
        if seq_df is None or "target_id" not in seq_df.columns or "sequence" not in seq_df.columns:
            raise RuntimeError(
                "Not in competition rerun, and could not find a sequences file to build a valid zero submission.\n"
                "Make sure test_sequences.csv is available under /kaggle/input/... or adjust candidate_seq_paths."
            )
    
        rows = []
        for row in seq_df.itertuples(index=False):
            tid = row.target_id
            seq = row.sequence
            for i, base in enumerate(seq, start=1):
                rec = {
                    "ID": f"{tid}_{i}",
                    "resname": base,
                    "resid": i,
                }
                for k in range(1, 6):
                    rec[f"x_{k}"] = 0.0
                    rec[f"y_{k}"] = 0.0
                    rec[f"z_{k}"] = 0.0
                rows.append(rec)
    
        out = pd.DataFrame(rows)
    
        # Enforce official column order
        cols = (
            ["ID", "resname", "resid"] +
            [f"{ax}_{k}" for k in range(1, 6) for ax in ("x", "y", "z")]
        )
        out = out[cols]
        out.to_csv("submission.csv", index=False)
        print("Not in Kaggle competition rerun. Wrote zero submission.csv and exiting early.")
        raise SystemExit(0)
    
    # If flag is truthy: do nothing, notebook continues normally.
    print("KAGGLE_IS_COMPETITION_RERUN detected -> continuing normally.")


Not in Kaggle competition rerun. Wrote zero submission.csv and exiting early.


SystemExit: 0

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# trRosettaRNA -> ×

# Proteinix

In [ ]:
import os

IS_SCORING_RUN = os.environ.get('KAGGLE_IS_COMPETITION_RERUN')
print(IS_SCORING_RUN)

In [ ]:
!pip install --no-deps /kaggle/input/datasets/tobimichigan/biotite-1-2/biotite-1.2.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

In [ ]:
# !cp -r /kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1 /kaggle/working/Protenix-v1

In [ ]:
# !cp -r /kaggle/input/stanford-rna-3d-folding-2/PDB_RNA /kaggle/working/Protenix-v1/PDB_RNA

In [ ]:
import gc
import json
import os

os.environ["LAYERNORM_TYPE"] = "torch"
os.environ.setdefault("RNA_MSA_DEPTH_LIMIT", "512")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# ---- user-configurable paths ----
DEFAULT_TEST_CSV = "/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv"
DEFAULT_OUTPUT = "/kaggle/working/protenix_submission.csv"
DEFAULT_CODE_DIR = "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1"# "/kaggle/input/protenix-v1/Protenix-v1"
DEFAULT_ROOT_DIR = "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1"# "/kaggle/input/protenix-v1/Protenix-v1"
# MODEL_NAME = "protenix_base_default_v1.0.0"
MODEL_NAME = "protenix_base_20250630_v1.0.0"
input_path = "/kaggle/input/stanford-rna-3d-folding-2"
N_SAMPLE = 5
SEED = 42

MAX_SEQ_LEN = int(os.environ.get("MAX_SEQ_LEN", "512"))
CHUNK_OVERLAP = int(os.environ.get("CHUNK_OVERLAP", "128")) # 64
MODEL_N_SAMPLE = int(os.environ.get("MODEL_N_SAMPLE", str(N_SAMPLE)))


#如果 TEST_TARGETS 为空，就按全量运行。

# TEST_TARGETS = os.environ.get("TEST_TARGETS", "9MME,9ZCC").strip()
# TEST_TARGETS = os.environ.get("TEST_TARGETS", "8ZNQ").strip()
TEST_TARGETS = False
if not IS_SCORING_RUN:
    TEST_TARGETS = os.environ.get("TEST_TARGETS", "9ZCC").strip()

def parse_bool(value: str, default: bool = False) -> str:
    normalized = str(value).strip().lower()
    if normalized in {"1", "true", "t", "yes", "y", "on"}:
        return "true"
    if normalized in {"0", "false", "f", "no", "n", "off"}:
        return "false"
    return "true" if default else "false"

# os.environ["USE_MSA"] = "true" # コメントアウトした方が良い可能性あり
# os.environ["USE_TEMPLATE"] = "true"
os.environ["USE_RNA_MSA"] = "true"

USE_MSA = parse_bool(os.environ.get("USE_MSA", "false"))
USE_TEMPLATE = parse_bool(os.environ.get("USE_TEMPLATE", "false"))
USE_RNA_MSA = parse_bool(os.environ.get("USE_RNA_MSA", "false"))
print(f"USE_MSA: {USE_MSA}")
print(f"USE_TEMPLATE: {USE_TEMPLATE}")
print(f"USE_RNA_MSA: {USE_RNA_MSA}")

TRIANGLE_ATTENTION = os.environ.get("TRIANGLE_ATTENTION", "torch").strip().lower()
TRIANGLE_MULTIPLICATIVE = os.environ.get("TRIANGLE_MULTIPLICATIVE", "torch").strip().lower()

def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = True
    torch.use_deterministic_algorithms(True)


def resolve_paths() -> tuple[str, str, str]:
    test_csv = os.environ.get("TEST_CSV", DEFAULT_TEST_CSV)
    output_csv = os.environ.get("SUBMISSION_CSV", DEFAULT_OUTPUT)
    code_dir = os.environ.get("PROTENIX_CODE_DIR", DEFAULT_CODE_DIR)
    root_dir = os.environ.get("PROTENIX_ROOT_DIR", DEFAULT_ROOT_DIR)
    return test_csv, output_csv, code_dir, root_dir


def ensure_required_files(root_dir: str) -> None:
    checkpoint_path = Path(root_dir) / "checkpoint" / f"{MODEL_NAME}.pt"
    components_path = Path(root_dir) / "common" / "components.cif"
    components_pkl_path = Path(root_dir) / "common" / "components.cif.rdkit_mol.pkl"
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {checkpoint_path}.\n"
            "Please include checkpoint/protenix_base_default_v1.0.0.pt in PROTENIX_ROOT_DIR."
        )
    if not components_path.exists():
        raise FileNotFoundError(
            f"Missing CCD file: {components_path}.\n"
            "Please include common/components.cif in PROTENIX_ROOT_DIR."
        )
    if not components_pkl_path.exists():
        raise FileNotFoundError(
            f"Missing CCD cache: {components_pkl_path}.\n"
            "Please include common/components.cif.rdkit_mol.pkl in PROTENIX_ROOT_DIR."
        )


def build_input_json(test_df: pd.DataFrame, json_path: str) -> None:
    data = []
    for _, row in test_df.iterrows():
        seq = row["sequence"]
        target_id = row["target_id"]
        data.append(
            {
                "name": target_id,
                "covalent_bonds": [],
                "sequences": [
                    {
                        "rnaSequence": {
                            "sequence": seq,
                            "count": 1,
                            # "msa": {
                            #     "precomputed_msa_dir": f"{input_path}/MSA/{target_id}.MSA.fasta",
                            #     "pairing_db": "rnacentral"
                            # }
                        }
                    }
                ],
            }
        )
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f)


def build_single_input_json(target_id: str, seq: str, json_path: str) -> None:
    data = [
        {
            "name": target_id,
            "covalent_bonds": [],
            "sequences": [
                {
                    "rnaSequence": {
                        "sequence": seq,
                        "count": 1,
                        # "msa": {
                        #     "precomputed_msa_dir": f"{input_path}/MSA/{target_id}.MSA.fasta",
                        #     "pairing_db": "rnacentral"
                        # }
                    }
                }
            ],
        }
    ]
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f)


def build_configs(input_json_path: str, dump_dir: str, model_name: str):
    from configs.configs_base import configs as configs_base
    from configs.configs_data import data_configs
    from configs.configs_inference import inference_configs
    from configs.configs_model_type import model_configs
    from protenix.config.config import parse_configs

    base_configs = {**configs_base, **{"data": data_configs}, **inference_configs}

    def deep_update(target, patch):
        for key, value in patch.items():
            if isinstance(value, dict) and key in target and isinstance(target[key], dict):
                deep_update(target[key], value)
            else:
                target[key] = value

    deep_update(base_configs, model_configs[model_name])

    arg_str = " ".join(
        [
            f"--model_name {model_name}",
            f"--input_json_path {input_json_path}",
            f"--dump_dir {dump_dir}",
            f"--use_msa {USE_MSA}",
            f"--use_template {USE_TEMPLATE}",
            f"--use_rna_msa {USE_RNA_MSA}",
            f"--sample_diffusion.N_sample {MODEL_N_SAMPLE}",
            f"--seeds {SEED}",
        ]
    )
    configs = parse_configs(
        configs=base_configs,
        arg_str=arg_str,
        fill_required_with_null=True,
    )
    return configs


def get_c1_mask(data: dict, atom_array) -> torch.Tensor:
    if atom_array is not None:
        try:
            if hasattr(atom_array, "centre_atom_mask"):
                centre_mask = atom_array.centre_atom_mask == 1
                if hasattr(atom_array, "is_rna"):
                    centre_mask = centre_mask & (atom_array.is_rna)
                return torch.from_numpy(centre_mask)
            if hasattr(atom_array, "atom_name"):
                if hasattr(atom_array, "is_rna"):
                    return torch.from_numpy(
                        (atom_array.atom_name == "C1'") & (atom_array.is_rna)
                    )
                return torch.from_numpy(atom_array.atom_name == "C1'")
        except Exception:
            pass

    features = data["input_feature_dict"]
    if "centre_atom_mask" in features:
        return features["centre_atom_mask"].long() == 1
    return features["atom_to_tokatom_idx"].long() == 12


def get_feature_c1_mask(data: dict) -> torch.Tensor:
    features = data["input_feature_dict"]
    if "centre_atom_mask" in features:
        return features["centre_atom_mask"].long() == 1
    return features["atom_to_tokatom_idx"].long() == 12


def coords_to_rows(target_id: str, seq: str, coords: np.ndarray) -> list[dict]:
    rows = []
    n_res = len(seq)
    for i in range(n_res):
        row = {
            "ID": f"{target_id}_{i + 1}",
            "resname": seq[i],
            "resid": i + 1,
        }
        for s in range(N_SAMPLE):
            if s < coords.shape[0] and i < coords.shape[1]:
                x, y, z = coords[s, i]
            else:
                x, y, z = 0.0, 0.0, 0.0
            row[f"x_{s + 1}"] = float(x)
            row[f"y_{s + 1}"] = float(y)
            row[f"z_{s + 1}"] = float(z)
        rows.append(row)
    return rows


def pad_samples(coords: np.ndarray, target_samples: int) -> np.ndarray:
    if coords.shape[0] >= target_samples:
        return coords
    if coords.shape[0] == 0:
        return np.zeros((target_samples, coords.shape[1], 3), dtype=coords.dtype)
    repeat = target_samples - coords.shape[0]
    extra = np.repeat(coords[:1], repeat, axis=0)
    return np.concatenate([coords, extra], axis=0)


def split_sequence(seq: str, max_len: int, overlap: int) -> list[tuple[int, int, str]]:
    if max_len <= 0:
        raise ValueError("max_len must be > 0")
    if overlap >= max_len:
        raise ValueError("overlap must be smaller than max_len")
    if len(seq) <= max_len:
        return [(0, len(seq), seq)]
    step = max_len - overlap
    chunks = []
    start = 0
    while start < len(seq):
        end = min(start + max_len, len(seq))
        chunks.append((start, end, seq[start:end]))
        if end == len(seq):
            break
        start += step
    return chunks

def main() -> None:
    test_csv, output_csv, code_dir, root_dir = resolve_paths()

    if not os.path.isdir(code_dir):
        raise FileNotFoundError(
            f"Missing PROTENIX_CODE_DIR: {code_dir}. Set PROTENIX_CODE_DIR to the repo path."
        )

    os.environ["PROTENIX_ROOT_DIR"] = root_dir
    sys.path.append(code_dir)

    ensure_required_files(root_dir)
    seed_everything(SEED)

    test_df = pd.read_csv(test_csv)
    test_df["sequence_len"] = test_df["sequence"].str.len()
    # test_df = test_df[test_df['sequence_len'] < 250]
    test_df = test_df[(test_df['sequence_len'] < 250) | (test_df["sequence_len"] >= 1000)]

    # print("MAX_SEQ_LEN =", MAX_SEQ_LEN)
    # print("Sequence length =", test_df["sequence_len"].max())
    
    if not IS_SCORING_RUN:
    #     test_df = test_df.head(5)
    
        if TEST_TARGETS:
            keep = {t.strip() for t in TEST_TARGETS.split(",") if t.strip()}
            test_df = test_df[test_df["target_id"].isin(keep)].reset_index(drop=True)
            if test_df.empty:
                raise ValueError("TEST_TARGETS did not match any target_id values")

    work_dir = Path("/kaggle/working")
    
    work_dir.mkdir(parents=True, exist_ok=True)
    input_json_path = str(work_dir / "protenix_input.json")
    build_input_json(test_df, input_json_path)

    chunks_dir = work_dir / "chunks"
    chunks_dir.mkdir(parents=True, exist_ok=True)

    from protenix.data.inference.infer_dataloader import InferenceDataset
    from runner.inference import InferenceRunner, update_gpu_compatible_configs, update_inference_configs

    configs = build_configs(input_json_path, str(work_dir / "outputs"), MODEL_NAME)
    configs = update_gpu_compatible_configs(configs)
    runner = InferenceRunner(configs)

    # dataset = InferenceDataset(configs)
    seq_by_id = dict(zip(test_df.target_id.tolist(), test_df.sequence.tolist()))

    all_predictions = []

    debug_path = Path(output_csv).with_suffix(".debug.csv")
    debug_rows = []

    # for i in tqdm(range(len(dataset)), total=len(dataset)):
    #     data, atom_array, error_message = dataset[i]
        # print(atom_array.res_id[:50])
        # print(atom_array.atom_name[:50])
        # target_id = data.get("sample_name", f"sample_{i}")
        # seq = seq_by_id.get(target_id, "")
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        target_id = row["target_id"]
        seq = row["sequence"]
        # data, atom_array, error_message = dataset[i]
        
        if len(seq) > MAX_SEQ_LEN:
            chunks = split_sequence(seq, MAX_SEQ_LEN, CHUNK_OVERLAP)
            print(
                f"[{target_id}] long sequence: {len(seq)} -> {len(chunks)} chunks "
                f"(max_len={MAX_SEQ_LEN}, overlap={CHUNK_OVERLAP})"
            )
            full_coords = np.zeros((N_SAMPLE, len(seq), 3), dtype=np.float32)
            for chunk_idx, (start, end, subseq) in enumerate(chunks):
                chunk_id = f"{target_id}__chunk{chunk_idx:03d}"
                chunk_json_path = str(chunks_dir / f"{chunk_id}.json")
                build_single_input_json(chunk_id, subseq, chunk_json_path)
                chunk_configs = build_configs(
                    chunk_json_path, str(work_dir / "outputs"), MODEL_NAME
                )
                chunk_configs = update_gpu_compatible_configs(chunk_configs)
                chunk_dataset = InferenceDataset(chunk_configs)
                chunk_data, chunk_atom_array, chunk_error = chunk_dataset[0]

                if chunk_error:
                    print(f"[{chunk_id}] data_error: {chunk_error.splitlines()[0]}")
                    debug_rows.append(
                        {
                            "target_id": target_id,
                            "seq_len": len(seq),
                            "c1_count": 0,
                            "is_rna_atoms": -1,
                            "coord_abs_max": 0.0,
                            "error": chunk_error,
                            "split_mode": "chunked",
                            "chunk_idx": chunk_idx,
                            "chunk_start": start,
                            "chunk_end": end,
                            "chunk_len": end - start,
                        }
                    )
                    continue

                new_configs = update_inference_configs(
                    configs, chunk_data["N_token"].item()
                )
                runner.update_model_configs(new_configs)

                prediction = runner.predict(chunk_data)
                coords = prediction["coordinate"]
                mask = get_c1_mask(chunk_data, chunk_atom_array)
                c1_count = int(mask.sum().item()) if hasattr(mask, "sum") else 0
                if c1_count == 0:
                    mask = get_feature_c1_mask(chunk_data)
                    c1_count = int(mask.sum().item()) if hasattr(mask, "sum") else 0
                    print(
                        f"[{chunk_id}] C1' atoms: {c1_count} (seq_len={len(subseq)}) fallback"
                    )
                else:
                    print(
                        f"[{chunk_id}] C1' atoms: {c1_count} (seq_len={len(subseq)})"
                    )
                is_rna_count = (
                    int(chunk_atom_array.is_rna.sum())
                    if hasattr(chunk_atom_array, "is_rna")
                    else -1
                )
                coord_abs_max = float(coords.abs().max().item())
                debug_rows.append(
                    {
                        "target_id": target_id,
                        "seq_len": len(seq),
                        "c1_count": c1_count,
                        "is_rna_atoms": is_rna_count,
                        "coord_abs_max": coord_abs_max,
                        "error": "",
                        "split_mode": "chunked",
                        "chunk_idx": chunk_idx,
                        "chunk_start": start,
                        "chunk_end": end,
                        "chunk_len": end - start,
                    }
                )
                # coords = coords[:, mask, :].detach().cpu().numpy()
                coords_all = prediction["coordinate"]  # (S, N_atom, 3)
                # print("coords_all shape:", coords_all.shape)
                # print("sample0 atom0-10 x:", coords_all[0, :10, 0])
                # print("sample0 atom20-30 x:", coords_all[0, 20:30, 0])

                res_ids = chunk_atom_array.res_id
                atom_names = chunk_atom_array.atom_name
                
                c1_indices = []
                for r in range(1, len(subseq) + 1):
                    idx = np.where(
                        (res_ids == r) &
                        (atom_names == "C1'")
                    )[0]
                    if len(idx) == 0:
                        continue
                    c1_indices.append(idx[0])
                
                coords = coords_all[:, c1_indices, :]
                coords = coords.detach().cpu().numpy()

                if coords.shape[1] != len(subseq):
                    padded = np.zeros((coords.shape[0], len(subseq), 3), dtype=np.float32)
                    min_len = min(coords.shape[1], len(subseq))
                    if min_len > 0:
                        padded[:, :min_len, :] = coords[:, :min_len, :]
                    coords = padded

                coords = pad_samples(coords, N_SAMPLE)

                full_coords[:, start:end, :] = coords[:, : end - start, :]
                del prediction, coords, mask, chunk_data, chunk_atom_array, chunk_dataset
                torch.cuda.empty_cache()
                gc.collect()

            rows = coords_to_rows(target_id, seq, full_coords)
            all_predictions.extend(rows)
            continue

        elif len(seq) <= MAX_SEQ_LEN:
            single_json = str(work_dir / f"{target_id}.json")
            build_single_input_json(target_id, seq, single_json)
        
            single_configs = build_configs(
                single_json,
                str(work_dir / "outputs"),
                MODEL_NAME,
            )
            single_configs = update_gpu_compatible_configs(single_configs)
        
            dataset = InferenceDataset(single_configs)
            data, atom_array, error_message = dataset[0]
            if error_message:
                print(f"[{target_id}] data_error: {error_message.splitlines()[0]}")
                debug_rows.append(
                    {
                        "target_id": target_id,
                        "seq_len": len(seq),
                        "c1_count": 0,
                        "is_rna_atoms": -1,
                        "coord_abs_max": 0.0,
                        "error": error_message,
                        "split_mode": "full",
                        "chunk_idx": -1,
                        "chunk_start": 0,
                        "chunk_end": len(seq),
                        "chunk_len": len(seq),
                    }
                )
                rows = coords_to_rows(
                    target_id,
                    seq,
                    np.zeros((0, 0, 3), dtype=np.float32),
                )
                all_predictions.extend(rows)
                continue
    
            new_configs = update_inference_configs(configs, data["N_token"].item())
            runner.update_model_configs(new_configs)
    
            prediction = runner.predict(data)
            coords = prediction["coordinate"]
            mask = get_c1_mask(data, atom_array)
            c1_count = int(mask.sum().item()) if hasattr(mask, "sum") else 0
            if c1_count == 0:
                mask = get_feature_c1_mask(data)
                c1_count = int(mask.sum().item()) if hasattr(mask, "sum") else 0
                print(
                    f"[{target_id}] C1' atoms: {c1_count} (seq_len={len(seq)}) fallback"
                )
            else:
                print(f"[{target_id}] C1' atoms: {c1_count} (seq_len={len(seq)})")
            is_rna_count = (
                int(atom_array.is_rna.sum()) if hasattr(atom_array, "is_rna") else -1
            )
            coord_abs_max = float(coords.abs().max().item())
            debug_rows.append(
                {
                    "target_id": target_id,
                    "seq_len": len(seq),
                    "c1_count": c1_count,
                    "is_rna_atoms": is_rna_count,
                    "coord_abs_max": coord_abs_max,
                    "error": "",
                    "split_mode": "full",
                    "chunk_idx": -1,
                    "chunk_start": 0,
                    "chunk_end": len(seq),
                    "chunk_len": len(seq),
                }
            )
            
            # print(f"{target_id} mask sum:", mask.sum().item(), "seq_len:", len(seq))
            
            # coords = coords[:, mask, :].detach().cpu().numpy()
            coords_all = prediction["coordinate"]  # (S, N_atom, 3)
            # print("coords_all shape:", coords_all.shape)
            # print("sample0 atom0-10 x:", coords_all[0, :10, 0])
            # print("sample0 atom20-30 x:", coords_all[0, 20:30, 0])
    
            res_ids = atom_array.res_id
            atom_names = atom_array.atom_name
            
            c1_indices = []
            for r in range(1, len(seq) + 1):
                idx = np.where(
                    (res_ids == r) &
                    (atom_names == "C1'")
                )[0]
                if len(idx) == 0:
                    continue
                c1_indices.append(idx[0])
            
            coords = coords_all[:, c1_indices, :]
            coords = coords.detach().cpu().numpy()
    
            if coords.shape[1] != len(seq):
                padded = np.zeros((coords.shape[0], len(seq), 3), dtype=np.float32)
                min_len = min(coords.shape[1], len(seq))
                if min_len > 0:
                    padded[:, :min_len, :] = coords[:, :min_len, :]
                coords = padded
    
            coords = pad_samples(coords, N_SAMPLE)
            # print(coords.shape)
            # print(np.abs(coords[0] - coords[1]).max())
    
            rows = coords_to_rows(target_id, seq, coords)
            all_predictions.extend(rows)
            del prediction, coords, mask, data, atom_array
            torch.cuda.empty_cache()
            gc.collect()

    if debug_rows:
        pd.DataFrame(debug_rows).to_csv(debug_path, index=False)

    sub = pd.DataFrame(all_predictions)
    cols = ["ID", "resname", "resid"] + [
        f"{c}_{i}" for i in range(1, N_SAMPLE + 1) for c in ["x", "y", "z"]
    ]
    coord_cols = [c for c in cols if c.startswith(("x_", "y_", "z_"))]
    sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)
    sub[cols].to_csv(output_csv, index=False)

    print(f"Saved submission to {output_csv}")


if __name__ == "__main__":
    main()

In [ ]:
protenix_pred = pd.read_csv('/kaggle/working/protenix_submission.csv')
protenix_pred

In [ ]:
# # import os
# # import random
# # import pandas as pd
# # import numpy as np
# # import torch
# # from tqdm import tqdm
# # import warnings
# # warnings.filterwarnings("ignore")  

# # #time0=time.time()

# # print('IMPORT OK !!!!')

# if os.path.exists('/kaggle/input/stanford-rna-3d-folding-2'):
#     # run on kaggle
#     TEST_SEQUENCES_PATH = '/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv'
#     DATA_KAGGLE_DIR = '/kaggle/input/stanford-rna-3d-folding-2'
#     # CHECKPOINT_PATH = '/kaggle/input/datasets/geraseva/protenix-checkpoints/model_v0.2.0.pt'
#     CHECKPOINT_PATH = '/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1/checkpoint/protenix_base_default_v1.0.0.pt'
#     # os.environ["PROTENIX_DATA_ROOT_DIR"] = '/kaggle/input/datasets/geraseva/protenix-checkpoints'
#     os.environ["PROTENIX_DATA_ROOT_DIR"] = '/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1/checkpoint'
# else:
#     # run locally
#     DATA_KAGGLE_DIR = '.' 
#     CHECKPOINT_PATH = '/protenix/release_data/checkpoint/model_v0.2.0.pt'
#     os.environ["PROTENIX_DATA_ROOT_DIR"] = '/protenix/release_data/ccd_cache'

# # from runner.inference import update_inference_configs, InferenceRunner   
# # from protenix.data.infer_data_pipeline import InferenceDataset
# # from configs.configs_base import configs as configs_base
# # from configs.configs_data import data_configs
# # from configs.configs_inference import inference_configs
# # from protenix.config.config import parse_configs


# def parse_output_to_df(output, seq, target_id):
#     df = []
#     chain_data = []
#     for i, res in enumerate(seq):
#         d={"ID": target_id, "resname": res, "resid": i + 1}
#         for n in range(len(output)):
#             d={**d, f'x_{n+1}': round(output[n,i,0].item(),3),
#                      f'y_{n+1}': round(output[n,i,1].item(),3),
#                      f'z_{n+1}': round(output[n,i,2].item(),3)}
#         chain_data.append(d)

#     if len(chain_data)!=0:
#         chain_df = pd.DataFrame(chain_data)
#         df.append(chain_df)
#         ##print(chain_df)
#     return df

# def seed_everything(seed=0):
#     np.random.seed(seed)
#     torch.random.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     os.environ["PYTHONHASHSEED"] = str(seed)
#     random.seed(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False
#     print(f"Random seed set to {seed}")

# seed_everything()

# class DictDataset(InferenceDataset):
#     def __init__(
#         self,
#         seq_list: list,
#         dump_dir: str,
#         id_list: list = None,
#         use_msa: bool = False,
#     ) -> None:

#         self.dump_dir = dump_dir
#         self.use_msa = use_msa
#         if isinstance(id_list,type(None)):
#             self.inputs = [{"sequences": 
#                             [{"rnaSequence": 
#                                 {"sequence": seq, 
#                                 "count": 1}}],
#                             "name": "query"} for seq in seq_list]
#         else:
#             self.inputs = [{"sequences": 
#                             [{"rnaSequence": 
#                                 {"sequence": seq, 
#                                 "count": 1}}],
#                             "name": i} for i, seq in zip(id_list,seq_list)]



# configs_base["use_deepspeed_evo_attention"] = (
# os.environ.get("USE_DEEPSPEED_EVO_ATTENTION", False) == "true")
# configs_base["model"]["N_cycle"] = 10 #10
# configs_base["sample_diffusion"]["N_sample"] = 5
# configs_base["sample_diffusion"]["N_step"] = 200
# inference_configs['load_checkpoint_path']=CHECKPOINT_PATH
# configs = {**configs_base, **{"data": data_configs}, **inference_configs}

# configs = parse_configs(
#         configs=configs,
#         fill_required_with_null=True,
#     )
# #print('CONFIGS:')
# #print(configs)
# #print('DATA CONFIGS:')
# #print(data_configs)
# runner=InferenceRunner(configs)


# test_df=pd.read_csv(TEST_SEQUENCES_PATH)
# test_df["sequence_len"] = test_df["sequence"].str.len()
# test_df = test_df[(test_df['sequence_len'] >= 200) & (test_df['sequence_len'] < 600)]
# test_df.reset_index(drop=True, inplace=True)
# print(test_df.shape)

# dataset = DictDataset(test_df.sequence, dump_dir='output', id_list=test_df.target_id, use_msa=False)
# num_data = len(dataset)
# # results = pd.DataFrame()
# for i, seq in tqdm(enumerate(test_df.sequence),total=num_data):
#     try:
#         data, atom_array, data_error_message=dataset[i]
#         target_id = data["sample_name"]
#         assert target_id==test_df.target_id[i]
#         assert data_error_message==''
        
#         new_configs = update_inference_configs(configs, data["N_token"].item())
#         runner.update_model_configs(new_configs)
#         prediction = runner.predict(data)
#         prediction=prediction['coordinate'][:,data['input_feature_dict']['atom_to_tokatom_idx']==12]

#         result = parse_output_to_df(prediction, seq, target_id)[0]
#     except Exception as exc:
#         print(exc)
#         print(f"Error message: {data_error_message}")        
#         print('Failed to predict', target_id)
#         result=pd.DataFrame(columns=['ID', 'resname', 'resid', 
#                                         'x_1', 'y_1', 'z_1', 
#                                         'x_2', 'y_2', 'z_2',
#                                         'x_3', 'y_3', 'z_3', 
#                                         'x_4', 'y_4', 'z_4', 
#                                         'x_5', 'y_5', 'z_5'], 
#                                         data=[[target_id, x, j+1] + [0.0]*15 for j, x in enumerate(seq)])
        
#     result['ID'] = result.apply(lambda x: x.ID + '_' + str(x.resid), axis=1)
#     if i == 0:
#         results = result.copy()
#     if i != 0:
#         results = pd.concat([results, result], axis=0)
#     torch.cuda.empty_cache()

# results.to_csv('protenix_submission.csv', index=False, mode='a', header=(i==0))

# # print(pd.read_csv('protenix_submission.csv'))

In [ ]:
# protenix_pred = results # pd.read_csv('protenix_submission.csv')

In [ ]:
# protenix_pred

# DRFold2

In [ ]:
import os

IS_SCORING_RUN = os.environ.get('KAGGLE_IS_COMPETITION_RERUN')
print(IS_SCORING_RUN)

In [ ]:
import time
import sys
import torch
import random
import shutil
import numpy as np
import pandas as pd

from Bio.Seq import Seq
from Bio import pairwise2

from tqdm import tqdm
from scipy.spatial import distance_matrix
from scipy.spatial.transform import Rotation as R

import warnings
warnings.filterwarnings('ignore')


test_sequences = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")

is_submission_mode = len(test_sequences) != 12

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

sys.argv = ['notebook', 'cuda' if torch.cuda.is_available() else 'cpu', 'fp32'] #'fp16' #'fp32'# 'bf16' #
device = sys.argv[1]
sys_dtype = sys.argv[2] if len(sys.argv) > 2 else 'fp32'

print('Using device:', device)
print('Using dtype:', sys_dtype)

# dr settings
NUM_CONF=5
MAX_LENGTH=480
MAX_CAT_LENGTH=2400

CFG_DIR='cfg_97'
CFG_MERGE=False
DR_SCORE=False
NO_SORT=False
GET_CENTER=True

FULL_ENERGY=False

OPTIM_LENGTH=0

DEVICE=device #'cuda' #'cpu'#
PREC=sys_dtype 

In [ ]:
if PREC=='fp16':
    torch.set_default_dtype(torch.float16)
if PREC=='bf16':
    torch.set_default_dtype(torch.bfloat16)

In [ ]:
from datetime import datetime
import pytz
print('LOGGING TIME OF START:',  datetime.strftime(datetime.now(pytz.timezone('Asia/Singapore')), "%Y-%m-%d %H:%M:%S"))


print('PIP INSTALL OK !!!!')
import os,sys

import pandas as pd
pd.set_option('display.max_columns', 20)
pd.set_option('display.expand_frame_repr', False)

import numpy as np
import torch
import torch.nn.functional as F
from timeit import default_timer as timer



# helper--
class dotdict(dict):
	__setattr__ = dict.__setitem__
	__delattr__ = dict.__delitem__

	def __getattr__(self, name):
		try:
			return self[name]
		except KeyError:
			raise AttributeError(name)

def time_to_str(t, mode='min'):
	if mode=='min':
		t  = int(t)/60
		hr = t//60
		min = t%60
		return '%2d hr %02d min'%(hr,min) 
	elif mode=='sec':
		t   = int(t)
		min = t//60
		sec = t%60
		return '%2d min %02d sec'%(min,sec)

	else:
		raise NotImplementedError

def gpu_memory_use():
    if torch.cuda.is_available():
        device = torch.device(0)
        free, total = torch.cuda.mem_get_info(device)
        used= (total - free) / 1024 ** 3
        return round(used,2)
    else:
        return 0

def set_aspect_equal(ax):
	x_limits = ax.get_xlim()
	y_limits = ax.get_ylim()
	z_limits = ax.get_zlim()

	# Compute the mean of each axis
	x_middle = np.mean(x_limits)
	y_middle = np.mean(y_limits)
	z_middle = np.mean(z_limits)

	# Compute the max range across all axes
	max_range = max(x_limits[1] - x_limits[0],
					y_limits[1] - y_limits[0],
					z_limits[1] - z_limits[0]) / 2.0

	# Set the new limits to ensure equal scaling
	ax.set_xlim(x_middle - max_range, x_middle + max_range)
	ax.set_ylim(y_middle - max_range, y_middle + max_range)
	ax.set_zlim(z_middle - max_range, z_middle + max_range)


print('torch',torch.__version__)
print('torch.cuda',torch.version.cuda)

print('IMPORT OK!!!')
MODE = 'submit' #'local' # submit

DATA_KAGGLE_DIR = '/kaggle/input/stanford-rna-3d-folding-2'

if MODE == 'local':
    valid_df = pd.read_csv(f'{DATA_KAGGLE_DIR}/validation_sequences.csv')
    label_df = pd.read_csv(f'{DATA_KAGGLE_DIR}/validation_labels.csv')
    label_df['target_id'] = label_df['ID'].apply(lambda x: '_'.join(x.split('_')[:-1]))

if MODE == 'submit':
    valid_df = pd.read_csv(f'{DATA_KAGGLE_DIR}/test_sequences.csv')
    # Sort test sequences by length to process shorter ones with DRfold2
    valid_df["sequence_len"] = valid_df["sequence"].str.len()
    # test_sequences = test_sequences[(test_sequences['sequence_len'] >= 200) & (test_sequences['sequence_len'] < 600)]
    # valid_df = valid_df[valid_df['sequence_len'] < 100]
    valid_df = valid_df[valid_df['sequence_len'] < 250]
    # valid_df = valid_df[valid_df['sequence_len'] < 350]
    valid_df.reset_index(drop=True, inplace=True)
    print(valid_df.shape)
 
    if not IS_SCORING_RUN:
        valid_df = valid_df.head(5)

print('len(valid_df)',len(valid_df))
print(valid_df.iloc[0])
print('')


print('MODE:', MODE)
print('SETTING OK!!!')

In [ ]:
sys.path.append('/kaggle/input/datasets/z1493916656/drfold-model-bf16/PotentialFold')
import a2b
def frame_coor_to_C1(coor, seq, BASE_COOR, OTHER_COOR):
    tx = torch.as_tensor(coor, dtype=torch.float32)
    basex = torch.from_numpy(Get_base(seq, BASE_COOR)).to(tx.dtype)
    otherx = torch.from_numpy(Get_base(seq, OTHER_COOR)).to(tx.dtype)
    L = len(seq)

    x = torch.rand((L, 21), dtype=tx.dtype, device=tx.device)

    biasq = tx.mean(dim=1, keepdim=True)            # (L, 1, 3)
    q = tx - biasq                                  # (L, 3, 3)

    m = torch.einsum('bnz,bny->bzy', basex, q).reshape(L, -1)
    x[:, :9] = m
    x[:, 9:18] = m

    x[:, 18:] = biasq.squeeze(1)
    rama = x.double()  

    #    otherx: (L, 5, 3) -> other_xyz: (L, 5, 3)
    other_xyz = a2b.quat2b(otherx.double(), rama[:, 9:]).float().cpu().numpy()

    c1_xyz = other_xyz[:, 4, :]
    return c1_xyz


def concat_coor(out1: dict, out2: dict) -> np.ndarray:
    coor1 = torch.as_tensor(out1['coor'], dtype=torch.float64)   # (L1,3,3)
    coor2 = torch.as_tensor(out2['coor'], dtype=torch.float64)   # (L2,3,3)

    f1 = coor1[-1]   # (3,3)
    f2 = coor2[0]    # (3,3)

    bias1 = f1.mean(dim=0)   # (3,)
    bias2 = f2.mean(dim=0)   # (3,)
    basex = f1 - bias1       # (3,3)
    q     = f2 - bias2       # (3,3)

    #    R_{ij} = sum_z basex_{iz} * q_{jz}
    R = torch.einsum('iz,jz->ij', basex, q)   # (3,3)

    t = bias1 - (R @ bias2)                  # (3,)

    L2 = coor2.shape[0]
    rama = torch.empty((L2, 12), dtype=torch.float64, device=coor2.device)
    R_flat = R.reshape(1, 9).repeat(L2, 1)    # (L2,9)
    t_rep  = t.reshape(1, 3).repeat(L2, 1)    # (L2,3)
    rama[:, :9] = R_flat
    rama[:, 9:] = t_rep

    coor2_aligned = a2b.quat2b(coor2, rama)   # torch.Tensor (L2,3,3)

    coor_cat = torch.cat([coor1, coor2_aligned[1:]], dim=0)  # (L1+L2-1,3,3)

    return coor_cat.cpu().numpy()

In [ ]:
def Get_base(seq, basenpy_standard):
    n_atoms = basenpy_standard.shape[1]
    basenpy = np.zeros([len(seq), n_atoms, 3])
    seqnpy = np.array(list(seq))
    basenpy[seqnpy=='A'] = basenpy_standard[0]
    basenpy[seqnpy=='a'] = basenpy_standard[0]
    basenpy[seqnpy=='G'] = basenpy_standard[1]
    basenpy[seqnpy=='g'] = basenpy_standard[1]
    basenpy[seqnpy=='C'] = basenpy_standard[2]
    basenpy[seqnpy=='c'] = basenpy_standard[2]
    basenpy[seqnpy=='U'] = basenpy_standard[3]
    basenpy[seqnpy=='u'] = basenpy_standard[3]
    basenpy[seqnpy=='T'] = basenpy_standard[3]
    basenpy[seqnpy=='t'] = basenpy_standard[3]
    return basenpy

In [ ]:
import numpy as np

def score_energy_one_simple(seq, target_id, out):

    coor = out['coor']  # (L, 3, 3)
    L = len(seq)

    d0_P_C4  = 1.60
    d0_C4_N  = 1.47
    k_bond   = 100.0 

    energy_bond = 0.0
    for i in range(L):
        P  = coor[i,0]
        C4 = coor[i,1]
        N  = coor[i,2]
        d_PC4 = np.linalg.norm(P - C4)
        d_C4N = np.linalg.norm(C4 - N)
        energy_bond += k_bond * (d_PC4 - d0_P_C4)**2
        energy_bond += k_bond * (d_C4N - d0_C4_N)**2

    theta0   = np.deg2rad(109.5)
    k_angle  = 20.0   

    energy_angle = 0.0
    for i in range(L):
        P  = coor[i,0]
        C4 = coor[i,1]
        N  = coor[i,2]
        v1 = P  - C4
        v2 = N  - C4
        cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1)*np.linalg.norm(v2) + 1e-8)
        theta = np.arccos(np.clip(cos_theta, -1.0, 1.0))
        energy_angle += k_angle * (theta - theta0)**2

    d0_stack = 3.4
    k_stack  = 5.0   # (kcal/mol/Å²)

    energy_stack = 0.0
    for i in range(L-1):
        C4_i   = coor[i  ,1]
        C4_ip1 = coor[i+1,1]
        d = np.linalg.norm(C4_i - C4_ip1)
        energy_stack += k_stack * (d - d0_stack)**2

    total_energy = energy_bond + energy_angle + energy_stack

    # print(f"[{target_id}] bond={energy_bond:.2f}, angle={energy_angle:.2f}, stack={energy_stack:.2f} → total={total_energy:.2f}")

    return total_energy

In [ ]:
import numpy as np

def score_energy_one_full(seq, target_id, out, paired=None,
                          # bond parameters
                          d0_P_C4=1.60, d0_C4_N=1.47, k_bond=100.0,
                          # angle parameters
                          theta0=np.deg2rad(109.5), k_angle=20.0,
                          # stacking parameters
                          d0_stack=3.4, k_stack=5.0,
                          # dihedral parameters
                          phi0=np.deg2rad(180.0), k_dihedral=5.0,
                          # hydrogen-bond parameters
                          d0_hb=2.9, k_hb=10.0,
                          # Lennard-Jones parameters
                          sigma=4.0, epsilon=0.1,
                          # Debye-Hückel electrostatics
                          q_P=-1.0, epsilon_r=80.0, kappa=10.0,
                          k_e=332.0637):
    coor = out['coor']
    L = len(seq)
    if paired is None:
        paired = out.get('paired', [])

    E_bond = 0.0
    for i in range(L):
        P  = coor[i,0]; C4 = coor[i,1]; N = coor[i,2]
        d_PC4 = np.linalg.norm(P - C4)
        d_C4N = np.linalg.norm(C4 - N)
        E_bond += k_bond * (d_PC4 - d0_P_C4)**2
        E_bond += k_bond * (d_C4N - d0_C4_N)**2
    E_angle = 0.0
    for i in range(L):
        P, C4, N = coor[i]
        v1 = P - C4; v2 = N - C4
        cost = np.dot(v1, v2) / (np.linalg.norm(v1)*np.linalg.norm(v2) + 1e-8)
        theta = np.arccos(np.clip(cost, -1.0, 1.0))
        E_angle += k_angle * (theta - theta0)**2

    E_stack = 0.0
    for i in range(L-1):
        d = np.linalg.norm(coor[i,1] - coor[i+1,1])
        E_stack += k_stack * (d - d0_stack)**2

    def torsion_angle(a, b, c, d):
        b1, b2, b3 = b-a, c-b, d-c
        n1 = np.cross(b1, b2); n2 = np.cross(b2, b3)
        n1 /= np.linalg.norm(n1) + 1e-8; n2 /= np.linalg.norm(n2) + 1e-8
        cos_phi = np.dot(n1, n2)
        return np.arccos(np.clip(cos_phi, -1, 1))

    E_dihedral = 0.0
    for i in range(L-1):
        a = coor[i,0]; b = coor[i,1]; c = coor[i,2]; d = coor[i+1,0]
        phi = torsion_angle(a, b, c, d)
        E_dihedral += k_dihedral * (phi - phi0)**2

    E_hb = 0.0
    for i, j in paired:
        d = np.linalg.norm(coor[i,2] - coor[j,2])
        E_hb += k_hb * (d - d0_hb)**2

    E_LJ = 0.0
    for i in range(L):
        for j in range(i+2, L): 
            r = np.linalg.norm(coor[i,1] - coor[j,1])
            sr6 = (sigma / (r + 1e-8))**6
            sr12 = sr6 * sr6
            E_LJ += 4 * epsilon * (sr12 - sr6)

    E_elec = 0.0
    for i in range(L):
        for j in range(i+1, L):
            r = np.linalg.norm(coor[i,0] - coor[j,0])
            prefac = k_e * q_P * q_P / epsilon_r
            E_elec += prefac * np.exp(-r / kappa) / (r + 1e-8)

    total_energy = (E_bond + E_angle + E_stack +
                    E_dihedral + E_hb + E_LJ + E_elec)

    # print(f"[{target_id}] bond={E_bond:.2f}, angle={E_angle:.2f}, stack={E_stack:.2f}, \
#          dihedral={E_dihedral:.2f}, hb={E_hb:.2f}, LJ={E_LJ:.2f}, elec={E_elec:.2f} -> total={total_energy:.2f}")
    return total_energy

In [ ]:
import os
import json
import pickle
import numpy as np
import tempfile
from Bio.PDB import PDBParser
sys.path.append('/kaggle/input/drfold-model/DRfold2/PotentialFold')
from Optimization import Structure

def score_energy_one(seq, target_id, out):
    with tempfile.TemporaryDirectory() as tmpdirname:
        fastafile = os.path.join(tmpdirname, 'tmp.fasta')
        with open(fastafile, 'w') as f:
            f.write(f'>{target_id}\n{seq}\n')
        retfile = os.path.join(tmpdirname, 'tmp.ret')
        with open(retfile, 'wb') as f:
            f.write(pickle.dumps(out))
        # foldconfig = '/kaggle/input/drfold-model/DRfold2/cfg_for_selection.json'
        # foldconfig = '/kaggle/input/drfold-model/DRfold2/cfg_for_folding.json'
        foldconfig = '/kaggle/input/drfold-model-bf16/cfg_for_folding.json'
        # foldconfig = 'cfg_for_folding.json'
        save_prefix = os.path.join(tmpdirname, 'tmp.json')
        stru=Structure(fastafile,[retfile],save_prefix,0,foldconfig)
        rama=stru.init_quat(0).data.numpy()
        energy=stru.obj_func_np(rama)
        return energy



def optimize_coor(seq, target_id, out):
    print('Optimizing structure for ', target_id)
    if len(out['coor'])>len(out['plddt']):
        if len(out['plddt'])>0:
            mean_plddt = np.mean(out['plddt'])
            out['plddt'] = np.concatenate([out['plddt'], np.full((len(out['coor'])-len(out['plddt'])), mean_plddt)])
        else:
            out['plddt'] = np.full((len(out['coor'])), 0.0)
            
    if len(out['coor'])<len(out['plddt']):
        mean_plddt = np.mean(out['plddt'])
        out['plddt'] = np.full((len(out['coor'])), mean_plddt)
    
    with tempfile.TemporaryDirectory() as tmpdirname:
        fastafile = os.path.join(tmpdirname, 'tmp.fasta')
        with open(fastafile, 'w') as f:
            f.write(f'>{target_id}\n{seq}\n')

        retfile = os.path.join(tmpdirname, 'tmp.ret')
        with open(retfile, 'wb') as f:
            pickle.dump(out, f)

        # foldconfig = '/kaggle/input/drfold-model/DRfold2/cfg_for_folding.json'
        foldconfig = '/kaggle/input/drfold-model-bf16/cfg_for_folding.json'
        # foldconfig = 'cfg_for_folding.json'
        save_prefix = os.path.join(tmpdirname, 'tmp')
        stru = Structure(fastafile, [retfile], save_prefix, 0, foldconfig)
        stru.foldning()

        pdb_file = save_prefix + '.pdb'
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(target_id, pdb_file)

        residues = [
            res for res in structure.get_residues()
            if res.id[0] == ' '
        ]
        L = len(seq)
        if len(residues) != L:
            raise ValueError(f"PDB 中残基数 ({len(residues)}) 与序列长度 ({L}) 不一致")

        atom_order = ['P', "C4'", 'N1/N9']
        coor = np.zeros((L, 3, 3), dtype=float)

        for i, res in enumerate(residues):
            coor[i, :, :] = np.nan  
            
            if 'P' in res:
                coord = res['P'].get_vector().get_array()
                coor[i, 0, :] = coord
            if "C4'" in res:
                coord = res["C4'"].get_vector().get_array()
                coor[i, 1, :] = coord
            if 'N1' in res:
                coord = res['N1'].get_vector().get_array()
                coor[i, 2, :] = coord
            elif 'N9' in res:
                coord = res['N9'].get_vector().get_array()
                coor[i, 2, :] = coord

        return coor

In [ ]:
sys.path.append('/kaggle/input/datasets/z1493916656/drfold-model-bf16')
sys.path.append('/kaggle/input/datasets/z1493916656/drfold-model-bf16/PotentialFold')
sys.path.append(f'/kaggle/input/datasets/z1493916656/drfold-model-bf16/{CFG_DIR}')
sys.path.append(f'/kaggle/input/datasets/z1493916656/drfold-model-bf16/{CFG_DIR}/RNALM2')

BASE_COOR = np.load('/kaggle/input/datasets/z1493916656/drfold-model-bf16/PotentialFold/lib/base.npy')
OTHER_COOR = np.load('/kaggle/input/datasets/z1493916656/drfold-model-bf16/PotentialFold/lib/other2.npy')
SIDE_COOR = np.load('/kaggle/input/datasets/z1493916656/drfold-model-bf16/PotentialFold/lib/side.npy')

from EvoMSA2XYZ import MSA2XYZ
from RNALM2.Model import RNA2nd
from data import parse_seq

In [ ]:
# data helper
def make_data(seq, device):
    aa_type = parse_seq(seq)
    base = Get_base(seq, BASE_COOR)
    seq_idx = np.arange(len(seq)) + 1

    msa = aa_type[None, :]
    msa = torch.from_numpy(msa)
    msa = torch.cat([msa, msa], 0)  # ???
    msa = F.one_hot(msa.long(), 6).float()

    base_x = torch.from_numpy(base).float()
    seq_idx = torch.from_numpy(seq_idx).long()

    msa, base_x, seq_idx = msa.to(device), base_x.to(device), seq_idx.to(device)
    return msa, base_x, seq_idx


def solution_to_submit_df(solution):
    submit_df = []
    for k,s in solution.items():
        df = coord_to_df(s.sequence, s.coord, s.target_id)
        submit_df.append(df)
    
    submit_df = pd.concat(submit_df)
    return submit_df
 

def coord_to_df(sequence, coord, target_id):
    L = len(sequence)
    df = pd.DataFrame()
    df['ID'] = [f'{target_id}_{i + 1}' for i in range(L)]
    df['resname'] = [s for s in sequence]
    df['resid'] = [i + 1 for i in range(L)]

    num_coord = len(coord)
    for j in range(num_coord):
        df[f'x_{j+1}'] = coord[j][:, 0]
        df[f'y_{j+1}'] = coord[j][:, 1]
        df[f'z_{j+1}'] = coord[j][:, 2]
    return df


out_dir = '/kaggle/working/model-output'
os.makedirs(out_dir, exist_ok=True)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def run_submit(valid_df):
    
    #load model (these are moified versions, not the same from their github repo)
    rnalm = RNA2nd(dict(
        s_in_dim=5,
        z_in_dim=2,
        s_dim= 512,
        z_dim= 128,
        N_elayers=18,
    ))
    rnalm_file = '/kaggle/input/datasets/z1493916656/drfold-model-bf16/model_hub/RCLM/epoch_67000'
    print(rnalm_file)
    print(
        rnalm.load_state_dict(torch.load(rnalm_file, map_location='cpu', weights_only=True), strict=False)
        #Unexpected key(s) in state_dict: "ss_head.linear.weight", "ss_head.linear.bias".
    )
    rnalm = rnalm.to(DEVICE)
    if(PREC=='fp16'):
        rnalm=rnalm.half()
        
    if PREC=='bf16':
        rnalm = rnalm.bfloat16()
        
    rnalm = rnalm.eval()

    #---
    msa2xyz = MSA2XYZ(
        seq_dim=6,
        msa_dim=7,
        N_ensemble=1,
        N_cycle=8,  # 8
        m_dim=64,
        s_dim=64,
        z_dim=64,
    )
    msa2xyz_file = [f'/kaggle/input/datasets/z1493916656/drfold-model-bf16/model_hub/{CFG_DIR}/model_{i}' for i in range(20)]
    if CFG_MERGE:
        msa2xyz_file = [
            f'/kaggle/input/datasets/z1493916656/drfold-model-bf16/model_hub/cfg_97/model_{i}'
            for i in range(20)
        ] + [
            f'/kaggle/input/datasets/z1493916656/drfold-model-bf16/model_hub/cfg_95/model_{i}'
            for i in range(20)
        ] + [
            f'/kaggle/input/datasets/z1493916656/drfold-model-bf16/model_hub/cfg_96/model_{i}'
            for i in range(20)
        ] + [
            f'/kaggle/input/datasets/z1493916656/drfold-model-bf16/model_hub/cfg_99/model_{i}'
            for i in range(20)
        ]
    num_msa2xyz = len(msa2xyz_file) 
    msa2xyz_state_dict = []
    for c in range(num_msa2xyz):
        if c==0: print(msa2xyz_file[c])
        m = torch.load(msa2xyz_file[c], map_location='cpu', weights_only=True)
        msa2xyz_state_dict.append(m)
        
    #print(msa2xyz.load_state_dict(msa2xyz_state_dict[0], strict=True))
    print(msa2xyz.load_state_dict(msa2xyz_state_dict[0], strict=False))
    msa2xyz = msa2xyz.to(DEVICE)
    if(PREC=='fp16'):
        msa2xyz=msa2xyz.half()

    if PREC=='bf16':
        msa2xyz = msa2xyz.bfloat16()
    msa2xyz = msa2xyz.eval()
    
    msa2xyz.msaxyzone.premsa.rnalm = rnalm

    #---
    # start here !!!!!!!!!!!!!!!!!!!!!!
    #valid_df = valid_df.iloc[[0,1]].reset_index(drop=True)


    drfold_pred = [] 
    total_time_taken = 0
    max_gpu_mem_used = 0

    for i, row in valid_df.iterrows():
        start_timer = timer()
        target_id = row.target_id  # 'R1116' #casp15 R1116: len(157)
        sequence = row.sequence
        seq = row.sequence  
        L = len(seq)
        if L > MAX_CAT_LENGTH:
            seq = seq[:MAX_CAT_LENGTH]
        # else:
        #     continue
        print(i, target_id, L, len(seq), seq[:75] + '...')

        
        if len(seq)>480:
            model_to_try=[16, 9, 1, 2, 0]
        elif len(seq)>200:
            # model_to_try = [0,1,2,8,9]
            model_to_try = [13, 6, 14, 5, 3]
        elif  len(seq)>100:
            # model_to_try = [0,2,4,6,8,10,12,14,16,18]#list(range(min(num_msa2xyz,10)))
            model_to_try = [13, 6, 14, 12, 7, 2, 5, 19, 10, 9]
            if CFG_MERGE:
                model_to_try = [24, 34, 20, 13, 6, 37, 28, 25, 14, 39]
        else:
            # model_to_try = list(range(num_msa2xyz))
            
            # model_to_try = list(range(20))
            model_to_try = [1, 2, 0, 8, 7, 5, 6, 14, 10, 18, 4, 13, 3, 17, 19, 11, 12, 15, 16, 9]
            
            # if CFG_MERGE:
            #     model_to_try = list(range(20)) + list(range(40,60)) + list(range(60, 80))
            
        if NO_SORT:
            model_to_try=model_to_try[:5]


        # 分段预测
        def predict_segment(seq):
            msa, base_x, seq_idx = make_data(seq, DEVICE)
            with torch.no_grad():
                if PREC=='fp16':
                    msa, base_x = msa.half(), base_x.half()
                if PREC=='bf16':
                    msa, base_x = msa.bfloat16(), base_x.bfloat16()
                return msa2xyz.pred(msa, seq_idx, None, base_x, np.array(list(seq)))

                
        energy = []
        coordinate=[]
        outputs=[]
        for c in model_to_try:
            msa2xyz.load_state_dict(msa2xyz_state_dict[c], strict=False)

            if len(seq) <= MAX_LENGTH:
                outs = [ predict_segment(seq) ]
            else:
                step = MAX_LENGTH - 1
                outs = []
                for s in range(0, len(seq), step):
                    seg = seq[s : min(s+MAX_LENGTH, len(seq))]
                    outs.append(predict_segment(seg))
                    
            out_cat = outs[0]
            for out_seg in outs[1:]:
                out_cat = {'coor': concat_coor(out_cat, out_seg)}
                    
            if NO_SORT:
                e=0
            elif len(model_to_try)>5 and DR_SCORE:
                e = score_energy_one(seq, target_id, out_cat)
            elif FULL_ENERGY:
                e = score_energy_one_full(seq, target_id, out_cat)
            else:
                e = score_energy_one_simple(seq, target_id, out_cat)
            energy.append(e) #tranucated sequence
            
            if L != len(seq):
                out_cat['coor'] = np.pad(out_cat['coor'], ((0, L - len(seq)), (0, 0), (0, 0)), 'constant', constant_values=0)
                
            outputs.append(out_cat)
            
            
            xyz = frame_coor_to_C1(out_cat['coor'], sequence, BASE_COOR, OTHER_COOR)
            
            coordinate.append(xyz)
            

            time_taken = timer() - start_timer
            total_time_taken += time_taken
            #print('time_taken:', time_to_str(time_taken, mode='sec'))

            gpu_mem_used = gpu_memory_use()
            max_gpu_mem_used = max(max_gpu_mem_used,gpu_mem_used)
            #print('gpu_mem_used:', gpu_mem_used, 'GB')

            print(f'{c:02d}   energy:{e:10.0f}   out_cat{str(out_cat["coor"].shape)}  time:{time_to_str(time_taken, mode="sec")}   gpu={gpu_mem_used} gb')

            
        #------- 
        torch.cuda.empty_cache()
        
        if GET_CENTER:
            energy = np.array(energy)
            energy_mean = np.mean(energy)
            energy= np.abs(energy - energy_mean)
        #select top5
        argsort = np.array(energy).argsort()
        argsort = argsort[:5]
        
        if L <= OPTIM_LENGTH:
            out_opt= outputs[argsort[0]]
            out_opt['coor'] = optimize_coor(seq, target_id, out_opt)
            coordinate[argsort[0]] = frame_coor_to_C1(out_opt['coor'], sequence, BASE_COOR, OTHER_COOR)
            torch.cuda.empty_cache()
            
        df = coord_to_df(row.sequence, [coordinate[k] for k in argsort], row.target_id)
        drfold_pred.append(df)
    
    print('----------------------------------------')
    print('MAX_LENGTH', MAX_LENGTH)
    print('### total_time_taken:', time_to_str(total_time_taken, mode='min'))
    print('### max_gpu_mem_used:', max_gpu_mem_used, 'GB')
    print('')

    drfold_pred = pd.concat(drfold_pred)
    drfold_pred.to_csv(f'/kaggle/working/drfold_submission.csv', index=False)
    print(drfold_pred)
    return drfold_pred

run_submit(valid_df)

print('SUBMIT OK!!!')

In [ ]:
drfold_pred = pd.read_csv('/kaggle/working/drfold_submission.csv')
drfold_pred

In [ ]:
# test_sequences = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")

# # Set up directories
# predictions_dir = "/kaggle/working/predictions"
# os.makedirs(predictions_dir, exist_ok=True)
# fasta_dir = "/kaggle/working/fasta_files"
# os.makedirs(fasta_dir, exist_ok=True)

# # Set time limit for DRfold2 (in seconds)
# DRFOLD_TIME_LIMIT = 7 * 60 * 60  # 7 hours
# start_time_global = time.time()

In [ ]:
# !cp -r /kaggle/input/datasets/jaejohn/drfold2-repo/DRfold2 /kaggle/working/
# %cd DRfold2
# %cd Arena
# !make Arena
# %cd ..

In [ ]:
# !cp -r /kaggle/input/datasets/jaejohn/drfold2/model_hub /kaggle/working/DRfold2/

In [ ]:
# %%writefile /kaggle/working/DRfold2/DRfold_infer.py
# import os,sys
# import torch
# import numpy as np
# from subprocess import Popen, PIPE, STDOUT

# # Get the directory where the script is located
# exp_dir = os.path.dirname(os.path.abspath(__file__))

# device = "cuda" if torch.cuda.is_available() else "cpu"
# # dlexps = ['cfg_95','cfg_96','cfg_97','cfg_99']
# dlexps = ['cfg_97']

# print(f"[DRfold2] Starting prediction pipeline on {device} device")

# # Get input FASTA file and output directory from command line arguments
# fastafile =  os.path.realpath(sys.argv[1])
# outdir = os.path.realpath(sys.argv[2])

# print(f"[DRfold2] Input: {fastafile}")
# print(f"[DRfold2] Output: {outdir}")

# # Initialize clustering flag
# pclu = False

# # If third argument is '1', enable clustering
# if len(sys.argv) == 4 and sys.argv[3] == '1': 
#     print('[DRfold2] Clustering enabled - will generate multiple models')
#     pclu = True
# else:
#     print('[DRfold2] Clustering disabled - will generate single model')

# # Create output directory if it doesn't exist
# if not os.path.isdir(outdir):
#     os.makedirs(outdir)
#     print(f"[DRfold2] Created output directory: {outdir}")

# # Create subdirectories for different outputs
# ret_dir = os.path.join(outdir,'rets_dir')  # For return files
# if not os.path.isdir(ret_dir):
#     os.makedirs(ret_dir)
#     print(f"[DRfold2] Created returns directory: {ret_dir}")

# folddir = os.path.join(outdir,'folds')     # For folded structures
# if not os.path.isdir(folddir):
#     os.makedirs(folddir)
#     print(f"[DRfold2] Created folds directory: {folddir}")

# refdir = os.path.join(outdir,'relax')      # For relaxed structures
# if not os.path.isdir(refdir):
#     os.makedirs(refdir)
#     print(f"[DRfold2] Created relaxation directory: {refdir}")

# # Helper function to run commands and capture output
# def run_cmd(cmd, description):
#     print(f"[DRfold2] {description}")
#     print(f"[DRfold2] Command: {cmd}")
    
#     # Execute the command and capture output in real-time
#     process = Popen(cmd, shell=True, stdout=PIPE, stderr=STDOUT, universal_newlines=True, bufsize=1)
    
#     # Print output line by line as it becomes available
#     for line in iter(process.stdout.readline, ''):
#         line = line.strip()
#         if line:
#             print(f"[DRfold2 subprocess] {line}")
    
#     # Get return code
#     return_code = process.wait()
#     if return_code == 0:
#         print(f"[DRfold2] {description} completed successfully")
#     else:
#         print(f"[DRfold2] {description} failed with return code {return_code}")
#     return return_code

# # Create paths for model directories and test scripts
# dlmains = [os.path.join(exp_dir, one_exp, 'test_modeldir.py') for one_exp in dlexps]
# dirs = [os.path.join(exp_dir, 'model_hub', one_exp) for one_exp in dlexps]

# # Check if processing has been done before
# if not os.path.isfile(ret_dir + '/done'): 
#     print("[DRfold2] Step 1/4: GENERATING INITIAL PREDICTIONS")
#     print(f"[DRfold2] No previous predictions found, will generate e2e and geo files")
    
#     # Run each model configuration
#     for idx, (dlmain, one_exp, mdir) in enumerate(zip(dlmains, dlexps, dirs)):
#         # Construct command to run the model
#         cmd = f'python {dlmain} {device} {fastafile} {ret_dir}/{one_exp}_ {mdir}'
#         description = f"Running model {idx+1}/{len(dlexps)}: {one_exp}"
#         run_cmd(cmd, description)

#     # Mark processing as complete
#     wfile = open(ret_dir+'/done','w')
#     wfile.write('1')
#     wfile.close()
#     print("[DRfold2] Initial predictions generation completed")
# else:
#     print("[DRfold2] Step 1/4: USING EXISTING PREDICTIONS")
#     print(f"[DRfold2] Found previous predictions in {ret_dir}, using existing e2e and geo files")

# # Helper function to get model PDB file
# def get_model_pdb(tdir,opt):
#     files = os.listdir(tdir)
#     files = [afile for afile in files if afile.startswith(opt)][0]
#     return files

# # Set up directory paths and configuration files
# cso_dir = folddir                                                    # Directory for coarse-grained structures
# clufile = os.path.join(folddir,'clu.txt')                            # Clustering results file
# config_sel = os.path.join(exp_dir,'cfg_for_selection.json')          # Selection configuration
# foldconfig = os.path.join(exp_dir,'cfg_for_folding.json')            # Folding configuration
# selpython = os.path.join(exp_dir,'PotentialFold','Selection.py')     # Selection script
# optpython = os.path.join(exp_dir,'PotentialFold','Optimization.py')  # Optimization script
# clupy = os.path.join(exp_dir,'PotentialFold','Clust.py')             # Clustering script
# arena = os.path.join(exp_dir,'Arena','Arena')                        # Arena executable for structure refinement

# # Set up initial save prefixes for optimization and selection
# optsaveprefix = os.path.join(cso_dir, f'opt_0')
# save_prefix = os.path.join(cso_dir, f'sel_0')

# # Get all .ret files from the return directory
# rets = os.listdir(ret_dir)
# rets = [afile for afile in rets if afile.endswith('.ret')]
# rets = [os.path.join(ret_dir,aret) for aret in rets ]
# ret_str = ' '.join(rets)

# print("[DRfold2] Step 2/4: SELECTION PROCESS")
# print(f"[DRfold2] Found {len(rets)} return files for selection")
# print(f"[DRfold2] Using selection config: {config_sel}")
# print(f"[DRfold2] Output prefix: {save_prefix}")

# # Run selection process
# cmd = f'python {selpython} {fastafile} {config_sel} {save_prefix} {ret_str}'
# run_cmd(cmd, "Running selection process")

# print("[DRfold2] Step 3/4: OPTIMIZATION PROCESS")
# print(f"[DRfold2] Using fold config: {foldconfig}")
# print(f"[DRfold2] Optimization output prefix: {optsaveprefix}")

# # Run optimization process
# cmd = f'python {optpython} {fastafile} {optsaveprefix} {ret_dir} {save_prefix} {foldconfig}'
# run_cmd(cmd, "Running optimization process")

# # Get the coarse-grained PDB and save refined structure
# cgpdb = os.path.join(folddir,get_model_pdb(folddir,'opt_0'))
# savepdb = os.path.join(refdir,'model_1.pdb')

# print("[DRfold2] Step 4/4: STRUCTURE REFINEMENT")
# print(f"[DRfold2] Found optimized structure: {cgpdb}")
# print(f"[DRfold2] Final output will be saved to: {savepdb}")

# cmd = f'{arena} {cgpdb} {savepdb} 7'
# run_cmd(cmd, "Running structure refinement")

# # If clustering is enabled (pclu=True)
# if pclu:
#     print("[DRfold2] ADDITIONAL STEP: CLUSTERING")
#     print(f"[DRfold2] Running clustering process, output: {clufile}")
    
#     # Run clustering process
#     cmd = f'python {clupy} {ret_dir} {clufile}'
#     run_cmd(cmd, "Running clustering")

#     # Read clustering results
#     lines = open(clufile).readlines()
#     lines = [aline.strip() for aline in lines]
#     lines = [aline for aline in lines if aline]
    
#     cluster_count = len(lines) - 1
#     print(f"[DRfold2] Found {cluster_count} additional clusters to process")

#     # Process each cluster
#     for i in range(1,len(lines)):
#         print(f"[DRfold2] PROCESSING CLUSTER {i}/{cluster_count}")
        
#         # Get return files for this cluster
#         rets = lines[i].split()
#         rets = [os.path.join(ret_dir,aret.replace('.pdb','.ret')) for aret in rets ]
#         ret_str = ' '.join(rets)

#         # Set up save prefixes for this cluster
#         optsaveprefix =  os.path.join(cso_dir,f'opt_{str(i+1)}')
#         save_prefix = os.path.join(cso_dir,f'sel_{str(i+1)}')
        
#         print(f"[DRfold2] Cluster {i} Selection Process")
#         print(f"[DRfold2] Found {len(rets)} return files for selection")
#         print(f"[DRfold2] Selection output prefix: {save_prefix}")

#         # Run selection process for this cluster
#         cmd = f'python {selpython} {fastafile} {config_sel} {save_prefix} {ret_str}'
#         run_cmd(cmd, f"Running selection for cluster {i}")
        
#         print(f"[DRfold2] Cluster {i} Optimization Process")
#         print(f"[DRfold2] Optimization output prefix: {optsaveprefix}")

#         # Run optimization process for this cluster
#         cmd = f'python {optpython} {fastafile} {optsaveprefix} {ret_dir} {save_prefix} {foldconfig}'
#         run_cmd(cmd, f"Running optimization for cluster {i}")

#         # Get the coarse-grained PDB and save refined structure for this cluster
#         cgpdb = os.path.join(folddir,get_model_pdb(folddir,f'opt_{str(i+1)}'))
#         savepdb = os.path.join(refdir,f'model_{str(i+1)}.pdb')
        
#         print(f"[DRfold2] Cluster {i} Refinement Process")
#         print(f"[DRfold2] Found optimized structure: {cgpdb}")
#         print(f"[DRfold2] Final output will be saved to: {savepdb}")

#         cmd = f'{arena} {cgpdb} {savepdb} 7'
#         run_cmd(cmd, f"Running refinement for cluster {i}")

# print("[DRfold2] PREDICTION PIPELINE COMPLETED SUCCESSFULLY")

In [ ]:
# %%writefile /kaggle/working/DRfold2/PotentialFold/operations.py
# """
# operations.py: Core Mathematical Operations for RNA Structure Analysis

# This module provides essential mathematical operations for manipulating and analyzing
# RNA 3D structures, organized into four main categories:

# 1. Basic Vector Operations:
#    Functions for selecting coordinates and calculating distances between points,
#    which form the foundation for all structural calculations.

# 2. Angle Calculations:
#    Functions for computing bond angles and dihedral (torsion) angles between atoms,
#    with differentiable implementations suitable for gradient-based optimization.

# 3. Rigid Body Transformations:
#    Functions for determining optimal rotations and translations between sets of
#    coordinates, enabling structure alignment and manipulation.

# 4. Sequence Utilities:
#    Functions for converting RNA sequence data into standard 3D coordinate templates,
#    allowing sequence-structure mapping.

# These operations support the core functionality of RNA structure prediction, analysis,
# and optimization throughout the codebase.
# """

# import os
# import torch
# import torch.nn as nn
# import numpy as np 
# import math, sys, math
# from io import BytesIO
# import torch.nn.functional as F
# from torch.autograd import Function
# from torch.nn.parameter import Parameter
# from subprocess import Popen, PIPE, STDOUT

# # Use consistent epsilon value across all functions
# EPS = 1e-8


# # === Basic Vector Operations ===
# def coor_selection(coor,mask):
#     #[L,n,3],[L,n],byte
#     return torch.masked_select(coor,mask.bool()).view(-1,3)

# def pair_distance(x1, x2, eps=1e-6, p=2):
#     # Use torch.cdist for p=2 (Euclidean) which is highly optimized
#     if p == 2:
#         return torch.cdist(x1, x2, p=2)
    
#     # For other p-norms, avoid memory expansion with broadcasting
#     x1_ = x1.unsqueeze(1)  # [n1, 1, dim]
#     x2_ = x2.unsqueeze(0)  # [1, n2, dim]
#     diff = torch.abs(x1_ - x2_)
#     out = torch.pow(diff + eps, p).sum(dim=2)
#     return torch.pow(out, 1. / p)


# # === Angle Calculations ===
# def angle(p0, p1, p2):
#     # [b 3] 
#     b0 = p0-p1
#     b1 = p2-p1

#     b0 = b0 / (torch.norm(b0, dim =-1, keepdim=True) + EPS)
#     b1 = b1 / (torch.norm(b1, dim =-1, keepdim=True) + EPS)
    
#     recos = torch.sum(b0*b1, -1)
#     recos = torch.clamp(recos, -0.9999, 0.9999)
#     return torch.acos(recos)

# class torsion(Function):
#     #PyTorch class to calculate differentiable torsion angle
#     #https://stackoverflow.com/questions/20305272/dihedral-torsion-angle-from-four-points-in-cartesian-coordinates-in-python
#     #https://salilab.org/modeller/manual/node492.html
#     @staticmethod
#     def forward(ctx, p0, p1, p2, p3):
#         # Save input points for backward pass
#         ctx.save_for_backward(p0, p1, p2, p3)

#         # Calculate bond vectors
#         b0 = p0 - p1
#         b1 = p2 - p1
#         b2 = p3 - p2

#         # Normalize the middle bond vector
#         b1_norm = torch.norm(b1, dim=-1, keepdim=True) + 1e-8
#         b1_unit = b1 / b1_norm

#         # Project the other bonds onto the plane perpendicular to middle bond
#         v = b0 - torch.sum(b0 * b1_unit, dim=-1, keepdim=True) * b1_unit
#         w = b2 - torch.sum(b2 * b1_unit, dim=-1, keepdim=True) * b1_unit

#         # Calculate torsion using the arctan2 formula (more stable than arccos)
#         x = torch.sum(v * w, dim=-1)                                # cosine component
#         y = torch.sum(torch.cross(b1_unit, v, dim=-1) * w, dim=-1)  # sine component

#         return torch.atan2(y, x)

    
#     @staticmethod
#     def backward(ctx, grad_output):
#         # Retrieve saved tensors from forward pass
#         p0, p1, p2, p3 = ctx.saved_tensors

#         # Calculate bond vectors
#         r01 = p0 - p1
#         r12 = p2 - p1
#         r23 = p3 - p2

#         # Calculate bond lengths with numerical stability
#         d01 = torch.norm(r01, dim=-1, keepdim=True) + 1e-8
#         d12 = torch.norm(r12, dim=-1, keepdim=True) + 1e-8
#         d23 = torch.norm(r23, dim=-1, keepdim=True) + 1e-8

#         # Normalize bond vectors
#         e01 = r01 / d01
#         e12 = r12 / d12
#         e23 = r23 / d23

#         # Calculate normal vectors to the two planes
#         n1 = torch.cross(e01, e12, dim=-1)
#         n2 = torch.cross(e12, e23, dim=-1)

#         # Normalize normal vectors
#         n1_norm = torch.norm(n1, dim=-1, keepdim=True) + 1e-8
#         n2_norm = torch.norm(n2, dim=-1, keepdim=True) + 1e-8
#         n1 = n1 / n1_norm
#         n2 = n2 / n2_norm

#         # Calculate gradients for each atom
#         # These are based on the analytical derivatives of dihedral angles
#         g0 = torch.cross(e01, n1, dim=-1) / d01
#         g1 = -g0 - torch.cross(e12, n1, dim=-1) / d12
#         g2 = torch.cross(e12, n2, dim=-1) / d12 - torch.cross(e23, n2, dim=-1) / d23
#         g3 = torch.cross(e23, n2, dim=-1) / d23

#         # Apply chain rule with incoming gradient
#         g0 = g0 * grad_output.unsqueeze(-1)
#         g1 = g1 * grad_output.unsqueeze(-1)
#         g2 = g2 * grad_output.unsqueeze(-1)
#         g3 = g3 * grad_output.unsqueeze(-1)

#         return g0, g1, g2, g3


# def dihedral(input1, input2, input3, input4):
#     return torsion.apply(input1, input2, input3, input4)



# # === Rigid Body Transformations ===
# def rigidFrom3Points(x):    
#     x1, x2, x3 = x[:, 0], x[:, 1], x[:, 2]
#     v1 = x3 - x2
#     v2 = x1 - x2
    
#     # Normalize v1 to get e1
#     e1 = F.normalize(v1, p=2, dim=-1)
    
#     # Project v2 onto e1 and subtract to get the component orthogonal to e1
#     u2 = v2 - e1 * (torch.einsum('bn,bn->b', e1, v2)[:, None])
    
#     # Normalize u2 to get e2
#     e2 = F.normalize(u2, p=2, dim=-1)
    
#     # Cross product to get e3
#     e3 = torch.cross(e1, e2, dim=-1)
    
#     return torch.stack([e1, e2, e3], dim=1)


# # return the direction from to_q to from_p
# def Kabsch_rigid(bases,x1,x2,x3):
#     # Early return for empty input
#     if x1.shape[0] == 0:
#         return torch.empty(0, 3, 3), torch.empty(0, 3)
    
#     the_dim=1
#     to_q = torch.stack([x1,x2,x3],dim=the_dim)
#     biasq=torch.mean(to_q,dim=the_dim,keepdim=True)
#     q=to_q-biasq
#     m = torch.einsum('bnz,bny->bzy',bases,q)
#     u, s, v = torch.svd(m)
#     vt = torch.transpose(v, 1, 2)
#     det = torch.det(torch.matmul(u, vt))
#     det = det.view(-1, 1, 1)
#     vt = torch.cat((vt[:, :2, :], vt[:, -1:, :] * det), 1)
#     r = torch.matmul(u, vt)
#     return r,biasq.squeeze()



# # === Sequence Utilities ===
# def Get_base(seq,basenpy_standard):
#     base_num = basenpy_standard.shape[1]
#     basenpy = np.zeros([len(seq),base_num,3])
#     seqnpy = np.array(list(seq))
#     basenpy[seqnpy=='A']=basenpy_standard[0]
#     basenpy[seqnpy=='a']=basenpy_standard[0]

#     basenpy[seqnpy=='G']=basenpy_standard[1]
#     basenpy[seqnpy=='g']=basenpy_standard[1]

#     basenpy[seqnpy=='C']=basenpy_standard[2]
#     basenpy[seqnpy=='c']=basenpy_standard[2]

#     basenpy[seqnpy=='U']=basenpy_standard[3]
#     basenpy[seqnpy=='u']=basenpy_standard[3]

#     basenpy[seqnpy=='T']=basenpy_standard[3]
#     basenpy[seqnpy=='t']=basenpy_standard[3]
    
#     return torch.from_numpy(basenpy).double()

In [ ]:
# %%writefile /kaggle/working/DRfold2/PotentialFold/Optimization.py
# #! /nfs/amino-home/liyangum/miniconda3/bin/python
# import torch
# import random
# import numpy as np 
# import os, json, sys

# import Cubic, Potential
# import operations
# import a2b, rigid
# import torch.optim as opt
# from scipy.optimize import minimize
# import pickle

# torch.manual_seed(6)
# np.random.seed(9)
# random.seed(9)


# Scale_factor = 1.0
# USEGEO = False

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# def readconfig(configfile=''):
#     config=[]
#     expdir=os.path.dirname(os.path.abspath(__file__))
#     if configfile=='':
#         configfile=os.path.join(expdir,'lib','ddf.json')
#     config=json.load(open(configfile,'r'))
#     return config 

    
# class Structure:
#     def __init__(self,fastafile,geofiles,saveprefix,initial_ret,foldconfig):
#         self.config=readconfig(foldconfig)
#         self.seqfile=fastafile
#         self.init_ret = initial_ret
#         self.foldconfig = foldconfig
#         self.geofiles = geofiles
#         self.rets = [pickle.load(open(refile,'rb')) for refile  in geofiles]
#         self.txs=[]
#         for ret in self.rets:
#             self.txs.append( torch.from_numpy(ret['coor']).double().to(device))
#         self.handle_geo()
#         self.pair = []
#         for ret in self.rets:
#             self.pair.append(torch.from_numpy(ret['plddt']).double().to(device))
#         self.saveprefix=saveprefix
#         self.seq=open(fastafile).readlines()[1].strip()
#         self.L=len(self.seq)
#         basenpy = np.load(os.path.join(os.path.dirname(os.path.abspath(__file__)),'lib','base.npy'))
#         self.basex = operations.Get_base(self.seq,basenpy).to(device)
#         othernpy = np.load(os.path.join(os.path.dirname(os.path.abspath(__file__)),'lib','other2.npy'))
#         self.otherx = operations.Get_base(self.seq,othernpy).to(device)
#         sidenpy = np.load(os.path.join(os.path.dirname(os.path.abspath(__file__)),'lib','side.npy'))
#         self.sidex = operations.Get_base(self.seq,sidenpy).to(device)
        
#         self.init_mask()
#         self.init_paras()
#         self._init_fape()
#         self.tx2ds = [td.to(device) for td in self.tx2ds]
#         self.local_weight = torch.ones(self.L,self.L).to(device)
        
#         for i in range(self.L):
#             for j in range(i+1,min(self.L,i+2)):
#                 self.local_weight[i,j] = self.local_weight[j,i] = 4
#             for j in range(i+2,min(self.L,i+3)):
#                 self.local_weight[i,j] = self.local_weight[j,i] = 3
#             for j in range(i+3,min(self.L,i+4)):
#                 self.local_weight[i,j] = self.local_weight[j,i] = 2

#     def _init_fape(self):
#         self.tx2ds = []
#         for tx in self.txs:
#             true_rot,true_trans = operations.Kabsch_rigid(self.basex,tx[:,0],tx[:,1],tx[:,2])
#             true_x2 = tx[:,None,:,:] - true_trans[None,:,None,:]
#             true_x2 = torch.einsum('ijnd,jde->ijne',true_x2,true_rot.transpose(-1,-2))
#             self.tx2ds.append(true_x2)
    
#     def handle_geo(self):
#         oldkeys=['dist_p','dist_c','dist_n']
#         newkeys=['pp','cc','nn']
#         self.geos=[]
#         for ret in self.rets:
#             geo = {}
#             for nk,ok in zip(newkeys,oldkeys):
#                 geo[nk] = torch.from_numpy(ret[ok].astype(np.float64)).to(device) + 0
#             self.geos.append(geo)


#     def init_mask(self):
#         halfmask=np.zeros([self.L,self.L])
#         fullmask=np.zeros([self.L,self.L])
#         for i in range(self.L):
#             for j in range(i+1,self.L):
#                 halfmask[i,j]=1
#                 fullmask[i,j]=1
#                 fullmask[j,i]=1
#         self.halfmask=(torch.DoubleTensor(halfmask) > 0.5).to(device)
#         self.fullmask=(torch.DoubleTensor(fullmask) > 0.5).to(device)
#         self.clash_mask = torch.zeros([self.L,self.L,22,22], device=device)
#         for i in range(self.L):
#             for j in range(i+1,self.L):
#                 self.clash_mask[i,j]=1

#         for i in range(self.L):
#              self.clash_mask[i,i,:6,7:]=1

#         for i in range(self.L-1):
#             self.clash_mask[i,i+1,:,0]=0
#             self.clash_mask[i,i+1,0,:]=0
#             self.clash_mask[i,i+1,:,5]=0
#             self.clash_mask[i,i+1,5,:]=0

#         self.side_mask = rigid.side_mask(self.seq).to(device)
#         self.side_mask = (self.side_mask[:,None,:,None] * self.side_mask[None,:,None,:]).to(device)
#         self.clash_mask = ((self.clash_mask > 0.5) * (self.side_mask > 0.5)).to(device)

#         self.geo_confimask_cc = []
#         self.geo_confimask_pp = []
#         self.geo_confimask_nn = []
#         for geo in self.geos:
#             confimask_cc = geo['cc'][:,:,-1] < 0.5
#             confimask_pp = geo['pp'][:,:,-1] < 0.5
#             confimask_nn = geo['nn'][:,:,-1] < 0.5
#             self.geo_confimask_cc.append(confimask_cc)
#             self.geo_confimask_pp.append(confimask_pp)
#             self.geo_confimask_nn.append(confimask_nn)


#     def init_paras(self):
#         self.geo_cc = []
#         self.geo_pp = []
#         self.geo_nn = []
#         self.cs_coefs = {'cc': [], 'pp': [], 'nn': []}
#         self.cs_knots = {'cc': [], 'pp': [], 'nn': []}
#         for geo in self.geos:
#             cc_cs,cc_decs=Cubic.dis_cubic(geo['cc'],2,40,36)
#             pp_cs,pp_decs=Cubic.dis_cubic(geo['pp'],2,40,36)
#             nn_cs,nn_decs=Cubic.dis_cubic(geo['nn'],2,40,36)
#             self.geo_cc.append([cc_cs,cc_decs])
#             self.geo_pp.append([pp_cs,pp_decs])
#             self.geo_nn.append([nn_cs,nn_decs])
            
#             L = self.L
#             cc_coefs_np  = np.stack([[cc_cs[i,j].c for j in range(L)] for i in range(L)], axis=0)
#             cc_knots_np  = np.stack([[cc_cs[i,j].x for j in range(L)] for i in range(L)], axis=0)
#             self.cs_coefs['cc'].append(torch.from_numpy(cc_coefs_np).to(device))
#             self.cs_knots['cc'].append(torch.from_numpy(cc_knots_np).to(device))
            
#             pp_coefs_np  = np.stack([[pp_cs[i,j].c for j in range(L)] for i in range(L)], axis=0)
#             pp_knots_np  = np.stack([[pp_cs[i,j].x for j in range(L)] for i in range(L)], axis=0)
#             self.cs_coefs['pp'].append(torch.from_numpy(pp_coefs_np).to(device))
#             self.cs_knots['pp'].append(torch.from_numpy(pp_knots_np).to(device))
            
#             nn_coefs_np  = np.stack([[nn_cs[i,j].c for j in range(L)] for i in range(L)], axis=0)
#             nn_knots_np  = np.stack([[nn_cs[i,j].x for j in range(L)] for i in range(L)], axis=0)
#             self.cs_coefs['nn'].append(torch.from_numpy(nn_coefs_np).to(device))
#             self.cs_knots['nn'].append(torch.from_numpy(nn_knots_np).to(device))


#     def compute_bb_clash(self,coor,other_coor):
#         com_coor = torch.cat([coor,other_coor],dim=1)
#         com_dis  = (com_coor[:,None,:,None,:] - com_coor[None,:,None,:,:]).norm(dim=-1)
#         dynamicmask2_vdw= (com_dis <= 3.15) * (self.clash_mask)
#         vdw_dynamic = Potential.LJpotential(com_dis[dynamicmask2_vdw],3.15)
#         return vdw_dynamic.sum()*self.config['weight_vdw']

#     def compute_full_clash(self,coor,other_coor,side_coor):
#         com_coor = torch.cat([coor[:,:2],other_coor,side_coor],dim=1)
#         com_dis  = (com_coor[:,None,:,None,:] - com_coor[None,:,None,:,:]).norm(dim=-1)
#         dynamicmask2_vdw= (com_dis <= 2.5) * (self.clash_mask)
#         vdw_dynamic = Potential.LJpotential(com_dis[dynamicmask2_vdw],2.5)
#         return vdw_dynamic.sum()*self.config['weight_vdw']


#     def _cubic_pair_energy(self, atom_map, geo_cs, geo_confimask, weight_key):
#         """General cubic-spline energy for CC/PP/NN pairs."""
#         min_dis, max_dis, bin_num = 2, 40, 36
#         dev = atom_map.device
#         upper_th = max_dis - ((max_dis - min_dis) / bin_num) * 0.5
#         lower_th = 2.5
#         total = torch.zeros((), device=dev, dtype=torch.double)
#         spline_key   = weight_key.split('_')[1]  # 'cc', 'pp', or 'nn'
#         coeffs_list  = self.cs_coefs[spline_key]
#         knots_list   = self.cs_knots[spline_key]
#         for block_idx, mask_block in enumerate(geo_confimask):
#             mask = (atom_map <= upper_th) & mask_block & self.fullmask & (atom_map >= lower_th)
#             idx = mask.nonzero(as_tuple=True)
#             if idx[0].numel() > 1:
#                 coef  = coeffs_list[block_idx][idx]
#                 knots = knots_list[block_idx][idx]
#                 part1 = Potential.cubic_distance(atom_map[mask], coef, knots, min_dis, max_dis, bin_num).sum() * self.config[weight_key] * 0.5
#             else:
#                 part1 = torch.zeros((), device=dev)
#             part2 = ((atom_map <= lower_th) & mask_block & self.fullmask).sum() * self.config[weight_key]
#             total = total + part1 + part2
#         return total

#     def compute_cc_energy(self, coor):
#         atom_map = operations.pair_distance(coor[:,1], coor[:,1])
#         return self._cubic_pair_energy(atom_map, self.geo_cc, self.geo_confimask_cc, 'weight_cc')
    
#     def compute_pp_energy(self, coor):
#         atom_map = operations.pair_distance(coor[:,0], coor[:,0])
#         return self._cubic_pair_energy(atom_map, self.geo_pp, self.geo_confimask_pp, 'weight_pp')
    
#     def compute_nn_energy(self, coor):
#         atom_map = operations.pair_distance(coor[:,-1], coor[:,-1])
#         return self._cubic_pair_energy(atom_map, self.geo_nn, self.geo_confimask_nn, 'weight_nn')

#     def compute_pccp_energy(self,coor):
#         p_atoms=coor[:,0]
#         c_atoms=coor[:,1]
#         pccpmap=operations.dihedral( p_atoms[self.pccpi], c_atoms[self.pccpi], c_atoms[self.pccpj] ,p_atoms[self.pccpj]                  )
#         neg_log = Potential.cubic_torsion(pccpmap,self.pccp_coe,self.pccp_x,36)
#         return neg_log.sum()*self.config['weight_pccp']

#     def compute_cnnc_energy(self,coor):
#         n_atoms=coor[:,-1]
#         c_atoms=coor[:,1]
#         pccpmap=operations.dihedral( c_atoms[self.cnnci], n_atoms[self.cnnci], n_atoms[self.cnncj] ,c_atoms[self.cnncj]                  )
#         neg_log = Potential.cubic_torsion(pccpmap,self.cnnc_coe,self.cnnc_x,36)
#         return neg_log.sum()*self.config['weight_cnnc']

#     def compute_pnnp_energy(self,coor):
#         n_atoms=coor[:,-1]
#         p_atoms=coor[:,0]
#         pccpmap=operations.dihedral( p_atoms[self.pnnpi], n_atoms[self.pnnpi], n_atoms[self.pnnpj] ,p_atoms[self.pnnpj]                  )
#         neg_log = Potential.cubic_torsion(pccpmap,self.pnnp_coe,self.pnnp_x,36)
#         return neg_log.sum()*self.config['weight_pnnp']

#     def compute_pcc_energy(self,coor):
#         p_atoms=coor[:,1]
#         c_atoms=coor[:,2]
#         pccmap=operations.angle( p_atoms[self.pcci], c_atoms[self.pcci], c_atoms[self.pccj]                   )
#         neg_log = Potential.cubic_angle(pccmap,self.pcc_coe,self.pcc_x,12)
#         return neg_log.sum()*self.config['weight_pcc']

#     def compute_fape_energy(self,coor,ep=1e-3,epmax=20):
#         energy= 0
#         for tx in self.tx2ds:
#             px_mean = coor[:,[1]]
#             p_rot   = operations.rigidFrom3Points(coor)
#             p_tran  = px_mean[:,0]
#             pred_x2 = coor[:,None,:,:] - p_tran[None,:,None,:] # Lx Lrot N , 3
#             pred_x2 = torch.einsum('ijnd,jde->ijne',pred_x2,p_rot.transpose(-1,-2)) # transpose should be equal to inverse
#             errmap=torch.sqrt( ((pred_x2 - tx)**2).sum(dim=-1) + ep )
#             energy = energy + torch.sum(  torch.clamp(errmap,max=epmax)        )
#         return energy * self.config['weight_fape']

#     def compute_bond_energy(self,coor,other_coor):
#         # 3.87
#         o3 = other_coor[:-1,-2]
#         p  = coor[1:,0]
#         dis = (o3-p).norm(dim=-1)
#         energy = ((dis-1.607)**2).sum()
#         return energy * self.config['weight_bond']

#     def tooth_func(self,errmap, ep = 0.05):
#         return -1/(errmap/10+ep) + (1/ep)

#     def reweight_func(self,ww):
#         reweighting = torch.pow(ww,self.config['pair_weight_power'])
#         reweighting[ww < self.config['pair_weight_min']] = 0
#         return reweighting

#     def compute_fape_energy_fromquat(self,x,coor,ep=1e-6,epmax=100):
#         energy= 0
#         p_rot,px_mean = a2b.Non2rot(x[:,:9],x.shape[0]),x[:,9:]
#         pred_x2 = coor[:,None,:,:] - px_mean[None,:,None,:] # Lx Lrot N , 3
#         pred_x2 = torch.einsum('ijnd,jde->ijne',pred_x2,p_rot.transpose(-1,-2)) # transpose should be equal to inverse
#         for tx,weightplddt in zip(self.tx2ds,self.pair):

#             tamplate_dist_map = torch.min( tx.norm(dim=-1), dim=2   )[0]
#             errmap=torch.sqrt( ((pred_x2 - tx)**2).sum(dim=-1) + ep ) 
#             energy = energy + torch.sum( ( (torch.clamp(errmap,max=self.config['FAPE_max'])**self.config['pair_error_power'])  * self.reweight_func(weightplddt[...,None]) * self.local_weight[...,None] )[tamplate_dist_map>self.config['pair_rest_min_dist']]    )

#         return energy * self.config['weight_fape']


#     def energy(self,rama):
#         coor=a2b.quat2b(self.basex,rama[:,9:])
#         other_coor = a2b.quat2b(self.otherx,rama[:,9:])
#         side_coor = a2b.quat2b(self.sidex,torch.cat([rama[:,:9],coor[:,-1]],dim=-1))
        
#         if self.config['weight_cc']>0:
#             E_cc= self.compute_cc_energy(coor) / len(self.rets)
#         else:
#             E_cc=0
#         if self.config['weight_pp']>0:
#             E_pp= self.compute_pp_energy(coor) / len(self.rets)
#         else:
#             E_pp=0
#         if self.config['weight_nn']>0:
#             E_nn= self.compute_nn_energy(coor) / len(self.rets)
#         else:
#             E_nn=0

#         if self.config['weight_pccp']>0:
#             E_pccp= self.compute_pccp_energy(coor) / len(self.rets)
#         else:
#             E_pccp=0

#         if self.config['weight_cnnc']>0:
#             E_cnnc= self.compute_cnnc_energy(coor)  / len(self.rets)
#         else:
#             E_cnnc=0

#         if self.config['weight_pnnp']>0:
#             E_pnnp= self.compute_pnnp_energy(coor) / len(self.rets)
#         else:
#             E_pnnp=0

#         if self.config['weight_vdw']>0:
#             E_vdw= self.compute_full_clash(coor,other_coor,side_coor)
#         else:
#             E_vdw=0

#         if self.config['weight_fape']>0:
#             E_fape= self.compute_fape_energy_fromquat(rama[:,9:],coor) / len(self.rets)
#         else:
#             E_fape=0
#         if self.config['weight_bond']>0:
#             E_bond= self.compute_bond_energy(coor,other_coor)
#         else:
#             E_bond=0
#         return  E_vdw + E_fape + E_bond + E_pp + E_cc + E_nn + E_pccp + E_cnnc + E_pnnp


#     def obj_func_grad_np(self,rama_):
#         rama=torch.DoubleTensor(rama_)
#         rama.requires_grad=True
#         if rama.grad:
#             rama.grad.zero_()
#         f=self.energy(rama.view(self.L,21))*Scale_factor
#         grad_value=autograd.grad(f,rama)[0]
#         return grad_value.data.numpy().astype(np.float64)
    
#     def obj_func_np(self,rama_):
#         rama=torch.DoubleTensor(rama_)
#         rama=rama.view(self.L,21)
#         with torch.no_grad():
#             f=self.energy(rama)*Scale_factor
#             return f.item()


#     def foldning(self):
#         ilter = self.init_ret
#         # 1) get initial quaternions (double precision)
#         try:
#             init_q = self.init_quat(ilter).double()
#         except:
#             init_q = self.init_quat_safe(ilter).double()

#         # 2) move to target device (GPU if available), enable grad
#         param = init_q.to(device).clone().detach().requires_grad_(True)

#         # 3) set up PyTorch LBFGS optimizer over `param`
#         optimizer = opt.LBFGS(
#             [param],
#             max_iter=self.config.get('max_iter', 300),
#             tolerance_grad=1e-6,
#             tolerance_change=1e-9,
#             history_size=10,
#             line_search_fn='strong_wolfe'
#         )

#         # 4) define the “closure” that LBFGS will call to reevaluate loss + gradients
#         def closure():
#             optimizer.zero_grad()                                 # clear old grads
#             E = self.energy(param.view(self.L,21)) * Scale_factor # compute ∂E/∂param
#             E.backward()
#             return E

#         # 5) run LBFGS until convergence (it calls closure repeatedly)
#         optimizer.step(closure)

#         # 6) write out final PDB
#         final_energy = self.energy(param.view(self.L,21)).item()
#         self.outpdb(param, self.saveprefix + '.pdb', energystr=str(final_energy))


#     def outpdb(self,rama,savefile,start=0,end=10000,energystr=''):
#         # bring baseframes and quaternion data onto CPU to prevent device mismatch
#         basex_cpu = self.basex.detach().cpu()
#         otherx_cpu = self.otherx.detach().cpu()
#         sidex_cpu = self.sidex.detach().cpu()
#         shaped_rama = rama.view(self.L,21).detach().cpu()
#         # compute backbone and other coords
#         coor_np = a2b.quat2b(basex_cpu, shaped_rama[:,9:]).detach().cpu().numpy()
#         other_np = a2b.quat2b(otherx_cpu, shaped_rama[:,9:]).detach().cpu().numpy()
#         coor = torch.FloatTensor(coor_np)
#         # compute side atom coords
#         side_coor_NP = a2b.quat2b(sidex_cpu, torch.cat([shaped_rama[:,:9], coor[:,-1]], dim=-1)).detach().cpu().numpy()
        
#         Atom_name=[' P  '," C4'",' N1 ']
#         Other_Atom_name = [" O5'"," C5'"," C3'"," O3'"," C1'"]
#         other_last_name = ['O',"C","C","O","C"]

#         side_atoms=         [' N1 ',' C2 ',' O2 ',' N2 ',' N3 ',' N4 ',' C4 ',' O4 ',' C5 ',' C6 ',' O6 ',' N6 ',' N7 ',' N8 ',' N9 ']
#         side_last_name =    ['N',      "C",   "O",   "N",   "N",   'N',   'C',   'O',   'C',   'C',   'O',   'N',    'N', 'N','N']

#         base_dict = rigid.base_table()
#         last_name=['P','C','N']
#         wstr=[f'REMARK {str(energystr)}']
#         templet='%6s%5d %4s %3s %1s%4d    %8.3f%8.3f%8.3f%6.2f%6.2f          %2s%2s'
#         count=1
#         for i in range(self.L):
#             if self.seq[i] in ['a','g','A','G']:
#                 Atom_name = [' P  '," C4'",' N9 ']
#                 #atoms = ['P','C4']

#             elif self.seq[i] in ['c','u','C','U']:
#                 Atom_name = [' P  '," C4'",' N1 ']
#             for j in range(coor_np.shape[1]):
#                 outs=('ATOM  ',count,Atom_name[j],self.seq[i],'A',i+1,coor_np[i][j][0],coor_np[i][j][1],coor_np[i][j][2],0,0,last_name[j],'')
#                 if i>=start-1 and i < end:
#                     wstr.append(templet % outs)
#                     count+=1

#             for j in range(other_np.shape[1]):
#                 outs=('ATOM  ',count,Other_Atom_name[j],self.seq[i],'A',i+1,other_np[i][j][0],other_np[i][j][1],other_np[i][j][2],0,0,other_last_name[j],'')
#                 if i>=start-1 and i < end:
#                     wstr.append(templet % outs)
#                     count+=1
            
#         wstr='\n'.join(wstr)
#         wfile=open(savefile,'w')
#         wfile.write(wstr)
#         wfile.close()
    
#     def outpdb_coor(self,coor_np,savefile,start=0,end=1000,energystr=''):
#         Atom_name=[' P  '," C4'",' N1 ']
#         last_name=['P','C','N']
#         wstr=[f'REMARK {str(energystr)}']
#         templet='%6s%5d %4s %3s %1s%4d    %8.3f%8.3f%8.3f%6.2f%6.2f          %2s%2s'
#         count=1
#         for i in range(self.L):
#             if self.seq[i] in ['a','g','A','G']:
#                 Atom_name = [' P  '," C4'",' N9 ']

#             elif self.seq[i] in ['c','u','C','U']:
#                 Atom_name = [' P  '," C4'",' N1 ']
#             for j in range(coor_np.shape[1]):
#                 outs=('ATOM  ',count,Atom_name[j],self.seq[i],'A',i+1,coor_np[i][j][0],coor_np[i][j][1],coor_np[i][j][2],0,0,last_name[j],'')
#                 if i>=start-1 and i < end:
#                     wstr.append(templet % outs)
#                 count+=1
            
#         wstr='\n'.join(wstr)
#         wfile=open(savefile,'w')
#         wfile.write(wstr)
#         wfile.close()


#     def init_quat(self,ii):
#         x = torch.rand([self.L,21])
#         x[:,18:] = self.txs[ii].mean(dim=1)
#         init_coor = self.txs[ii]
#         biasq = torch.mean(init_coor,dim=1,keepdim=True)
#         q = init_coor - biasq
#         m = torch.einsum('bnz,bny->bzy',self.basex,q).reshape([self.L,-1])
#         x[:,:9] = x[:,9:18] = m
#         x.requires_grad_()
#         return x

#     def init_quat_safe(self,ii):
#         x = torch.rand([self.L,21])
#         x[:,18:] = self.txs[ii].mean(dim=1)
#         init_coor = self.txs[ii]
#         biasq = torch.mean(init_coor,dim=1,keepdim=True)
#         q = init_coor - biasq + torch.rand([self.L,3,3])
#         m = (torch.einsum('bnz,bny->bzy',self.basex,q) + torch.eye(3)[None,:,:]).reshape([self.L,-1])
#         x[:,:9] = x[:,9:18] = m
#         x.requires_grad_()
#         return x


# if __name__ == '__main__': 

#     fastafile=sys.argv[1]
#     saveprefix=sys.argv[2]
#     retdirs  =sys.argv[3]
#     ret_score = sys.argv[4]
#     foldconfig = sys.argv[5]

#     savepare = os.path.dirname(saveprefix)
#     if not os.path.isdir(savepare):
#         os.makedirs(savepare)

#     num_of_models = readconfig(foldconfig)['num_of_models']

#     score_dict = readconfig(ret_score)
#     sorted_items = sorted(score_dict.items(), key=lambda x: x[1])
#     lowest_n_keys = [item[0] for item in sorted_items][:num_of_models]
#     bestkey = lowest_n_keys[0] + ''
#     print("Before sort:", lowest_n_keys)
#     lowest_n_keys.sort()
#     print("After sort:", lowest_n_keys)
#     bestindex = lowest_n_keys.index(bestkey)

#     current_ret = bestkey
#     retfiles = [os.path.join(retdirs, afile) for afile in lowest_n_keys]
#     stru = Structure(fastafile, retfiles, saveprefix + '_from_' + current_ret, bestindex, foldconfig)
#     stru.foldning()

In [ ]:
# %%writefile /kaggle/working/DRfold2/PotentialFold/Selection.py
# #! /nfs/amino-home/liyangum/miniconda3/bin/python
# import numpy
# import torch
# import torch.autograd as autograd
# import numpy as np 

# import random
# import Cubic, Potential
# import operations
# import os, json, sys

# import a2b, rigid
# import torch.optim as opt
# from torch.nn.parameter import Parameter
# import torch.nn as nn
# import math
# from scipy.optimize import fmin_l_bfgs_b,fmin_cg,fmin_bfgs
# from scipy.optimize import minimize
# import lbfgs_rosetta
# import pickle
# import shutil

# torch.manual_seed(6)
# torch.set_num_threads(4)
# np.random.seed(9)
# random.seed(9)

# Scale_factor = 1.0
# USEGEO = False

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# def readconfig(configfile=''):
#     config=[]
#     expdir=os.path.dirname(os.path.abspath(__file__))
#     if configfile=='':
#         configfile=os.path.join(expdir,'lib','ddf.json')
#     config=json.load(open(configfile,'r'))
#     return config 

    
# class Structure:
#     def __init__(self, fastafile, geofiles, foldconfig, saveprefix):
#         # Load Configuration and Inputs
#         self.config = readconfig(foldconfig)
#         self.seqfile = fastafile
#         self.foldconfig = foldconfig
#         self.geofiles = geofiles

#         # Load Model Results
#         self.rets = [pickle.load(open(refile, 'rb')) for refile  in geofiles]
        
#         # Extract Coordinates
#         self.txs = []
#         for ret in self.rets:
#             self.txs.append(torch.from_numpy(ret['coor']).double().to(device))
        
#         # Handle Geometrical Data
#         self.handle_geo()

#         # Extract pLDDT Scores
#         self.pair = []
#         for ret in self.rets:
#             self.pair.append( torch.from_numpy(ret['plddt']).double().to(device))
        
#         # Store Output and Sequence Info
#         self.saveprefix = saveprefix
#         self.seq = open(fastafile).readlines()[1].strip()
#         self.L = len(self.seq)
        
#         # Load Reference Arrays for Structure Construction
#         basenpy = np.load(os.path.join(os.path.dirname(os.path.abspath(__file__)), 'lib', 'base.npy'))
#         self.basex = operations.Get_base(self.seq, basenpy).double().to(device)
        
#         othernpy = np.load(os.path.join(os.path.dirname(os.path.abspath(__file__)), 'lib', 'other2.npy'))
#         self.otherx = operations.Get_base(self.seq, othernpy).double().to(device)
        
#         sidenpy = np.load(os.path.join(os.path.dirname(os.path.abspath(__file__)), 'lib', 'side.npy'))
#         self.sidex = operations.Get_base(self.seq, sidenpy).double().to(device)        
        
#         # Initialize Masks, Parameters, and FAPE
#         self.init_mask()
#         self.init_paras()
#         self._init_fape()
    

#     def _init_fape(self):
#         self.tx2ds = []
#         for tx in self.txs:
#             true_rot, true_trans = operations.Kabsch_rigid(self.basex, tx[:, 0], tx[:, 1], tx[:, 2])
#             true_x2 = tx[:, None, :, :] - true_trans[None, :, None, :]
#             true_x2 = torch.einsum('ijnd,jde->ijne', true_x2, true_rot.transpose(-1,-2))
#             self.tx2ds.append(true_x2)
    

#     def handle_geo(self):
#         oldkeys = ['dist_p', 'dist_c', 'dist_n']
#         newkeys = ['pp', 'cc', 'nn']
#         self.geos = []
#         geo = {'pp':0, 'cc':0, 'nn':0}
        
#         for ret in self.rets:    
#             for nk, ok in zip(newkeys, oldkeys):
#                 geo[nk] = geo[nk] + (ret[ok].astype(np.float64) /(len(self.rets)))
#         self.geos.append(geo)


#     def init_mask(self):
#         halfmask=np.zeros([self.L,self.L])
#         fullmask=np.zeros([self.L,self.L])
#         for i in range(self.L):
#             for j in range(i+1,self.L):
#                 halfmask[i,j]=1
#                 fullmask[i,j]=1
#                 fullmask[j,i]=1
#         self.halfmask=torch.DoubleTensor(halfmask) > 0.5
#         self.fullmask=torch.DoubleTensor(fullmask) > 0.5
#         self.clash_mask = torch.zeros([self.L,self.L,22,22])
#         for i in range(self.L):
#             for j in range(i+1,self.L):
#                 self.clash_mask[i,j]=1

#         for i in range(self.L):
#              self.clash_mask[i,i,:6,7:]=1

#         for i in range(self.L-1):
#             self.clash_mask[i,i+1,:,0]=0
#             self.clash_mask[i,i+1,0,:]=0
#             self.clash_mask[i,i+1,:,5]=0
#             self.clash_mask[i,i+1,5,:]=0

#         self.side_mask = rigid.side_mask(self.seq)
#         self.side_mask = self.side_mask[:,None,:,None] * self.side_mask[None,:,None,:]
#         self.clash_mask = (self.clash_mask > 0.5) * (self.side_mask > 0.5)

#         self.geo_confimask_cc = []
#         self.geo_confimask_pp = []
#         self.geo_confimask_nn = []
#         for geo in self.geos:
#             confimask_cc = torch.DoubleTensor(geo['cc'][:,:,-1]) < 0.5
#             confimask_pp = torch.DoubleTensor(geo['pp'][:,:,-1]) < 0.5
#             confimask_nn = torch.DoubleTensor(geo['nn'][:,:,-1]) < 0.5
#             self.geo_confimask_cc.append(confimask_cc)
#             self.geo_confimask_pp.append(confimask_pp)
#             self.geo_confimask_nn.append(confimask_nn)

#         # Move masks and confimasks to the GPU/CPU device
#         self.halfmask = self.halfmask.to(device)
#         self.fullmask = self.fullmask.to(device)
#         self.clash_mask = self.clash_mask.to(device)
#         self.side_mask = self.side_mask.to(device)
#         # geo_confimasks are lists
#         self.geo_confimask_cc = [m.to(device) for m in self.geo_confimask_cc]
#         self.geo_confimask_pp = [m.to(device) for m in self.geo_confimask_pp]
#         self.geo_confimask_nn = [m.to(device) for m in self.geo_confimask_nn]


#     def init_paras(self):
#         self.geo_cc = []
#         self.geo_pp = []
#         self.geo_nn = []
#         self.cs_coefs = {'cc': [], 'pp': [], 'nn': []}
#         self.cs_knots = {'cc': [], 'pp': [], 'nn': []}
#         for geo in self.geos:
#             cc_cs, cc_decs = Cubic.dis_cubic(geo['cc'], 2, 40, 36)
#             pp_cs, pp_decs = Cubic.dis_cubic(geo['pp'], 2, 40, 36)
#             nn_cs, nn_decs = Cubic.dis_cubic(geo['nn'], 2, 40, 36)
#             self.geo_cc.append([cc_cs, cc_decs])
#             self.geo_pp.append([pp_cs, pp_decs])
#             self.geo_nn.append([nn_cs, nn_decs])
#             L = self.L
#             cc_coefs_np = np.stack([[cc_cs[i,j].c for j in range(L)] for i in range(L)], axis=0)
#             cc_knots_np = np.stack([[cc_cs[i,j].x for j in range(L)] for i in range(L)], axis=0)
#             self.cs_coefs['cc'].append(torch.from_numpy(cc_coefs_np).to(device))
#             self.cs_knots['cc'].append(torch.from_numpy(cc_knots_np).to(device))
#             pp_coefs_np = np.stack([[pp_cs[i,j].c for j in range(L)] for i in range(L)], axis=0)
#             pp_knots_np = np.stack([[pp_cs[i,j].x for j in range(L)] for i in range(L)], axis=0)
#             self.cs_coefs['pp'].append(torch.from_numpy(pp_coefs_np).to(device))
#             self.cs_knots['pp'].append(torch.from_numpy(pp_knots_np).to(device))
#             nn_coefs_np = np.stack([[nn_cs[i,j].c for j in range(L)] for i in range(L)], axis=0)
#             nn_knots_np = np.stack([[nn_cs[i,j].x for j in range(L)] for i in range(L)], axis=0)
#             self.cs_coefs['nn'].append(torch.from_numpy(nn_coefs_np).to(device))
#             self.cs_knots['nn'].append(torch.from_numpy(nn_knots_np).to(device))
     

#     def _cubic_pair_energy(self, atom_map, geo_cs, geo_confimask, weight_key):
#         """General cubic-spline energy for CC/PP/NN pairs."""
#         min_dis, max_dis, bin_num = 2, 40, 36
#         dev = atom_map.device
#         upper_th = max_dis - ((max_dis - min_dis) / bin_num) * 0.5
#         lower_th = 2.5
#         total = torch.zeros((), device=dev, dtype=torch.double)
#         spline_key = weight_key.split('_')[1]
#         coeffs_list = self.cs_coefs[spline_key]
#         knots_list = self.cs_knots[spline_key]
#         for block_idx, mask_block in enumerate(geo_confimask):
#             mask = (atom_map <= upper_th) & mask_block & self.fullmask & (atom_map >= lower_th)
#             idx = mask.nonzero(as_tuple=True)
#             if idx[0].numel() > 1:
#                 coef = coeffs_list[block_idx][idx]
#                 knots = knots_list[block_idx][idx]
#                 part1 = Potential.cubic_distance(atom_map[mask], coef, knots, min_dis, max_dis, bin_num).sum() * self.config[weight_key] * 0.5
#             else:
#                 part1 = torch.zeros((), device=dev, dtype=torch.double)
#             part2 = ((atom_map <= lower_th) & mask_block & self.fullmask).sum() * self.config[weight_key]
#             total = total + part1 + part2
#         return total

#     # GPU-friendly torsion and angle energy helpers
#     def _cubic_torsion_energy(self, atom_map, coef, x_vals, weight_key, num_bin):
#         energy = Potential.cubic_torsion(atom_map, coef, x_vals, num_bin)
#         return energy.sum() * self.config[weight_key]

#     def _cubic_angle_energy(self, atom_map, coef, x_vals, weight_key, num_bin):
#         energy = Potential.cubic_angle(atom_map, coef, x_vals, num_bin)
#         return energy.sum() * self.config[weight_key]

#     def compute_cc_energy(self, coor):
#         atom_map = operations.pair_distance(coor[:,1], coor[:,1])
#         return self._cubic_pair_energy(atom_map, self.geo_cc, self.geo_confimask_cc, 'weight_cc')

#     def compute_pp_energy(self, coor):
#         atom_map = operations.pair_distance(coor[:,0], coor[:,0])
#         return self._cubic_pair_energy(atom_map, self.geo_pp, self.geo_confimask_pp, 'weight_pp')

#     def compute_nn_energy(self, coor):
#         atom_map = operations.pair_distance(coor[:,-1], coor[:,-1])
#         return self._cubic_pair_energy(atom_map, self.geo_nn, self.geo_confimask_nn, 'weight_nn')

#     def compute_pccp_energy(self, coor):
#         # P-C-C-P dihedral energy on GPU
#         p = coor[:, 0]
#         c = coor[:, 1]
#         dia = operations.dihedral(
#             p[self.pccpi], c[self.pccpi], c[self.pccpj], p[self.pccpj]
#         )
#         return self._cubic_torsion_energy(dia, self.pccp_coe, self.pccp_x, 'weight_pccp', 36)

#     def compute_cnnc_energy(self, coor):
#         # C-N-N-C dihedral energy on GPU
#         n = coor[:, -1]
#         c = coor[:, 1]
#         dia = operations.dihedral(
#             c[self.cnnci], n[self.cnnci], n[self.cnncj], c[self.cnncj]
#         )
#         return self._cubic_torsion_energy(dia, self.cnnc_coe, self.cnnc_x, 'weight_cnnc', 36)

#     def compute_pnnp_energy(self, coor):
#         # P-N-N-P dihedral energy on GPU
#         n = coor[:, -1]
#         p = coor[:, 0]
#         dia = operations.dihedral(
#             p[self.pnnpi], n[self.pnnpi], n[self.pnnpj], p[self.pnnpj]
#         )
#         return self._cubic_torsion_energy(dia, self.pnnp_coe, self.pnnp_x, 'weight_pnnp', 36)

#     def compute_pcc_energy(self, coor):
#         # P-C-C angle energy on GPU
#         p = coor[:, 1]
#         c = coor[:, 2]
#         ang = operations.angle(
#             p[self.pcci], c[self.pcci], c[self.pccj]
#         )
#         return self._cubic_angle_energy(ang, self.pcc_coe, self.pcc_x, 'weight_pcc', 12)

#     def compute_fape_energy(self,coor,ep=1e-3,epmax=20):
#         energy= 0
#         for tx in self.tx2ds:
#             px_mean = coor[:,[1]]
#             p_rot   = operations.rigidFrom3Points(coor)
#             p_tran  = px_mean[:,0]
#             pred_x2 = coor[:,None,:,:] - p_tran[None,:,None,:] # Lx Lrot N , 3
#             pred_x2 = torch.einsum('ijnd,jde->ijne',pred_x2,p_rot.transpose(-1,-2)) # transpose should be equal to inverse
#             errmap=torch.sqrt( ((pred_x2 - tx)**2).sum(dim=-1) + ep )
#             energy = energy + torch.sum(  torch.clamp(errmap,max=epmax)        )
#         return energy * self.config['weight_fape']

#     def compute_bond_energy(self,coor,other_coor):
#         # 3.87
#         o3 = other_coor[:-1,-2]
#         p  = coor[1:,0]
#         dis = (o3-p).norm(dim=-1)
#         energy = ((dis-1.607)**2).sum()
#         return energy * self.config['weight_bond']

#     def tooth_func(self,errmap, ep = 0.05):
#         return -1/(errmap/10+ep) + (1/ep)
    
#     def reweight_func(self,ww):
#         reweighting = torch.pow(ww,self.config['pair_weight_power'])
#         reweighting[ww < self.config['pair_weight_min']] = 0
#         return reweighting
    
#     def compute_fape_energy_fromquat(self,x,coor,ep=1e-6,epmax=100):
#         energy= 0
#         p_rot,px_mean = a2b.Non2rot(x[:,:9],x.shape[0]),x[:,9:]
#         pred_x2 = coor[:,None,:,:] - px_mean[None,:,None,:] # Lx Lrot N , 3
#         pred_x2 = torch.einsum('ijnd,jde->ijne',pred_x2,p_rot.transpose(-1,-2)) # transpose should be equal to inverse

#         for tx,weightplddt in zip(self.tx2ds,self.pair):
#             tamplate_dist_map = torch.min( tx.norm(dim=-1), dim=2   )[0]
#             errmap=torch.sqrt( ((pred_x2 - tx)**2).sum(dim=-1) + ep ) 
#             energy = energy + torch.sum( ( (torch.clamp(errmap,max=self.config['FAPE_max'])**self.config['pair_error_power'])  * self.reweight_func(weightplddt[...,None]) )[tamplate_dist_map>self.config['pair_rest_min_dist']]    )

#         return energy * self.config['weight_fape']
    
#     def compute_fape_energy_fromcoor(self,coor,ep=1e-6,epmax=100):
#         energy= 0
        
#         p_rot,px_mean = operations.Kabsch_rigid(self.basex,coor[:,0],coor[:,1],coor[:,2])
#         pred_x2 = coor[:,None,:,:] - px_mean[None,:,None,:] # Lx Lrot N , 3
#         pred_x2 = torch.einsum('ijnd,jde->ijne',pred_x2,p_rot.transpose(-1,-2)) # transpose should be equal to inverse
        
#         for tx,weightplddt in zip(self.tx2ds,self.pair):
#             tamplate_dist_map = torch.min( tx.norm(dim=-1), dim=2   )[0]
#             errmap=torch.sqrt( ((pred_x2 - tx)**2).sum(dim=-1) + ep ) 
#             energy = energy + torch.sum( ( (torch.clamp(errmap,max=self.config['FAPE_max'])**self.config['pair_error_power'])  * self.reweight_func(weightplddt[...,None]) )[tamplate_dist_map>self.config['pair_rest_min_dist']]    )

#         return energy * self.config['weight_fape']
    
    
#     def energy(self, rama):
#         coor = a2b.quat2b(self.basex, rama[:, 9:])
#         other_coor = a2b.quat2b(self.otherx, rama[:, 9:])
#         side_coor = a2b.quat2b(self.sidex, torch.cat([rama[:, :9], coor[:, -1]], dim=-1))

#         E_cc = self.compute_cc_energy(coor) / len(self.geofiles) if self.config['weight_cc'] > 0 else 0
#         E_pp = self.compute_pp_energy(coor) / len(self.geofiles) if self.config['weight_pp'] > 0 else 0
#         E_nn = self.compute_nn_energy(coor) / len(self.geofiles) if self.config['weight_nn'] > 0 else 0
#         E_pccp = self.compute_pccp_energy(coor) / len(self.geofiles) if self.config['weight_pccp'] > 0 else 0
#         E_cnnc = self.compute_cnnc_energy(coor) / len(self.geofiles) if self.config['weight_cnnc'] > 0 else 0
#         E_pnnp = self.compute_pnnp_energy(coor) / len(self.geofiles) if self.config['weight_pnnp'] > 0 else 0
#         E_vdw = self.compute_full_clash(coor, other_coor, side_coor) if self.config['weight_vdw'] > 0 else 0
#         E_fape = self.compute_fape_energy_fromquat(rama[:, 9:], coor) / len(self.geofiles) if self.config['weight_fape'] > 0 else 0
#         E_bond = self.compute_bond_energy(coor, other_coor) if self.config['weight_bond'] > 0 else 0

#         return E_vdw + E_fape + E_bond + E_pp + E_cc + E_nn + E_pccp + E_cnnc + E_pnnp


#     def energy_from_coor(self, coor):
#         E_cc = self.compute_cc_energy(coor) if self.config['weight_cc'] > 0 else 0
#         E_pp = self.compute_pp_energy(coor) if self.config['weight_pp'] > 0 else 0
#         E_nn = self.compute_nn_energy(coor) if self.config['weight_nn'] > 0 else 0
#         E_fape = (self.compute_fape_energy_fromcoor(coor) / len(self.geofiles)) if self.config['weight_fape'] > 0 else 0
#         print(E_fape, E_pp, E_cc, E_nn)
#         return E_fape + E_pp + E_cc + E_nn 

#     def obj_func_grad_np(self,rama_):
#         rama=torch.DoubleTensor(rama_)
#         rama.requires_grad=True
#         if rama.grad:
#             rama.grad.zero_()
#         f=self.energy(rama.view(self.L,21))*Scale_factor
#         grad_value=autograd.grad(f,rama)[0]
#         return grad_value.data.numpy().astype(np.float64)
    
#     def obj_func_np(self,rama_):
#         rama=torch.DoubleTensor(rama_)
#         rama=rama.view(self.L,21)
#         with torch.no_grad():
#             f = self.energy(rama)*Scale_factor
#             return f.item()

#     def saveconfig(self,dict,confile):
#         json_object = json.dumps(dict, indent = 4)
#         wfile = open(confile,'w')
#         wfile.write(json_object)
#         wfile.close()
    
#     def scoring(self):
#         geoscale = self.config['geo_scale']
#         self.config['weight_pp'] = geoscale * self.config['weight_pp']
#         self.config['weight_cc'] = geoscale * self.config['weight_cc']
#         self.config['weight_nn'] = geoscale * self.config['weight_nn']
#         self.config['weight_pccp'] = geoscale * self.config['weight_pccp']
#         self.config['weight_cnnc'] = geoscale * self.config['weight_cnnc']
#         self.config['weight_pnnp'] = geoscale * self.config['weight_pnnp']  
        
#         energy_dict = {}
#         saveenergy_dict  = {}
        
#         with torch.no_grad():
#             for retfile, tx in zip(self.geofiles, self.txs):
#                 one = self.energy_from_coor(tx)
#                 aaretfile = os.path.basename(retfile) 
#                 energy_dict[aaretfile] = one.item()
#                 saveenergy_dict[retfile] = one.item()
#             self.saveconfig(energy_dict, self.saveprefix)


#     def foldning(self):
#         minenergy=1e16
#         count=0
#         for tx in self.txs:
#             count+=1
        
#         minirama=None

#         ilter = self.init_ret
#         selected_ret = self.geofiles[ilter]
#         try:
#             rama=self.init_quat(ilter).data.numpy()
#             self.config=readconfig(os.path.join(os.path.dirname(os.path.abspath(__file__)),'lib','vdw.json'))
#             rama = fmin_l_bfgs_b(func=self.obj_func_np, x0=rama,  fprime=self.obj_func_grad_np,iprint=10,maxfun=100)[0]
#             rama = rama.flatten()
#         except:
#             rama=self.init_quat_safe(ilter).data.numpy()
#             self.config=readconfig(os.path.join(os.path.dirname(os.path.abspath(__file__)),'lib','vdw.json'))
#             rama = fmin_l_bfgs_b(func=self.obj_func_np, x0=rama,  fprime=self.obj_func_grad_np,iprint=10,maxfun=100)[0]
#             rama = rama.flatten()
            
#         self.config=readconfig(self.foldconfig)
#         geoscale = self.config['geo_scale']
#         self.config['weight_pp'] =geoscale * self.config['weight_pp']
#         self.config['weight_cc'] =geoscale * self.config['weight_cc']
#         self.config['weight_nn'] =geoscale * self.config['weight_nn']
#         self.config['weight_pccp'] =geoscale * self.config['weight_pccp']
#         self.config['weight_cnnc'] =geoscale * self.config['weight_cnnc']
#         self.config['weight_pnnp'] =geoscale * self.config['weight_pnnp']
#         for i in range(3):
#             line_min = lbfgs_rosetta.ArmijoLineMinimization(self.obj_func_np,self.obj_func_grad_np,True,len(rama),120)
#             lbfgs_opt = lbfgs_rosetta.lbfgs(self.obj_func_np,self.obj_func_grad_np)
#             rama=lbfgs_opt.run(rama,256,lbfgs_rosetta.absolute_converge_test,line_min,8000,self.obj_func_np,self.obj_func_grad_np,1e-9)
#         newrama=rama+0.0
#         newrama=torch.DoubleTensor(newrama) 
#         current_energy =self.obj_func_np(rama)

#         if current_energy < minenergy:
#             print(current_energy,minenergy)
#             minenergy=current_energy
#             self.outpdb(newrama,self.saveprefix+'.pdb',energystr=str(current_energy))


#     def outpdb(self,rama,savefile,start=0,end=10000,energystr=''):
#         coor_np=a2b.quat2b(self.basex,rama.view(self.L,21)[:,9:]).data.numpy()
#         other_np=a2b.quat2b(self.otherx,rama.view(self.L,21)[:,9:]).data.numpy()
#         shaped_rama=rama.view(self.L,21)
#         coor = torch.FloatTensor(coor_np)
#         side_coor_NP = a2b.quat2b(self.sidex,torch.cat([shaped_rama[:,:9],coor[:,-1]],dim=-1)).data.numpy()
        
#         Atom_name=[' P  '," C4'",' N1 ']
#         Other_Atom_name = [" O5'"," C5'"," C3'"," O3'"," C1'"]
#         other_last_name = ['O',"C","C","O","C"]

#         side_atoms=         [' N1 ',' C2 ',' O2 ',' N2 ',' N3 ',' N4 ',' C4 ',' O4 ',' C5 ',' C6 ',' O6 ',' N6 ',' N7 ',' N8 ',' N9 ']
#         side_last_name =    ['N',      "C",   "O",   "N",   "N",   'N',   'C',   'O',   'C',   'C',   'O',   'N',    'N', 'N','N']

#         base_dict = rigid.base_table()
        
#         last_name=['P','C','N']
#         wstr=[f'REMARK {str(energystr)}']
#         templet='%6s%5d %4s %3s %1s%4d    %8.3f%8.3f%8.3f%6.2f%6.2f          %2s%2s'
#         count=1
#         for i in range(self.L):
#             if self.seq[i] in ['a','g','A','G']:
#                 Atom_name = [' P  '," C4'",' N9 ']

#             elif self.seq[i] in ['c','u','C','U']:
#                 Atom_name = [' P  '," C4'",' N1 ']
#             for j in range(coor_np.shape[1]):
#                 outs=('ATOM  ',count,Atom_name[j],self.seq[i],'A',i+1,coor_np[i][j][0],coor_np[i][j][1],coor_np[i][j][2],0,0,last_name[j],'')
#                 if i>=start-1 and i < end:
#                     wstr.append(templet % outs)
#                     count+=1

#             for j in range(other_np.shape[1]):
#                 outs=('ATOM  ',count,Other_Atom_name[j],self.seq[i],'A',i+1,other_np[i][j][0],other_np[i][j][1],other_np[i][j][2],0,0,other_last_name[j],'')
#                 if i>=start-1 and i < end:
#                     wstr.append(templet % outs)
#                     count+=1
            
#         wstr='\n'.join(wstr)
#         wfile=open(savefile,'w')
#         wfile.write(wstr)
#         wfile.close()
    
    
#     def outpdb_coor(self,coor_np,savefile,start=0,end=1000,energystr=''):
#         Atom_name=[' P  '," C4'",' N1 ']
#         last_name=['P','C','N']
#         wstr=[f'REMARK {str(energystr)}']
#         templet='%6s%5d %4s %3s %1s%4d    %8.3f%8.3f%8.3f%6.2f%6.2f          %2s%2s'
#         count=1
#         for i in range(self.L):
#             if self.seq[i] in ['a','g','A','G']:
#                 Atom_name = [' P  '," C4'",' N9 ']

#             elif self.seq[i] in ['c','u','C','U']:
#                 Atom_name = [' P  '," C4'",' N1 ']
            
#             for j in range(coor_np.shape[1]):
#                 outs=('ATOM  ',count,Atom_name[j],self.seq[i],'A',i+1,coor_np[i][j][0],coor_np[i][j][1],coor_np[i][j][2],0,0,last_name[j],'')
#                 if i>=start-1 and i < end:
#                     wstr.append(templet % outs)
#                 count+=1
            
#         wstr='\n'.join(wstr)
#         wfile=open(savefile,'w')
#         wfile.write(wstr)
#         wfile.close()


# if __name__ == '__main__': 

#     fastafile = sys.argv[1]
#     foldconfig = sys.argv[2]
#     save_prefix = sys.argv[3]
#     retfiles = sys.argv[4:]

#     save_parent_dir = os.path.dirname(save_prefix)
#     if not os.path.isdir(save_parent_dir):
#         os.makedirs(save_parent_dir)

#     retfiles.sort()
#     print(retfiles)

#     stru = Structure(fastafile, retfiles, foldconfig, save_prefix)    
#     stru.scoring()

In [ ]:
# %%writefile /kaggle/working/DRfold2/PotentialFold/Cubic.py
# import numpy as np 
# from scipy.interpolate import CubicSpline,UnivariateSpline
# import os
# from torch.autograd import Function
# import torch
# import math

# def fit_dis_cubic(dis_matrix,min_dis,max_dis,num_bin):
#     # convert torch Tensor on GPU to numpy array for SciPy
#     if isinstance(dis_matrix, torch.Tensor):
#         dis_matrix = dis_matrix.detach().cpu().numpy()
#     dis_region=np.zeros(num_bin)
#     for i in range(num_bin):
#         dis_region[i]=min_dis+(i+0.5)*(max_dis-min_dis)*1.0/num_bin
#     L=dis_matrix.shape[0]
#     csnp=[]
#     decsnp=[]
#     for i in range(L):
#         css=[]
#         decss=[]
#         for j in range(L):
#             y=-np.log(      (dis_matrix[i,j,1:-1]+1e-8) / (dis_matrix[i,j,[-2]]+1e-8)              )
#             x=dis_region
#             x[0]=-0.0001
#             y[0]= max(10,y[1]+4)
#             cs= CubicSpline(x,y)
#             decs=cs.derivative()
#             css.append(cs)
#             decss.append(decs)
#         csnp.append(css)
#         decsnp.append(decss)
#     return np.array(csnp),np.array(decsnp)

# def dis_cubic(out,min_dis,max_dis,num_bin):
#     print('fitting cubic distance')
#     cs,decs=fit_dis_cubic(out,min_dis,max_dis,num_bin)
#     return cs,decs



# def cubic_matrix_torsion(dis_matrix,min_dis,max_dis,num_bin):
#     dis_region=np.zeros(num_bin)
#     bin_size=(max_dis-min_dis)/num_bin
#     for i in range(num_bin):
#         dis_region[i]=min_dis+(i+0.5)*(max_dis-min_dis)*1.0/num_bin
#     L=dis_matrix.shape[0]
#     csnp=[]
#     decsnp=[]
#     for i in range(L):
#         css=[]
#         decss=[]
#         for j in range(L):
#             y=-np.log(      dis_matrix[i,j,:-1]+1e-8             )
#             x=dis_region
#             x=np.append(x,x[-1]+bin_size)
#             y=np.append(y,y[0])
#             cs= CubicSpline(x,y,bc_type='periodic')
#             decs=cs.derivative()
#             css.append(cs)
#             decss.append(decs)
#         csnp.append(css)
#         decsnp.append(decss)
#     return np.array(csnp),np.array(decsnp)
# def torsion_cubic(out,min_dis,max_dis,num_bin):
#     print('fitting cubic')
#     cs,decs=cubic_matrix_torsion(out,min_dis,max_dis,num_bin)
#     return cs,decs

# def cubic_matrix_angle(dis_matrix,min_dis,max_dis,num_bin): # 0 - np.pi 12
#     dis_region=np.zeros(num_bin)
#     bin_size=(max_dis-min_dis)/num_bin
#     for i in range(num_bin):
#         dis_region[i]=min_dis+(i+0.5)*(max_dis-min_dis)*1.0/num_bin
#     L=dis_matrix.shape[0]
#     csnp=[]
#     decsnp=[]
#     for i in range(L):
#         css=[]
#         decss=[]
#         for j in range(L):
#             y=-np.log(      dis_matrix[i,j,:-1]+1e-8             )
#             x=dis_region

#             x=np.concatenate([[x[0]-bin_size*3,x[0]-bin_size*2,x[0]-bin_size], x,[x[-1]+bin_size,x[-1]+bin_size*2,x[-1]+bin_size*3]               ])
#             y=np.concatenate([ [y[2],y[1],y[0]],y,[y[-1],y[-2],y[-3]]                                                                                                                    ])

#             cs= CubicSpline(x,y)
#             decs=cs.derivative()

#             css.append(cs)
#             decss.append(decs)
#         csnp.append(css)
#         decsnp.append(decss)

#     return np.array(csnp),np.array(decsnp)
# def angle_cubic(out,min_dis,max_dis,num_bin):

#     print('fitting angle cubic')
#     cs,decs=cubic_matrix_angle(out,min_dis,max_dis,num_bin)

#     return cs,decs

In [ ]:
# # Define an improved function to run DRfold2 that captures output
# def predict_rna_structures_drfold2(sequence, target_id):
#     """Use DRfold2 to predict RNA structures with proper output capture"""
#     import subprocess
#     from subprocess import PIPE, STDOUT
    
#     # Create FASTA file for this sequence
#     fasta_path = os.path.join(fasta_dir, f"{target_id}.fasta")
#     with open(fasta_path, "w") as f:
#         f.write(f">{target_id}\n{sequence}\n")
    
#     # Run DRfold2 with proper output capture
#     output_dir = os.path.join(predictions_dir, target_id)
#     cmd = f"python /kaggle/working/DRfold2/DRfold_infer.py {fasta_path} {output_dir} 1"
    
#     print(f"Running command: {cmd}")
#     process = subprocess.Popen(
#         cmd, 
#         shell=True, 
#         stdout=PIPE, 
#         stderr=STDOUT,
#         universal_newlines=True,
#         bufsize=1
#     )
    
#     # Print output in real-time
#     for line in iter(process.stdout.readline, ''):
#         line = line.strip()
#         if line:
#             print(line)
    
#     # Get return code and check success
#     return_code = process.wait()
#     if return_code != 0:
#         print(f"DRfold2 failed with return code {return_code}")
#         return None
    
#     # Clean up FASTA file to save space
#     os.remove(fasta_path)
    
#     # Extract coordinates
#     relax_dir = os.path.join(output_dir, "relax")
#     if not os.path.isdir(relax_dir):
#         print(f"Warning: No relax directory found for {target_id}")
#         relax_dir = output_dir
    
#     # Get up to 5 PDB files
#     pdb_files = sorted([f for f in os.listdir(relax_dir) if f.endswith(".pdb")])[:5]
    
#     if not pdb_files:
#         print(f"Warning: No PDB files found for {target_id}")
#         # Return None to indicate failure
#         return None
    
#     # Parse PDB files to extract C1' coordinates
#     predictions = []
#     for pdb_file in pdb_files:
#         file_path = os.path.join(relax_dir, pdb_file)
        
#         # Read PDB file
#         coords = []
#         with open(file_path, "r") as f:
#             residue_map = {}
#             for line in f:
#                 if line.startswith("ATOM") and " C1' " in line:
#                     parts = line.split()
#                     resid = int(parts[5])  # Residue ID as integer
#                     x, y, z = float(parts[6]), float(parts[7]), float(parts[8])
#                     residue_map[resid] = (x, y, z)
            
#             # Ensure we have coordinates for all residues
#             for j in range(1, len(sequence) + 1):
#                 if j in residue_map:
#                     coords.append(residue_map[j])
#                 else:
#                     # If residue not found, use zeros
#                     print(f"Warning: Residue {j} not found in {pdb_file} for {target_id}")
#                     coords.append((0.0, 0.0, 0.0))
        
#         predictions.append(coords)
    
#     # Clean up PDB files to save space
#     if is_submission_mode:
#         shutil.rmtree(output_dir)
    
#     # If we have fewer than 5 predictions, duplicate the last one
#     while len(predictions) < 5:
#         predictions.append(predictions[-1] if predictions else [(0.0, 0.0, 0.0) for _ in range(len(sequence))])
    
#     return predictions[:5]  # Return exactly 5 predictions

In [ ]:
# # Vectorized version of process_labels function
# def process_labels_vectorized(labels_df):
#     # Extract target_id from ID column (remove last part after underscore)
#     labels_df = labels_df.copy()
#     labels_df['target_id'] = labels_df['ID'].str.rsplit('_', n=1).str[0]
    
#     # Sort by target_id and resid for proper ordering
#     labels_df = labels_df.sort_values(['target_id', 'resid'])
    
#     # Group by target_id and convert coordinates to arrays
#     coords_dict = {}
#     for target_id, group in labels_df.groupby('target_id'):
#         # Extract coordinates as numpy array in one operation
#         coords_dict[target_id] = group[['x_1', 'y_1', 'z_1']].values
    
#     return coords_dict

# def find_similar_sequences(query_seq, train_seqs_df, train_coords_dict, top_n=5):
#     similar_seqs = []
#     query_seq_obj = Seq(query_seq)

#     for _, row in train_seqs_df.iterrows():
#         target_id = row['target_id']
#         train_seq = row['sequence']

#         # Skip if coordinates not available
#         if target_id not in train_coords_dict:
#             continue

#         # Skip if sequence is too different in length (more than 40% difference)
#         if abs(len(train_seq) - len(query_seq)) / max(len(train_seq), len(query_seq)) > 0.4:
#             continue

#         # Perform sequence alignment
#         alignments = pairwise2.align.globalms(query_seq_obj, train_seq, 2.9, -1, -10, -0.5, one_alignment_only=True)

#         if alignments:
#             alignment = alignments[0]
#             similarity_score = alignment.score / (2 * min(len(query_seq), len(train_seq)))
#             similar_seqs.append((target_id, train_seq, similarity_score, train_coords_dict[target_id]))

#     # Sort by similarity score (higher is better) and return top N
#     similar_seqs.sort(key=lambda x: x[2], reverse=True)
#     return similar_seqs[:top_n]


# def adaptive_rna_constraints(coordinates, sequence, confidence=1.0):
#     # Make a copy of coordinates to refine
#     refined_coords = coordinates.copy()
#     n_residues = len(sequence)
    
#     # Calculate constraint strength (inverse of confidence)
#     # High confidence templates receive gentler constraints
#     constraint_strength = 0.8 * (1.0 - min(confidence, 0.8))
    
#     # 1. Sequential distance constraints (consecutive nucleotides)
#     # More flexible distance range (statistical distribution from PDB)
#     seq_min_dist = 5.5  # Minimum sequential distance
#     seq_max_dist = 6.5  # Maximum sequential distance
    
#     for i in range(n_residues - 1):
#         current_pos = refined_coords[i]
#         next_pos = refined_coords[i+1]
        
#         # Calculate current distance
#         current_dist = np.linalg.norm(next_pos - current_pos)
        
#         # Only adjust if significantly outside expected range
#         if current_dist < seq_min_dist or current_dist > seq_max_dist:
#             # Calculate target distance (midpoint of range)
#             target_dist = (seq_min_dist + seq_max_dist) / 2
            
#             # Get direction vector
#             direction = next_pos - current_pos
#             direction = direction / (np.linalg.norm(direction) + 1e-10)
            
#             # Apply partial adjustment based on constraint strength
#             adjustment = (target_dist - current_dist) * constraint_strength
            
#             # Only adjust the next position to preserve the overall fold
#             refined_coords[i+1] = current_pos + direction * (current_dist + adjustment)
    
#     # 2. Steric clash prevention (more conservative)
#     min_allowed_distance = 3.8  # Minimum distance between non-consecutive C1' atoms
    
#     # Calculate all pairwise distances
#     dist_matrix = distance_matrix(refined_coords, refined_coords)
    
#     # Find severe clashes (atoms too close)
#     severe_clashes = np.where((dist_matrix < min_allowed_distance) & (dist_matrix > 0))
    
#     # Fix severe clashes
#     for idx in range(len(severe_clashes[0])):
#         i, j = severe_clashes[0][idx], severe_clashes[1][idx]
        
#         # Skip consecutive nucleotides and previously processed pairs
#         if abs(i - j) <= 1 or i >= j:
#             continue
            
#         # Get current positions and distance
#         pos_i = refined_coords[i]
#         pos_j = refined_coords[j]
#         current_dist = dist_matrix[i, j]
        
#         # Calculate necessary adjustment but scale by constraint strength
#         direction = pos_j - pos_i
#         direction = direction / (np.linalg.norm(direction) + 1e-10)
        
#         # Calculate partial adjustment
#         adjustment = (min_allowed_distance - current_dist) * constraint_strength
        
#         # Move points apart
#         refined_coords[i] = pos_i - direction * (adjustment / 2)
#         refined_coords[j] = pos_j + direction * (adjustment / 2)
    
#     # 3. Very light base-pair constraining (if confidence is low)
#     if constraint_strength > 0.3:  # Only apply if template confidence is low
#         # Simple Watson-Crick base pairs
#         pairs = {'A': 'U', 'U': 'A', 'G': 'C', 'C': 'G'}
        
#         # Scan for potential base pairs
#         for i in range(n_residues):
#             base_i = sequence[i]
#             complement = pairs.get(base_i)
            
#             if not complement:
#                 continue
                
#             # Look for complementary bases within a reasonable range
#             for j in range(i + 3, min(i + 20, n_residues)):
#                 if sequence[j] == complement:
#                     # Calculate current distance
#                     current_dist = np.linalg.norm(refined_coords[i] - refined_coords[j])
                    
#                     # Only consider if distance suggests potential pairing
#                     if 8.0 < current_dist < 14.0:
#                         # Target 10.5Å as generic base-pair C1'-C1' distance
#                         target_dist = 10.5
                        
#                         # Calculate very gentle adjustment (scaled by constraint_strength)
#                         adjustment = (target_dist - current_dist) * (constraint_strength * 0.3)
                        
#                         # Get direction vector
#                         direction = refined_coords[j] - refined_coords[i]
#                         direction = direction / (np.linalg.norm(direction) + 1e-10)
                        
#                         # Apply very gentle adjustment to both positions
#                         refined_coords[i] = refined_coords[i] - direction * (adjustment / 2)
#                         refined_coords[j] = refined_coords[j] + direction * (adjustment / 2)
                        
#                         # Only consider one potential pair per base (closest match)
#                         break
    
#     return refined_coords

# def adapt_template_to_query(query_seq, template_seq, template_coords, alignment=None):
#     if alignment is None:
#         from Bio.Seq import Seq
#         from Bio import pairwise2
        
#         query_seq_obj = Seq(query_seq)
#         template_seq_obj = Seq(template_seq)
#         alignments = pairwise2.align.globalms(query_seq_obj, template_seq_obj, 2.9, -1, -10, -0.5, one_alignment_only=True)
        
#         if not alignments:
#             return generate_improved_rna_structure(query_seq)
            
#         alignment = alignments[0]
    
#     aligned_query = alignment.seqA
#     aligned_template = alignment.seqB
    
#     query_coords = np.zeros((len(query_seq), 3))
#     query_coords.fill(np.nan)
    
#     # Map template coordinates to query
#     query_idx = 0
#     template_idx = 0
    
#     for i in range(len(aligned_query)):
#         query_char = aligned_query[i]
#         template_char = aligned_template[i]
        
#         if query_char != '-' and template_char != '-':
#             if template_idx < len(template_coords):
#                 query_coords[query_idx] = template_coords[template_idx]
#             template_idx += 1
#             query_idx += 1
#         elif query_char != '-' and template_char == '-':
#             query_idx += 1
#         elif query_char == '-' and template_char != '-':
#             template_idx += 1
    
#     # IMPROVED GAP FILLING - maintains RNA backbone geometry
#     backbone_distance = 5.9  # Typical C1'-C1' distance
    
#     # Fill gaps by maintaining realistic backbone connectivity
#     for i in range(len(query_coords)):
#         if np.isnan(query_coords[i, 0]):
#             # Find nearest valid neighbors
#             prev_valid = next_valid = None
            
#             for j in range(i-1, -1, -1):
#                 if not np.isnan(query_coords[j, 0]):
#                     prev_valid = j
#                     break
                    
#             for j in range(i+1, len(query_coords)):
#                 if not np.isnan(query_coords[j, 0]):
#                     next_valid = j
#                     break
            
#             if prev_valid is not None and next_valid is not None:
#                 # Interpolate along realistic RNA backbone path
#                 gap_size = next_valid - prev_valid
#                 total_distance = np.linalg.norm(query_coords[next_valid] - query_coords[prev_valid])
#                 expected_distance = gap_size * backbone_distance
                
#                 # If gap is compressed, extend it realistically
#                 if total_distance < expected_distance * 0.7:
#                     direction = query_coords[next_valid] - query_coords[prev_valid]
#                     direction = direction / (np.linalg.norm(direction) + 1e-10)
                    
#                     # Place intermediate points along extended path
#                     for k, idx in enumerate(range(prev_valid + 1, next_valid)):
#                         progress = (k + 1) / gap_size
#                         base_pos = query_coords[prev_valid] + direction * expected_distance * progress
                        
#                         # Add slight curvature for realism
#                         perpendicular = np.cross(direction, [0, 0, 1])
#                         if np.linalg.norm(perpendicular) < 1e-6:
#                             perpendicular = np.cross(direction, [1, 0, 0])
#                         perpendicular = perpendicular / (np.linalg.norm(perpendicular) + 1e-10)
                        
#                         curve_amplitude = 2.0 * np.sin(progress * np.pi)
#                         query_coords[idx] = base_pos + perpendicular * curve_amplitude
#                 else:
#                     # Linear interpolation for normal gaps
#                     for k, idx in enumerate(range(prev_valid + 1, next_valid)):
#                         weight = (k + 1) / gap_size
#                         query_coords[idx] = (1 - weight) * query_coords[prev_valid] + weight * query_coords[next_valid]
            
#             elif prev_valid is not None:
#                 # Extend from previous position
#                 if prev_valid > 0 and not np.isnan(query_coords[prev_valid-1, 0]):
#                     direction = query_coords[prev_valid] - query_coords[prev_valid-1]
#                     direction = direction / (np.linalg.norm(direction) + 1e-10)
#                 else:
#                     direction = np.array([1.0, 0.0, 0.0])
                
#                 steps_needed = i - prev_valid
#                 for step in range(1, steps_needed + 1):
#                     pos_idx = prev_valid + step
#                     if pos_idx < len(query_coords):
#                         query_coords[pos_idx] = query_coords[prev_valid] + direction * backbone_distance * step
            
#             elif next_valid is not None:
#                 # Work backwards from next position
#                 direction = np.array([-1.0, 0.0, 0.0])  # Default backward direction
#                 steps_needed = next_valid - i
#                 for step in range(steps_needed, 0, -1):
#                     pos_idx = next_valid - step
#                     if pos_idx >= 0:
#                         query_coords[pos_idx] = query_coords[next_valid] - direction * backbone_distance * step
    
#     # Final cleanup
#     query_coords = np.nan_to_num(query_coords)
#     return query_coords




# def generate_improved_rna_structure(sequence):
#     """
#     Generate a more realistic RNA structure fallback based on sequence patterns
#     and basic RNA structure principles.
    
#     Args:
#         sequence: RNA sequence string
        
#     Returns:
#         Array of 3D coordinates
#     """
#     n_residues = len(sequence)
#     coordinates = np.zeros((n_residues, 3))
    
#     # Analyze sequence to predict structural elements
#     # Look for complementary regions that could form base pairs
#     potential_stems = identify_potential_stems(sequence)
    
#     # Default parameters
#     radius_helix = 10.0
#     radius_loop = 15.0
#     rise_per_residue_helix = 2.5
#     rise_per_residue_loop = 1.5
#     angle_per_residue_helix = 0.6
#     angle_per_residue_loop = 0.3
    
#     # Assign structural classifications
#     structure_types = assign_structure_types(sequence, potential_stems)
    
#     # Generate coordinates based on predicted structure
#     current_pos = np.array([0.0, 0.0, 0.0])
#     current_direction = np.array([0.0, 0.0, 1.0])
#     current_angle = 0.0
    
#     for i in range(n_residues):
#         if structure_types[i] == 'stem':
#             # Part of a helical stem
#             current_angle += angle_per_residue_helix
#             coordinates[i] = [
#                 radius_helix * np.cos(current_angle), 
#                 radius_helix * np.sin(current_angle), 
#                 current_pos[2] + rise_per_residue_helix
#             ]
#             current_pos = coordinates[i]
#         elif structure_types[i] == 'loop':
#             # Part of a loop
#             current_angle += angle_per_residue_loop
#             z_shift = rise_per_residue_loop * np.sin(current_angle * 0.5)
#             coordinates[i] = [
#                 radius_loop * np.cos(current_angle), 
#                 radius_loop * np.sin(current_angle), 
#                 current_pos[2] + z_shift
#             ]
#             current_pos = coordinates[i]
#         else:
#             # Single-stranded region
#             # Add some randomness to make it look more realistic
#             jitter = np.random.normal(0, 1, 3) * 2.0
#             coordinates[i] = current_pos + jitter
#             current_pos = coordinates[i]
            
#     return coordinates

# def identify_potential_stems(sequence):
#     """
#     Identify potential stem regions by looking for self-complementary segments.
    
#     Args:
#         sequence: RNA sequence string
        
#     Returns:
#         List of tuples (start1, end1, start2, end2) representing potentially paired regions
#     """
#     complementary_bases = {'A': 'U', 'U': 'A', 'G': 'C', 'C': 'G'}
#     min_stem_length = 3
#     potential_stems = []
    
#     # Simple stem identification
#     for i in range(len(sequence) - min_stem_length):
#         for j in range(i + min_stem_length + 3, len(sequence) - min_stem_length + 1):
#             # Check if regions could form a stem
#             potential_stem_len = min(min_stem_length, len(sequence) - j)
#             is_stem = True
            
#             for k in range(potential_stem_len):
#                 if sequence[i+k] not in complementary_bases or \
#                    complementary_bases[sequence[i+k]] != sequence[j+potential_stem_len-k-1]:
#                     is_stem = False
#                     break
            
#             if is_stem:
#                 potential_stems.append((i, i+potential_stem_len-1, j, j+potential_stem_len-1))
    
#     return potential_stems

# def assign_structure_types(sequence, potential_stems):
#     """
#     Assign each nucleotide to a structural element type.
    
#     Args:
#         sequence: RNA sequence string
#         potential_stems: List of tuples representing stem regions
        
#     Returns:
#         List of structure types ('stem', 'loop', 'single')
#     """
#     structure_types = ['single'] * len(sequence)
    
#     # Mark stem regions
#     for stem in potential_stems:
#         start1, end1, start2, end2 = stem
#         for i in range(end1 - start1 + 1):
#             structure_types[start1 + i] = 'stem'
#             structure_types[end2 - i] = 'stem'
    
#     # Mark loop regions (regions between paired regions)
#     for i in range(len(potential_stems) - 1):
#         _, end1, start2, _ = potential_stems[i]
#         next_start1, _, _, _ = potential_stems[i+1]
        
#         if next_start1 > end1 + 1 and start2 > next_start1:
#             for j in range(end1 + 1, next_start1):
#                 structure_types[j] = 'loop'
    
#     return structure_types



# # Function to create a more realistic RNA structure when no good templates are found
# def generate_rna_structure(sequence, seed=None):
#     if seed is not None:
#         np.random.seed(seed)
#         random.seed(seed)
    
#     n_residues = len(sequence)
#     coordinates = np.zeros((n_residues, 3))
    
#     # Initialize the first few residues in a helix
#     for i in range(min(3, n_residues)):
#         angle = i * 0.6
#         coordinates[i] = [10.0 * np.cos(angle), 10.0 * np.sin(angle), i * 2.5]
    
#     # Add more complex folding patterns
#     current_direction = np.array([0.0, 0.0, 1.0])  # Start moving along z-axis
    
#     # Define base-pairing tendencies (G-C and A-U pairs)
#     for i in range(3, n_residues):
#         # Check for potential base-pairing in the sequence
#         has_pair = False
#         pair_idx = -1
        
#         # Simple detection of complementary bases (G-C, A-U)
#         complementary = {'G': 'C', 'C': 'G', 'A': 'U', 'U': 'A'}
#         current_base = sequence[i]
        
#         # Look for potential base-pairing within a window before the current position
#         window_size = min(i, 15)  # Look back up to 15 bases
#         for j in range(i-window_size, i):
#             if j >= 0 and sequence[j] == complementary.get(current_base, 'X'):
#                 # Found a potential pair
#                 has_pair = True
#                 pair_idx = j
#                 break
        
#         if has_pair and i - pair_idx <= 10 and random.random() < 0.7:
#             # Try to create a base-pair by positioning this nucleotide near its pair
#             pair_pos = coordinates[pair_idx]
            
#             # Create a position that's roughly opposite to the pair
#             random_offset = np.random.normal(0, 1, 3) * 2.0
#             base_pair_distance = 10.0 + random.uniform(-1.0, 1.0)
            
#             # Calculate a vector from base-pair toward center of structure
#             center = np.mean(coordinates[:i], axis=0)
#             direction = center - pair_pos
#             direction = direction / (np.linalg.norm(direction) + 1e-10)
            
#             # Position new nucleotide in the general direction of the "center"
#             coordinates[i] = pair_pos + direction * base_pair_distance + random_offset
            
#             # Update direction for next nucleotide
#             current_direction = np.random.normal(0, 0.3, 3)
#             current_direction = current_direction / (np.linalg.norm(current_direction) + 1e-10)
            
#         else:
#             # No base-pairing detected, continue with the current fold direction
#             # Randomly rotate current direction to simulate RNA flexibility
#             if random.random() < 0.3:
#                 # More significant direction change
#                 angle = random.uniform(0.2, 0.6)
#                 axis = np.random.normal(0, 1, 3)
#                 axis = axis / (np.linalg.norm(axis) + 1e-10)
#                 rotation = R.from_rotvec(angle * axis)
#                 current_direction = rotation.apply(current_direction)
#             else:
#                 # Small random changes in direction
#                 current_direction += np.random.normal(0, 0.15, 3)
#                 current_direction = current_direction / (np.linalg.norm(current_direction) + 1e-10)
            
#             # Distance between consecutive nucleotides (3.5-4.5Å is typical)
#             step_size = random.uniform(3.5, 4.5)
            
#             # Update position
#             coordinates[i] = coordinates[i-1] + step_size * current_direction
    
#     return coordinates


# def predict_rna_structures(sequence, target_id, train_seqs_df, train_coords_dict, n_predictions=5):
#     predictions = []
    
#     # Find similar sequences in the training data
#     similar_seqs = find_similar_sequences(sequence, train_seqs_df, train_coords_dict, top_n=n_predictions)
    
#     # If we found any similar sequences, use them as templates
#     if similar_seqs:
#         for i, (template_id, template_seq, similarity_score, template_coords) in enumerate(similar_seqs):
#             # Adapt template coordinates to the query sequence
#             adapted_coords = adapt_template_to_query(sequence, template_seq, template_coords)
            
#             if adapted_coords is not None:
#                 # Apply adaptive constraints based on template similarity
#                 # For high similarity templates, apply very gentle constraints
#                 refined_coords = adaptive_rna_constraints(adapted_coords, sequence, confidence=similarity_score)
                
#                 # Add some randomness (less for better templates)
#                 random_scale = max(0.05, 0.8 - similarity_score)  # Reduced randomness
#                 randomized_coords = refined_coords.copy()
#                 randomized_coords += np.random.normal(0, random_scale, randomized_coords.shape)
                
#                 predictions.append(randomized_coords)
                
#                 if len(predictions) >= n_predictions:
#                     break
    
#     # If we don't have enough predictions from templates, generate de novo structures
#     while len(predictions) < n_predictions:
#         seed_value = hash(target_id) % 10000 + len(predictions) * 1000
#         de_novo_coords = generate_rna_structure(sequence, seed=seed_value)
        
#         # Apply stronger constraints to de novo structures (lower confidence)
#         refined_de_novo = adaptive_rna_constraints(de_novo_coords, sequence, confidence=0.2)
        
#         predictions.append(refined_de_novo)
    
#     return predictions[:n_predictions]

In [ ]:
# # Initialize counters and range settings
# if is_submission_mode:
#     DRFOLD_START_IDX = 14
#     DRFOLD_END_IDX = len(test_sequences) - 1
# else:
#     DRFOLD_START_IDX = 0
#     DRFOLD_END_IDX = 0

# drfold_processed = 0
# template_processed = 0

# # train_coords_dict = process_labels_vectorized(train_labels_final)

In [ ]:
# # Sort test sequences by length to process shorter ones with DRfold2
# test_sequences["sequence_len"] = test_sequences["sequence"].str.len()
# # test_sequences = test_sequences[(test_sequences['sequence_len'] >= 200) & (test_sequences['sequence_len'] < 600)]
# test_sequences = test_sequences[test_sequences['sequence_len'] < 100]
# test_sequences.reset_index(drop=True, inplace=True)
# print(test_sequences.shape)

# if not IS_SCORING_RUN:
#     test_sequences = test_sequences.head(5)

# # List to store all prediction records
# all_predictions = []

# # For each sequence in the test set
# for idx, row in test_sequences.iterrows():
#     target_id = row['target_id']
#     sequence = row['sequence']
    
#     # Generate 5 different structure predictions
#     print(f"Using DRfold2 for target {target_id} (index {idx})")
#     predictions = predict_rna_structures_drfold2(sequence, target_id)
    
#     # For each residue in the sequence
#     for j in range(len(sequence)):
#         pred_row = {
#             'ID': f"{target_id}_{j+1}",
#             'resname': sequence[j],
#             'resid': j + 1
#         }
        
#         # Add coordinates from all 5 predictions
#         for i in range(5):
#             pred_row[f'x_{i+1}'] = predictions[i][j][0]
#             pred_row[f'y_{i+1}'] = predictions[i][j][1]
#             pred_row[f'z_{i+1}'] = predictions[i][j][2]
        
#         all_predictions.append(pred_row)
    
#     # Free up memory
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

# # Create DataFrame with predictions
# drfold_pred = pd.DataFrame(all_predictions)

# # Ensure the submission file has the correct format
# column_order = ['ID', 'resname', 'resid']
# for i in range(1, 6):
#     for coord in ['x', 'y', 'z']:
#         column_order.append(f'{coord}_{i}')
        
# drfold_pred = drfold_pred[column_order]

# # Clean the working directory before saving
# print("Cleaning working directory...")
# for item in os.listdir("/kaggle/working/"):
#     item_path = os.path.join("/kaggle/working/", item)
#     if os.path.isfile(item_path) and item != "submission.csv":
#         os.remove(item_path)
#     elif os.path.isdir(item_path) and item != "predictions" and item != "fasta_files" and item != "DRfold2":
#         shutil.rmtree(item_path)

# # Save the submission
# drfold_pred.to_csv('/kaggle/working/drfold_submission.csv', index=False)
# print(f"Submission file saved to /kaggle/working/drfold_submission.csv")

In [ ]:
# drfold_pred

# TBM

In [ ]:
!pip install --no-index /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl

In [ ]:
import pandas as pd
import numpy as np
import random
import time
import warnings
import os, sys

warnings.filterwarnings('ignore')

DATA_PATH = '/kaggle/input/stanford-rna-3d-folding-2/'
train_seqs = pd.read_csv(DATA_PATH + 'train_sequences.csv')
test_seqs = pd.read_csv(DATA_PATH + 'test_sequences.csv')
train_labels = pd.read_csv(DATA_PATH + 'train_labels.csv')

sys.path.append(os.path.join(DATA_PATH, "extra"))

# --- Robust import for Kaggle's extra/parse_fasta_py.py (it may miss typing imports) ---
try:
    import typing as _typing
    import builtins as _builtins

    # Make these names available during module import-time annotation evaluation
    _builtins.Dict  = getattr(_typing, "Dict")
    _builtins.Tuple = getattr(_typing, "Tuple")
    _builtins.List  = getattr(_typing, "List")

    from parse_fasta_py import parse_fasta as _parse_fasta_raw

    # Normalize output to: {chain_id: sequence_string}
    def parse_fasta(fasta_content: str):
        d = _parse_fasta_raw(fasta_content)
        out = {}
        for k, v in d.items():
            # some variants return (sequence, headers/lines) or similar
            out[k] = v[0] if isinstance(v, tuple) else v
        return out

except Exception:
    # Fallback FASTA parser: {chain_id: sequence_string}
    def parse_fasta(fasta_content: str):
        out = {}
        cur = None
        seq_parts = []
        for line in str(fasta_content).splitlines():
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur is not None:
                    out[cur] = "".join(seq_parts)
                header = line[1:]
                # First token is usually chain id in this dataset
                cur = header.split()[0]
                seq_parts = []
            else:
                seq_parts.append(line.replace(" ", ""))
        if cur is not None:
            out[cur] = "".join(seq_parts)
        return out

def parse_stoichiometry(stoich: str):
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    out = []
    for part in str(stoich).split(';'):
        ch, cnt = part.split(':')
        out.append((ch.strip(), int(cnt)))
    return out

def get_chain_segments(row):
    """
    Returns list of (start,end) segments in row['sequence'] corresponding to chain copies in stoichiometry order.
    Falls back to single segment if parsing fails.
    """
    seq = row['sequence']
    stoich = row.get('stoichiometry', '')
    all_seq = row.get('all_sequences', '')

    if pd.isna(stoich) or pd.isna(all_seq) or str(stoich).strip()=="" or str(all_seq).strip()=="":
        return [(0, len(seq))]

    try:
        chain_dict = parse_fasta(all_seq)  # dict: chain_id -> sequence
        order = parse_stoichiometry(stoich)
        segs = []
        pos = 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None:
                return [(0, len(seq))]
            for _ in range(cnt):
                L = len(base)
                segs.append((pos, pos + L))
                pos += L
        if pos != len(seq):
            return [(0, len(seq))]
        return segs
    except Exception:
        return [(0, len(seq))]

def build_segments_map(df):
    seg_map = {}
    stoich_map = {}
    for _, r in df.iterrows():
        tid = r['target_id']
        seg_map[tid] = get_chain_segments(r)
        stoich_map[tid] = str(r.get('stoichiometry', '') if not pd.isna(r.get('stoichiometry', '')) else '')
    return seg_map, stoich_map

train_segs_map, train_stoich_map = build_segments_map(train_seqs)
test_segs_map,  test_stoich_map  = build_segments_map(test_seqs)

def process_labels(labels_df):
    coords_dict = {}
    # Faster + safer prefix extraction
    prefixes = labels_df['ID'].str.rsplit('_', n=1).str[0]
    for id_prefix, group in labels_df.groupby(prefixes):
        coords_dict[id_prefix] = group.sort_values('resid')[['x_1', 'y_1', 'z_1']].values
    return coords_dict

train_coords_dict = process_labels(train_labels)

from Bio.Align import PairwiseAligner

aligner = PairwiseAligner()
aligner.mode = 'global'
aligner.match_score = 2
aligner.mismatch_score = -1.5

# Stronger gap penalties discourage "sliding" (critical: residue numbering must match)
aligner.open_gap_score   = -8
aligner.extend_gap_score = -0.4

# Also penalize terminal gaps (prevents end-gap semi-global behavior)
aligner.query_left_open_gap_score  = -8
aligner.query_left_extend_gap_score = -0.4
aligner.query_right_open_gap_score = -8
aligner.query_right_extend_gap_score = -0.4
aligner.target_left_open_gap_score = -8
aligner.target_left_extend_gap_score = -0.4
aligner.target_right_open_gap_score = -8
aligner.target_right_extend_gap_score = -0.4

def find_similar_sequences(query_seq, train_seqs_df, train_coords_dict, top_n=5):
    similar_seqs = []
    
    # Pre-filter: Iterate only valid targets
    # Note: aligner.score is much faster than generating full alignments
    for _, row in train_seqs_df.iterrows():
        target_id, train_seq = row['target_id'], row['sequence']
        if target_id not in train_coords_dict: continue
        
        # Length filter (keep your original logic)
        if abs(len(train_seq) - len(query_seq)) / max(len(train_seq), len(query_seq)) > 0.3: continue
        
        # FAST SCORE: Calculates score without traceback overhead
        raw_score = aligner.score(query_seq, train_seq)
        
        normalized_score = raw_score / (2 * min(len(query_seq), len(train_seq)))
        similar_seqs.append((target_id, train_seq, normalized_score, train_coords_dict[target_id]))
    
    similar_seqs.sort(key=lambda x: x[2], reverse=True)
    return similar_seqs[:top_n]

def adapt_template_to_query(query_seq, template_seq, template_coords):
    # Generate the alignment object
    # aligner.align returns an iterator; we take the first optimal alignment
    alignment = next(iter(aligner.align(query_seq, template_seq)))
    
    new_coords = np.full((len(query_seq), 3), np.nan)
    
    # VECTORIZED MAPPING:
    # alignment.aligned returns lists of (start, end) tuples for matched segments.
    # This avoids the slow python loop "for char_q, char_t in zip..."
    for (q_start, q_end), (t_start, t_end) in zip(*alignment.aligned):
        # Map the coordinate chunk directly
        t_chunk = template_coords[t_start:t_end]
        
        # Safety check to ensure shapes match (handles edge cases)
        if len(t_chunk) == (q_end - q_start):
            new_coords[q_start:q_end] = t_chunk

    # --- Interpolation Logic (Unchanged) ---
    for i in range(len(new_coords)):
        if np.isnan(new_coords[i, 0]):
            prev_v = next((j for j in range(i-1, -1, -1) if not np.isnan(new_coords[j, 0])), -1)
            next_v = next((j for j in range(i+1, len(new_coords)) if not np.isnan(new_coords[j, 0])), -1)
            if prev_v >= 0 and next_v >= 0:
                w = (i - prev_v) / (next_v - prev_v)
                new_coords[i] = (1-w)*new_coords[prev_v] + w*new_coords[next_v]
            elif prev_v >= 0: new_coords[i] = new_coords[prev_v] + [3, 0, 0]
            elif next_v >= 0: new_coords[i] = new_coords[next_v] + [3, 0, 0]
            else: new_coords[i] = [i*3, 0, 0]
            
    return np.nan_to_num(new_coords)

def adaptive_rna_constraints(coordinates, target_id, confidence=1.0, passes=2):
    """
    Evaluation-driven constraints:
    - US-align is show-only rigid body => internal geometry errors are fatal
    - apply within each chain segment (no fake bond across chain breaks)
    """
    coords = coordinates.copy()
    segments = test_segs_map.get(target_id, [(0, len(coords))])

    # stronger corrections when confidence is low
    strength = 0.75 * (1.0 - min(confidence, 0.97))
    strength = max(strength, 0.02)

    for _ in range(passes):
        for (s, e) in segments:
            X = coords[s:e]
            L = e - s
            if L < 3:
                coords[s:e] = X
                continue

            # (1) bond i,i+1 to ~5.95Å (vectorized, symmetric)
            d = X[1:] - X[:-1]
            dist = np.linalg.norm(d, axis=1) + 1e-6
            target = 5.95
            scale = (target - dist) / dist
            adj = (d * scale[:, None]) * (0.22 * strength)
            X[:-1] -= adj
            X[1:]  += adj

            # (2) soft i,i+2 to ~10.2Å (vectorized, symmetric)
            d2 = X[2:] - X[:-2]
            dist2 = np.linalg.norm(d2, axis=1) + 1e-6
            target2 = 10.2
            scale2 = (target2 - dist2) / dist2
            adj2 = (d2 * scale2[:, None]) * (0.10 * strength)
            X[:-2] -= adj2
            X[2:]  += adj2

            # (3) Laplacian smoothing (removes kinks US-align cannot fix)
            lap = 0.5 * (X[:-2] + X[2:]) - X[1:-1]
            X[1:-1] += (0.06 * strength) * lap

            # (4) light self-avoidance (prevents steric collapse)
            if L >= 25:
                k = min(L, 160) if L > 220 else L
                if k < L:
                    idx = np.linspace(0, L - 1, k).astype(int)
                else:
                    idx = np.arange(L)

                P = X[idx]
                diff = P[:, None, :] - P[None, :, :]
                distm = np.linalg.norm(diff, axis=2) + 1e-6
                sep = np.abs(idx[:, None] - idx[None, :])

                mask = (sep > 2) & (distm < 3.2)
                if np.any(mask):
                    force = (3.2 - distm) / distm
                    vec = (diff * force[:, :, None] * mask[:, :, None]).sum(axis=1)
                    X[idx] += (0.015 * strength) * vec

            coords[s:e] = X

    return coords

def _rotmat(axis, ang):
    axis = np.asarray(axis, float)
    axis = axis / (np.linalg.norm(axis) + 1e-12)
    x, y, z = axis
    c, s = np.cos(ang), np.sin(ang)
    C = 1.0 - c
    return np.array([
        [c + x*x*C,     x*y*C - z*s, x*z*C + y*s],
        [y*x*C + z*s,   c + y*y*C,   y*z*C - x*s],
        [z*x*C - y*s,   z*y*C + x*s, c + z*z*C]
    ], dtype=float)

def apply_hinge(coords, seg, rng, max_angle_deg=25):
    s, e = seg
    L = e - s
    if L < 30:
        return coords
    pivot = s + int(rng.integers(10, L - 10))
    axis = rng.normal(size=3)
    ang = np.deg2rad(float(rng.uniform(-max_angle_deg, max_angle_deg)))
    R = _rotmat(axis, ang)
    X = coords.copy()
    p0 = X[pivot].copy()
    X[pivot+1:e] = (X[pivot+1:e] - p0) @ R.T + p0
    return X

def jitter_chains(coords, segments, rng, max_angle_deg=12, max_trans=1.5):
    X = coords.copy()
    global_center = X.mean(axis=0, keepdims=True)
    for (s, e) in segments:
        axis = rng.normal(size=3)
        ang = np.deg2rad(float(rng.uniform(-max_angle_deg, max_angle_deg)))
        R = _rotmat(axis, ang)
        shift = rng.normal(size=3)
        shift = shift / (np.linalg.norm(shift) + 1e-12) * float(rng.uniform(0.0, max_trans))
        c = X[s:e].mean(axis=0, keepdims=True)
        X[s:e] = (X[s:e] - c) @ R.T + c + shift
    # recenter
    X -= X.mean(axis=0, keepdims=True) - global_center
    return X

def smooth_wiggle(coords, segments, rng, amp=0.8):
    X = coords.copy()
    for (s, e) in segments:
        L = e - s
        if L < 20:
            continue
        n_ctrl = 6
        ctrl_x = np.linspace(0, L - 1, n_ctrl)
        ctrl_disp = rng.normal(0, amp, size=(n_ctrl, 3))
        t = np.arange(L)
        disp = np.vstack([np.interp(t, ctrl_x, ctrl_disp[:, k]) for k in range(3)]).T
        X[s:e] += disp
    return X

def predict_rna_structures(row, train_seqs_df, train_coords_dict, n_predictions=5):
    tid = row['target_id']
    seq = row['sequence']

    # Data constraint: should already be canonical A/C/G/U
    assert set(seq).issubset(set("ACGU")), f"Non-ACGU in {tid}; do not remap here."

    segments = test_segs_map.get(tid, [(0, len(seq))])

    # Grab a larger candidate pool, then sample for diversity (best-of-5)
    cands = find_similar_sequences(query_seq=seq, train_seqs_df=train_seqs_df, train_coords_dict=train_coords_dict, top_n=30)
    assert all(len(c[3]) == len(c[1]) for c in cands), "Template coords/seq length mismatch"
    predictions = []
    used = set()

    for i in range(n_predictions):
        seed = (abs(hash(tid)) + i * 10007) % (2**32)
        rng = np.random.default_rng(seed)

        if not cands:
            # hard fallback (straight line per chain)
            coords = np.zeros((len(seq), 3), dtype=float)
            for (s, e) in segments:
                for j in range(s+1, e):
                    coords[j] = coords[j-1] + [5.95, 0, 0]
            predictions.append(coords)
            continue

        # Choose template:
        # i=0 => best template; others => sample among top-K with weights, avoid duplicates
        if i == 0:
            t_id, t_seq, sim, t_coords = cands[0]
        else:
            K = min(12, len(cands))
            sims = np.array([cands[k][2] for k in range(K)], float)
            w = np.exp((sims - sims.max()) / 0.08)
            # penalize already used templates
            for k in range(K):
                if cands[k][0] in used:
                    w[k] *= 0.10
            w = w / (w.sum() + 1e-12)
            k = int(rng.choice(np.arange(K), p=w))
            t_id, t_seq, sim, t_coords = cands[k]

        used.add(t_id)

        # Transfer coords with diagonal-guard mapping (no sliding)
        adapted = adapt_template_to_query(query_seq=seq, template_seq=t_seq, template_coords=t_coords)

        # Diversity transforms (then re-refine constraints)
        if i == 0:
            X = adapted
        elif i == 1:
            # mild noise
            X = adapted + rng.normal(0, max(0.01, (0.40 - sim) * 0.06), adapted.shape)
        elif i == 2:
            # hinge within the longest chain
            longest = max(segments, key=lambda se: se[1] - se[0])
            X = apply_hinge(adapted, longest, rng, max_angle_deg=22)
        elif i == 3:
            # inter-chain jitter (small, safe)
            X = jitter_chains(adapted, segments, rng, max_angle_deg=10, max_trans=1.0)
        else:
            # smooth low-frequency deformation
            X = smooth_wiggle(adapted, segments, rng, amp=0.7)

        refined = adaptive_rna_constraints(X, tid, confidence=sim, passes=2)
        predictions.append(refined)

    return predictions

all_predictions = []
start_time = time.time()
for idx, row in test_seqs.iterrows():
    if idx % 10 == 0: print(f"Processing {idx} | {time.time()-start_time:.1f}s")
    tid, seq = row['target_id'], row['sequence']
    preds = predict_rna_structures(row, train_seqs, train_coords_dict)
    for j in range(len(seq)):
        res = {'ID': f"{tid}_{j+1}", 'resname': seq[j], 'resid': j+1}
        for i in range(5):
            res[f'x_{i+1}'], res[f'y_{i+1}'], res[f'z_{i+1}'] = preds[i][j]
        all_predictions.append(res)

sub = pd.DataFrame(all_predictions)
cols = ['ID', 'resname', 'resid'] + [f'{c}_{i}' for i in range(1,6) for c in ['x','y','z']]

# Safety: competition clips coords; do it explicitly to avoid out-of-range explosions
coord_cols = [c for c in cols if c.startswith(('x_','y_','z_'))]
sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)

# sub[cols].to_csv('submission.csv', index=False)
# print("submission.csv! saved")

In [ ]:
pred_tbm = sub

In [ ]:
# print('Submission.csv shape:', rosetta_pred.shape)
pred_tbm.to_csv('/kaggle/working/pred_tbm.csv', index=False)

In [ ]:
pred_tbm

# RNAPro

In [ ]:
!cp -r /kaggle/input/datasets/theoviel/rnapro-src/RNAPro .
!cp /kaggle/input/datasets/theoviel/rnapro-src/rnapro-private-best-500m.ckpt .

In [ ]:
cd RNAPro

In [ ]:
!pip install -e . --no-deps

In [ ]:
DIST = "/kaggle/working/RNAPro/release_data/ccd_cache/"
!mkdir -p $DIST

In [ ]:
# updated file paths
!cp /kaggle/input/datasets/jaejohn/rnapro-ccd-cache/ccd_cache/components.cif $DIST
!cp /kaggle/input/datasets/jaejohn/rnapro-ccd-cache/ccd_cache/components.cif.rdkit_mol.pkl $DIST

In [ ]:
!pip install /kaggle/input/datasets/tobimichigan/biotite-1-2/biotite-1.2.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl --no-deps

In [ ]:
!python preprocess/convert_templates_to_pt_files.py --input_csv /kaggle/working/pred_tbm.csv --output_name templates.pt

In [ ]:
IS_SCORING_RUN = os.environ.get('KAGGLE_IS_COMPETITION_RERUN')
print(IS_SCORING_RUN)

In [ ]:
# %%python
df = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")
df["sequence_len"] = df["sequence"].str.len()
# df = df[(df['sequence_len'] >= 200) & (df['sequence_len'] < 600)]
# df = df[df['sequence_len'] < 600]
df = df[df['sequence_len'] < 1000]
df.reset_index(drop=True, inplace=True)
print(df.shape)
if not IS_SCORING_RUN:
    # df = df.head(5)
    TEST_TARGETS = "8ZNQ,9ZCC"
    keep = {t.strip() for t in TEST_TARGETS.split(",") if t.strip()}
    df = df[df["target_id"].isin(keep)].reset_index(drop=True)
    print(df.shape)
    if df.empty:
        raise ValueError("TEST_TARGETS did not match any target_id values")
df.to_csv('/kaggle/working/sample_sequences.csv', index=False)

In [ ]:
%%writefile runner/inference.py
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import os
import shutil
import logging
import traceback
import warnings
import argparse
from contextlib import nullcontext
from os.path import join as opjoin
from typing import Any, Mapping

import json
import torch
import pandas as pd
import numpy as np
import gc
from biotite.structure.io import pdbx

from configs.configs_base import configs as configs_base
from configs.configs_data import data_configs
from configs.configs_inference import inference_configs
from runner.dumper import DataDumper

from rnapro.config import parse_sys_args
from rnapro.config.config import ConfigManager, ArgumentNotSet
from rnapro.data.infer_data_pipeline import get_inference_dataloader
from rnapro.model.RNAPro import RNAPro
from rnapro.utils.distributed import DIST_WRAPPER
from rnapro.utils.seed import seed_everything
from rnapro.utils.torch_utils import to_device

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

logger = logging.getLogger(__name__)

logging.basicConfig(level=logging.INFO)
logging.getLogger("rnapro.data").setLevel(logging.INFO)
logging.getLogger("rnapro").setLevel(logging.INFO)


def parse_configs(configs: dict, arg_str: str = None, fill_required_with_null: bool = False):
    manager = ConfigManager(configs, fill_required_with_null=fill_required_with_null)
    parser = argparse.ArgumentParser()
    parser.add_argument("--max_len", type=int, default=1000, required=False)

    for key, (dtype, default_value, allow_none, required) in manager.config_infos.items():
        parser.add_argument("--" + key, type=str, default=ArgumentNotSet(), required=required)
    merged_configs = manager.merge_configs(vars(parser.parse_args(arg_str.split())) if arg_str else {})
    merged_configs.max_len = parser.parse_args(arg_str.split()).max_len
    return merged_configs

class dotdict(dict):
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__
    def __getattr__(self, name):
        try: return self[name]
        except KeyError: raise AttributeError(name)

class InferenceRunner(object):
    def __init__(self, configs: Any) -> None:
        self.configs = configs
        self.init_env()
        self.init_basics()
        self.init_model()
        self.load_checkpoint()
        self.init_dumper(need_atom_confidence=configs.need_atom_confidence, sorted_by_ranking_score=configs.sorted_by_ranking_score)

    def init_env(self) -> None:
        self.use_cuda = torch.cuda.device_count() > 0
        if self.use_cuda:
            self.device = torch.device("cuda:{}".format(DIST_WRAPPER.local_rank))
            torch.cuda.set_device(self.device)
        else:
            self.device = torch.device("cpu")

    def init_basics(self) -> None:
        self.dump_dir = self.configs.dump_dir
        self.error_dir = opjoin(self.dump_dir, "ERR")
        os.makedirs(self.dump_dir, exist_ok=True)
        os.makedirs(self.error_dir, exist_ok=True)

    def init_model(self) -> None:
        self.model = RNAPro(self.configs).to(self.device)

    def load_checkpoint(self) -> None:
        checkpoint_path = self.configs.load_checkpoint_path
        checkpoint = torch.load(checkpoint_path, self.device)
        sample_key = [k for k in checkpoint["model"].keys()][0]
        if sample_key.startswith("module."):
            checkpoint["model"] = {k[len("module."):]: v for k, v in checkpoint["model"].items()}
        self.model.load_state_dict(state_dict=checkpoint["model"], strict=True)
        self.model.eval()

    def init_dumper(self, need_atom_confidence: bool = False, sorted_by_ranking_score: bool = True):
        self.dumper = DataDumper(base_dir=self.dump_dir, need_atom_confidence=need_atom_confidence, sorted_by_ranking_score=sorted_by_ranking_score)

    @torch.no_grad()
    def predict(self, data: Mapping[str, Mapping[str, Any]]) -> dict[str, torch.Tensor]:
        eval_precision = {"fp32": torch.float32, "bf16": torch.bfloat16, "fp16": torch.float16}[self.configs.dtype]
        enable_amp = torch.autocast(device_type="cuda", dtype=eval_precision) if torch.cuda.is_available() else nullcontext()
        data = to_device(data, self.device)
        with enable_amp:
            prediction, _, _ = self.model(input_feature_dict=data["input_feature_dict"], label_full_dict=None, label_dict=None, mode="inference")
        return prediction

    def update_model_configs(self, new_configs: Any) -> None:
        self.model.configs = new_configs


def update_inference_configs(configs: Any, N_token: int):
    # Force enable AMP (skip_amp=False) to save memory
    configs.skip_amp.confidence_head = False
    configs.skip_amp.sample_diffusion = False
    return configs


def infer_predict(runner: InferenceRunner, configs: Any) -> None:
    try:
        dataloader = get_inference_dataloader(configs=configs)
    except Exception as e:
        logger.error(f"Dataloader error: {e}")
        traceback.print_exc()
        return

    # --- Patch for chunked inference ---
    if getattr(configs, "chunk_target_id", None) and getattr(configs, "chunk_span", None):
        chunk_id = configs.chunk_target_id
        original_id = chunk_id.split("_chk")[0]
        start, end = configs.chunk_span
        dataset = dataloader.dataset
        if hasattr(dataset, "template_features"):
            if original_id in dataset.template_features:
                orig_feats = dataset.template_features[original_id]
                # Slice template features
                chunk_feats = {}
                logger.info(f"Processing template features for {chunk_id} (span: {start}-{end})")
                
                for k, v in orig_feats.items():
                    if isinstance(v, (torch.Tensor, np.ndarray)):
                        try:
                            shape = v.shape
                            logger.info(f"  Feature '{k}': shape={shape}")
                            
                            # Case 1: (N, L, L, ...)
                            if len(shape) >= 3 and shape[1] == shape[2] and shape[1] >= end:
                                chunk_feats[k] = v[:, start:end, start:end]
                                logger.info(f"    -> Sliced dim 1,2: {chunk_feats[k].shape}")
                            # Case 2: (L, L, ...)
                            elif len(shape) >= 2 and shape[0] == shape[1] and shape[0] >= end:
                                chunk_feats[k] = v[start:end, start:end]
                                logger.info(f"    -> Sliced dim 0,1: {chunk_feats[k].shape}")
                            # Case 3: (N, L, ...)
                            elif len(shape) >= 2 and shape[1] >= end:
                                chunk_feats[k] = v[:, start:end]
                                logger.info(f"    -> Sliced dim 1: {chunk_feats[k].shape}")
                            # Case 4: (L, ...)
                            elif len(shape) >= 1 and shape[0] >= end:
                                chunk_feats[k] = v[start:end]
                                logger.info(f"    -> Sliced dim 0: {chunk_feats[k].shape}")
                            else:
                                chunk_feats[k] = v
                                logger.info(f"    -> Kept original")
                        except IndexError as e:
                            logger.warning(f"IndexError slicing template feature {k} for {chunk_id}: {e}")
                            chunk_feats[k] = v
                    else:
                        logger.info(f"  Feature '{k}': type={type(v)} (not sliced)")
                        chunk_feats[k] = v
                
                dataset.template_features[chunk_id] = chunk_feats
                logger.info(f"Mapped and sliced template for {chunk_id} from {original_id} ({start}:{end})")
            else:
                logger.warning(f"Original template {original_id} not found for chunk {chunk_id}")

    for seed in configs.seeds:
        seed_everything(seed=seed, deterministic=configs.deterministic)
        for batch in dataloader:
            try:
                data, atom_array, data_error_message = batch[0]
                sample_name = data["sample_name"]
                if len(data_error_message) > 0: continue

                new_configs = update_inference_configs(configs, data["N_token"].item())
                runner.update_model_configs(new_configs)
                prediction = runner.predict(data)
                runner.dumper.dump(
                    dataset_name="", pdb_id=sample_name, seed=seed,
                    pred_dict=prediction, atom_array=atom_array, entity_poly_type=data["entity_poly_type"]
                )
                
                # Explicitly cleanup memory
                del prediction
                del data
                torch.cuda.empty_cache()
                gc.collect()
                
            except Exception as e:
                logger.error(f"Inference error for {sample_name}: {e}")
                traceback.print_exc()
                if hasattr(torch.cuda, "empty_cache"): torch.cuda.empty_cache()
                gc.collect()


def make_dummy_solution(valid_df):
    solution = dotdict()
    for i, row in valid_df.iterrows():
        solution[row.target_id] = dotdict(target_id=row.target_id, sequence=row.sequence, coord=[])
    return solution

def solution_to_submit_df(solution):
    submit_df = []
    for k, s in solution.items():
        L = len(s.sequence)
        df = pd.DataFrame()
        df["ID"] = [f"{s.target_id}_{i + 1}" for i in range(L)]
        df["resname"] = list(s.sequence)
        df["resid"] = [i + 1 for i in range(L)]
        for j in range(len(s.coord)):
            df[f"x_{j+1}"] = s.coord[j][:, 0]
            df[f"y_{j+1}"] = s.coord[j][:, 1]
            df[f"z_{j+1}"] = s.coord[j][:, 2]
        submit_df.append(df)
    return pd.concat(submit_df)


def extract_c1_coordinates(cif_file_path):
    try:
        with open(cif_file_path, "r") as f:
            cif_data = pdbx.CIFFile.read(f)
        atom_array = pdbx.get_structure(cif_data, model=1)
        mask_c1 = np.char.strip(atom_array.atom_name.astype(str)) == "C1'"
        c1_atoms = atom_array[mask_c1]
        sort_indices = np.argsort(c1_atoms.res_id)
        return c1_atoms[sort_indices].coord
    except Exception as e:
        return None

# ==========================================
# 3D Assembly (Kabsch & Stitching) Functions
# ==========================================
def kabsch_umeyama(A, B):
    """ Kabsch algorithm to align chunk B onto chunk A """
    centroid_A = np.mean(A, axis=0)
    centroid_B = np.mean(B, axis=0)
    AA = A - centroid_A
    BB = B - centroid_B
    H = BB.T @ AA
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    t = centroid_A - R @ centroid_B
    return R, t

def stitch_chunks(chunks_coords, overlaps):
    """ Overlap領域で各チャンクの座標を結合し、線形補間でブレンドする """
    final_coords = chunks_coords[0].copy()

    for i in range(1, len(chunks_coords)):
        curr_coords = chunks_coords[i]
        overlap = overlaps[i-1]

        A = final_coords[-overlap:]  # 前のチャンクの後方部分
        B = curr_coords[:overlap]    # 今のチャンクの前方部分

        # Kabschで回転行列と並進ベクトルを計算
        R, t = kabsch_umeyama(A, B)

        # チャンク全体に変換を適用
        curr_coords_transformed = (curr_coords @ R.T) + t

        # Overlap部分を線形補間（スムージング）
        weights = np.linspace(1, 0, overlap).reshape(-1, 1)
        blended_overlap = A * weights + curr_coords_transformed[:overlap] * (1 - weights)

        # 最終座標を更新（Overlap部分を上書きし、残りを追加）
        final_coords[-overlap:] = blended_overlap
        final_coords = np.vstack([final_coords, curr_coords_transformed[overlap:]])

    return final_coords

# ==========================================
# Modified runner processing
# ==========================================
def run_ptx(target_id, sequence, configs, template_idx, runner, chunk_span=None):
    temp_dir = f"./{configs.dump_dir}/input"
    os.makedirs(temp_dir, exist_ok=True)

    input_json = [{"sequences": [{"rnaSequence": {"sequence": sequence, "count": 1}}], "name": target_id}]
    input_json_path = os.path.join(temp_dir, f"{target_id}_input.json")
    with open(input_json_path, "w") as f:
        json.dump(input_json, f, indent=4)

    configs.input_json_path = input_json_path
    configs.template_idx = int(template_idx)
    
    # If target_id contains "_chk", pass it as chunk_target_id to trigger mapping in infer_predict
    if "_chk" in target_id:
        configs.chunk_target_id = target_id
        configs.chunk_span = chunk_span # Tuple (start, end)
    else:
        configs.chunk_target_id = None
        configs.chunk_span = None

    infer_predict(runner, configs)

    cif_file_path = f"{configs.dump_dir}/{target_id}/seed_42/predictions/{target_id}_sample_0.cif"
    coord = extract_c1_coordinates(cif_file_path)

    if coord is None:
        coord = np.zeros((len(sequence), 3), dtype=np.float32)
    elif coord.shape[0] < len(sequence):
        pad = np.zeros((len(sequence) - coord.shape[0], 3), dtype=np.float32)
        coord = np.concatenate([coord, pad], axis=0)

    return coord


def run() -> None:
    configs = {**configs_base, **{"data": data_configs}, **inference_configs}
    configs = parse_configs(configs=configs, arg_str=parse_sys_args(), fill_required_with_null=True)
    valid_df = pd.read_csv(configs.sequences_csv)
    runner = InferenceRunner(configs)
    solution = make_dummy_solution(valid_df)

    chunk_size = 500 # configs.max_len
    overlap_size = 100 # オーバーラップさせる塩基数

    for idx, row in valid_df.iterrows():
        target_id = row.target_id
        sequence = row.sequence
        L = len(sequence)
        print(f"\n -> Processing {target_id} (Length: {L})")

        if L < 1000:
            # 短い配列はそのまま予測
            for template_idx in range(2):
                coord = run_ptx(target_id, sequence, configs, template_idx, runner, chunk_span=(0, L))
                solution[target_id].coord.append(coord)
        else:
            # 長い配列はチャンクに分割
            print(f"    Sequence exceeds max_len ({chunk_size}), triggering chunked prediction...")
            starts, ends = [], []
            start = 0
            while start < L:
                end = min(start + chunk_size, L)
                starts.append(start)
                ends.append(end)
                if end == L: break
                start = end - overlap_size

            for template_idx in range(1):
                chunk_coords = []
                for i in range(len(starts)):
                    c_seq = sequence[starts[i]:ends[i]]
                    c_id = f"{target_id}_chk{i}"
                    print(f"    -> Predicting chunk {i+1}/{len(starts)} ({starts[i]} to {ends[i]})")

                    # 各チャンクを推論                  
                    coord = run_ptx(c_id, c_seq, configs, template_idx, runner, chunk_span=(starts[i], ends[i]))
                    chunk_coords.append(coord)               

                # 重なり合う部分（overlap）のサイズを計算して結合
                overlaps = [ends[i-1] - starts[i] for i in range(1, len(starts))]
                final_coord = stitch_chunks(chunk_coords, overlaps)
                solution[target_id].coord.append(final_coord)

    print('\n\n -> Inference done! Saving to rnapro_submission.csv')
    submit_df = solution_to_submit_df(solution)
    submit_df = submit_df.fillna(0.0)
    submit_df.to_csv("/kaggle/working/rnapro_submission.csv", index=False)

if __name__ == "__main__":
    run()

In [ ]:
%%writefile rnapro_inference_kaggle.sh

export LAYERNORM_TYPE=torch # fast_layernorm, torch


# Inference parameters (RNAPro)
SEED=42
N_SAMPLE=1
N_STEP=200
N_CYCLE=10

# Paths
DUMP_DIR="../output"
# Set a valid checkpoint file path below
CHECKPOINT_PATH="../rnapro-private-best-500m.ckpt"

# Template/MSA settings
TEMPLATE_DATA="./release_data/kaggle/templates.pt"
# Note: template_idx supports 5 choices and maps to top-k:
# 0->top1, 1->top2, 2->top3, 3->top4, 4->top5
TEMPLATE_IDX=0
RNA_MSA_DIR="/kaggle/input/stanford-rna-3d-folding-2/MSA"

# SEQUENCES_CSV="/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv"
SEQUENCES_CSV="/kaggle/working/sample_sequences.csv"

# RibonanzaNet2 path (keep as-is per request)
RIBONANZA_PATH="/kaggle/input/models/shujun717/ribonanzanet2/pytorch/alpha/1"

# Model selection: keep to an existing key to align defaults (N_step=200, N_cycle=10)
MODEL_NAME="rnapro_base"
mkdir -p "${DUMP_DIR}"

python3 runner/inference.py \
    --model_name "${MODEL_NAME}" \
    --seeds ${SEED} \
    --dump_dir "${DUMP_DIR}" \
    --load_checkpoint_path "${CHECKPOINT_PATH}" \
    --use_msa true \
    --use_template "ca_precomputed" \
    --model.use_template "ca_precomputed" \
    --model.use_RibonanzaNet2 true \
    --model.template_embedder.n_blocks 2 \
    --model.ribonanza_net_path "${RIBONANZA_PATH}" \
    --template_data "${TEMPLATE_DATA}" \
    --template_idx ${TEMPLATE_IDX} \
    --rna_msa_dir "${RNA_MSA_DIR}" \
    --model.N_cycle ${N_CYCLE} \
    --sample_diffusion.N_sample ${N_SAMPLE} \
    --sample_diffusion.N_step ${N_STEP} \
    --load_strict true \
    --num_workers 0 \
    --triangle_attention "torch" \
    --triangle_multiplicative "torch" \
    --sequences_csv "${SEQUENCES_CSV}" \
    --max_len 1000


# --triangle_attention supports 'triattention', 'cuequivariance', 'deepspeed', 'torch'
# --triangle_multiplicative supports 'cuequivariance', 'torch'
# --max_len 1000: Sequences longer than max_len will be skipped to avoid oom

In [ ]:
!bash ./rnapro_inference_kaggle.sh

In [ ]:
!mv submission.csv ..

In [ ]:
pred_rnapro = pd.read_csv("/kaggle/working/rnapro_submission.csv")
pred_rnapro

# Boltz2

In [ ]:
!pip install /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl

In [ ]:
!cp -r /kaggle/input/datasets/lbugnon/boltz-src-minimal ./
# Install boltz from src (some weird bug in kaggle notebooks with fairscale so it was removed)
!pip install --no-index --no-build-isolation -e ./boltz-src-minimal

In [ ]:
!pip install --no-index /kaggle/input/datasets/youhanlee/boltz-dependencies/mashumaro-3.14-py3-none-any.whl --no-deps
!pip install --no-index /kaggle/input/datasets/youhanlee/boltz-dependencies/ihm-2.2-py3-none-any.whl --no-deps
!pip install --no-index /kaggle/input/datasets/youhanlee/boltz-dependencies/modelcif-1.3-py3-none-any.whl --no-deps

In [ ]:
# need to create tar so botlz do not try to download, also clean the trash dataset creation does
!mkdir boltz_cache
!cp -r /kaggle/input/datasets/lbugnon/boltz2 boltz_cache
!mv boltz_cache/boltz2/mols/mols/* boltz_cache/boltz2/mols/
!rm -r boltz_cache/boltz2/mols/mols/
!tar -cf boltz_cache/boltz2/mols.tar boltz_cache/boltz2/mols

In [ ]:
import pandas as pd

MAX_LENGTH = 1000

sequences = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")
print(sequences.shape)

sequences["sequence_len"] = sequences["sequence"].str.len()

sequences = sequences[sequences['sequence_len'] < 1000]
sequences.reset_index(drop=True, inplace=True)

# mask = sequences["sequence_len"] >= MAX_LENGTH
# sequences.loc[mask, "sequence"] = (
#     sequences.loc[mask, "sequence"].str.slice(0, MAX_LENGTH)
# )

# # 長さを再計算
# sequences["sequence_len"] = sequences["sequence"].str.len()

# # sequences = sequences[sequences['sequence_len'] < 100]
# sequences = sequences[sequences['sequence_len'] < 1000]
print(sequences.shape)

print(max(sequences["sequence_len"].values))

In [ ]:
# boltz２用のyaml作成
import yaml
import os
from pathlib import Path

def create_boltz_yaml(row, output_dir):

    data = {
        "id": str(row["target_id"]),
        "sequences": [
            {
                "rna": {
                    "id": str(row["stoichiometry"]).split(":")[0],
                    "sequence": str(row["sequence"])
                }
            }
        ]
    }

    # --- リガンド処理 ---
    if pd.notna(row["ligand_SMILES"]):

        smiles_list = str(row["ligand_SMILES"]).split(";")
        ids_list = str(row["ligand_ids"]).split(";")

        ligands = []
        for s, i in zip(smiles_list, ids_list):
            ligands.append({
                "id": i.strip(),
                "smiles": s.strip()
            })

        if ligands:
            data["ligands"] = ligands

    # --- 書き出し ---
    file_path = Path(output_dir) / f"{row['target_id']}.yaml"

    with open(file_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(
            data,
            f,
            sort_keys=False,
            allow_unicode=True
        )

    return file_path

# 実行例
output_path = "/kaggle/working/inputs"
os.makedirs(output_path, exist_ok=True)

for _, row in sequences.iterrows():
    yaml_file = create_boltz_yaml(row, output_path)
print(f"Created")

In [ ]:
# with open('/kaggle/working/inputs/9E74.yaml', encoding='utf-8')as f:
#     document = yaml.safe_load(f)
# document

In [ ]:
!rm -r /kaggle/working/boltz_repeat_0*

In [ ]:
import time
from random import shuffle
import gc
import torch
        
!mkdir inputs

use_all_chains = False
max_tokens = 1000
max_repeats = 999
pred_repeats = 1

nucleotides = {"A", "G", "C", "U"}
aminoacids = {"A", "R", "N", "D", "C", "E", "Q", "G", "H", "I", "L", "K", "M", "F", "P", "S", "T", "W", "Y", "V"}

import pytorch_lightning as pl
from functools import partial

# 1. Trainerの初期化を「改造」する
# これにより、Boltzが内部でTrainerを作るときに、強制的に logger=False にします
original_trainer_init = pl.Trainer.__init__

def patched_trainer_init(self, *args, **kwargs):
    # ロガーを無効化し、エラーの元になるロギングを物理的に遮断
    kwargs["logger"] = False
    # ついでに、進捗バー以外のログ出力も最小限にする
    kwargs["enable_checkpointing"] = False
    return original_trainer_init(self, *args, **kwargs)

# Trainerのコンストラクタを差し替え
pl.Trainer.__init__ = patched_trainer_init

for k in range(len(sequences)):
    
    t0 = time.time()
    name = sequences.iloc[k].target_id
    print(f"preparing {name} ({k+1} of {len(sequences)})")
    
    pred_seq = sequences.iloc[k].sequence
    
    # ignore super large seqs
    if len(sequences.iloc[k].sequence)>max_tokens:
        print(f"skipped {name}")
        continue

    # A for the chain to predict. Firs load all chains. This may fail if all_sequences is empty or different format
    try:
        chains, chain_ind = {}, 1
        for entry in sequences.iloc[k].all_sequences.split("\n"):
            if entry.startswith(">"):
                # get number of repeats
                repeats = min(len(entry.split("|")[1].split(",")), max_repeats)
            else:
                if entry == pred_seq:
                    chains[0] = entry, "rna"
                elif use_all_chains:
                    entry_type = "rna" if set(entry)<=nucleotides else "protein"  if set(entry) <= aminoacids else None
                    if entry_type is None:
                        continue
                    chains[chain_ind] = entry, entry_type
                    chain_ind += 1
                if use_all_chains:
                    for i in range(1, repeats):
                        chains[chain_ind] = entry, entry_type
                        chain_ind += 1
    
        # Now keep the chain 0 plus all chains randomly picked to fit max_tokens (asuming chain 0 is ok under tokens, otherwise will fail)
        filtered_chains, ntokens = {}, 0
        other_chains = [c for c in chains if c!=0]
        shuffle(other_chains)
        for c in [0] + other_chains:
            ntokens += len(chains[c][0])
            if ntokens >= max_tokens:
                break
            filtered_chains[c] = chains[c]
        chains = filtered_chains
    except:
        # use only target sequence 
        chains = {0: (pred_seq, "rna")}
        
    # save fasta, chain 0 first
    yaml_data = {
        "id": name,
        "sequences": []
    }
    
    # --- sequences ---
    for chain in sorted(chains.keys()):
        seq, seq_type = chains[chain]
        yaml_data["sequences"].append({
            seq_type: {
                "id": chr(ord("A") + chain),
                "sequence": seq
            }
        })
    
    # --- ligands（存在する場合のみ）---
    row = sequences.iloc[k]
    
    if pd.notna(row["ligand_SMILES"]):
        smiles_list = str(row["ligand_SMILES"]).split(";")
        ids_list = str(row["ligand_ids"]).split(";")
    
        ligands = []
        for s, i in zip(smiles_list, ids_list):
            s = s.strip()
            i = i.strip()
            if s:
                ligands.append({
                    "id": i,
                    "smiles": s
                })
    
        if ligands:
            yaml_data["ligands"] = ligands
    
    # --- save ---
    with open(f"/kaggle/working/inputs/{name}.yaml", "w") as fout:
        yaml.safe_dump(yaml_data, fout, sort_keys=False)

    # --- setting ---
    boltz_args = {
        "diffusion_samples": 10,
        "diffusion_samples_affinity": 5, # 10,
        "devices": 2,
        "num_workers": 4,
        "max_parallel_samples": 2,
        "output_format": "pdb",
        "cache": "boltz_cache/boltz2/",
    }

    # --- inferring ---
    print(name, ":")
    !cat /kaggle/working/inputs/{name}.yaml
    print()
    for repeat in range(pred_repeats):
        cmd = f"""
        boltz predict /kaggle/working/inputs/{name}.yaml \
        --diffusion_samples {boltz_args['diffusion_samples']} \
        --diffusion_samples_affinity {boltz_args['diffusion_samples_affinity']} \
        --devices {boltz_args['devices']} \
        --num_workers {boltz_args['num_workers']} \
        --max_parallel_samples {boltz_args['max_parallel_samples']} \
        --output_format {boltz_args['output_format']} \
        --cache {boltz_args['cache']} \
        --out_dir boltz_repeat_{repeat}
        """
        !{cmd}
        # !boltz predict /kaggle/working/inputs/{name}.yaml --diffusion_samples 10 --diffusion_samples_affinity 10 --devices 2 --num_workers 1	--max_parallel_samples 1 --output_format pdb --cache boltz_cache/boltz2/ --out_dir boltz_repeat_{repeat} 
        gc.collect()
        torch.cuda.empty_cache()
        
    #     break
    # break

In [ ]:
# {'confidence_score': 0.659386396408081,
#  'ptm': 0.4263673722743988,
#  'iptm': 0.0,
#  'ligand_iptm': 0.0,
#  'protein_iptm': 0.0,
#  'complex_plddt': 0.7176411747932434,
#  'complex_iplddt': 0.7176411747932434,
#  'complex_pde': 0.4417988955974579,
#  'complex_ipde': 0.0,
#  'chains_ptm': {'0': 0.4263673722743988},
#  'pair_chains_iptm': {'0': {'0': 0.4263673722743988}}}

In [ ]:
# ==========================================
# 1. 座標抽出の関数
# ==========================================
import json
import gemmi
import numpy as np
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

def extract_coords(pdb_file, cif_file):
    """mmCIFファイルからC1'原子の座標を抽出"""
    try:
        structure = gemmi.read_structure(f"{pdb_file}")
        structure.make_mmcif_document().write_file(f"{cif_file}")
        mmcif_dict = MMCIF2Dict(cif_file)
        x_coords = mmcif_dict["_atom_site.Cartn_x"]
        y_coords = mmcif_dict["_atom_site.Cartn_y"]
        z_coords = mmcif_dict["_atom_site.Cartn_z"]
        atom_names = mmcif_dict["_atom_site.label_atom_id"]

        c1_coords = []
        for i, atom in enumerate(atom_names):
            if atom == "C1'":
                c1_coords.append([float(x_coords[i]), float(y_coords[i]), float(z_coords[i])])
        return np.array(c1_coords)
    except Exception as e:
        print(f"Error parsing {cif_file}: {e}")
        return None

# ==========================================
# 2. モデル選択のロジック
# ==========================================

def get_top_n_predictions(diffusion_results_dir: Path, file_prefix: str, n: int = 3):
    """10個の結果からスコア上位n個の情報を取得"""
    results = []
    for i in range(10):
        json_path = diffusion_results_dir / file_prefix / f"confidence_{file_prefix}_model_{i}.json"
        # print(json_path)
        if not json_path.exists():
            continue
        
        with open(json_path, 'r') as f:
            data = json.load(f)

        results.append({
            "idx": i,
            "plddt": data.get("complex_plddt", -float("inf")),
            "conf": data.get("confidence_score", -float("inf"))
        })
    
    # スコア順にソートして上位n個を昇順で返す
    results.sort(key=lambda x: (x["plddt"], x["conf"]), reverse=True)
    # top_n = results[:n]
    # top_n.sort(key=lambda x: (x["plddt"], x["conf"]), reverse=False)
    return results[:n] # top_n

In [ ]:
# ==========================================
# 3. メイン処理
# ==========================================

def main():
    df_test = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")
    submission = pd.read_csv('/kaggle/input/stanford-rna-3d-folding-2/sample_submission.csv')

    # すべての座標カラムをfloat型に初期化
    coord_cols = ['x_1', 'y_1', 'z_1', 
                  'x_2', 'y_2', 'z_2',
                  'x_3', 'y_3', 'z_3',
                  'x_4', 'y_4', 'z_4',
                  'x_5', 'y_5', 'z_5',]
    submission[coord_cols] = submission[coord_cols].astype(float)

    for target_id, sequence in zip(df_test["target_id"], df_test["sequence"]):
        print(f'--- Processing {target_id} ---')
        diffusion_results_dir = Path(f"/kaggle/working/RNAPro/boltz_repeat_0/boltz_results_{target_id}/predictions/")
        # diffusion_results_dir = Path(f"/kaggle/working/boltz_repeat_0/boltz_results_{target_id}/predictions/")

        seq_len = len(sequence)
        top_models = get_top_n_predictions(diffusion_results_dir, target_id, n=5)
        
        # ターゲットに該当する行のマスク
        target_mask = submission['ID'].apply(lambda x: x.startswith(f"{target_id}_"))
        expected_count = target_mask.sum()

        # 各モデル（1〜5番目）の座標を格納
        for rank, model_info in enumerate(top_models, start=1):
            m_idx = model_info["idx"]
            cif_file = diffusion_results_dir / target_id / f"{target_id}_model_{m_idx}.cif"
            pdb_file = diffusion_results_dir / target_id / f"{target_id}_model_{m_idx}.pdb"
            coords = extract_coords(pdb_file, cif_file)
            
            if coords is not None:
                # 座標数の調整（足りなければ0埋め、多ければカット）
                if len(coords) < expected_count:
                    coords = np.vstack([coords, np.zeros((expected_count - len(coords), 3))])
                else:
                    coords = coords[:expected_count]
                
                # 対応する列（x_N, y_N, z_N）に代入
                cols = [f'x_{rank}', f'y_{rank}', f'z_{rank}']
                submission.loc[target_mask, cols] = coords
                print(f"  Rank {rank} (Model {m_idx}) loaded.")
            else:
                print(f"  Warning: Rank {rank} coordinates not found.")

        # モデルが3つ未満の場合や、1000塩基超の場合の処理は、
        # 初期値（0.0）のままになるか、必要に応じてここでゼロ埋めを明示します。

    # 保存
    output_path = '/kaggle/working/boltz_submission.csv'
    submission.to_csv(output_path, index=False)
    print(f"\nFinal submission saved to {output_path}")

if __name__ == "__main__":
    main()

In [ ]:
boltz_pred = pd.read_csv("/kaggle/working/boltz_submission.csv")
boltz_pred
# 信頼度高い順

In [ ]:
boltz_pred.shape

# 結合

In [ ]:
# TBM、Boltzは信頼度が高いものを1, 低いものを5

In [ ]:
# test_seq[["target_id", "sequence_len"]]

In [ ]:
# sequences boltz_pred pred_tbm

In [ ]:
# boltz_pred = pd.read_csv('/kaggle/input/stanford-rna-3d-folding-2/sample_submission.csv')
# pred_tbm = pd.read_csv('/kaggle/input/stanford-rna-3d-folding-2/sample_submission.csv')
# test_seqs = pd.read_csv('/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv')
# sequences = pd.read_csv('/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv')
# pred_rnapro = pd.read_csv('/kaggle/input/stanford-rna-3d-folding-2/sample_submission.csv')
# drfold_pred = pd.read_csv('/kaggle/input/stanford-rna-3d-folding-2/sample_submission.csv')
# protenix_pred = pd.read_csv('/kaggle/input/stanford-rna-3d-folding-2/sample_submission.csv')
# boltz_pred.head()

In [ ]:
pred_tbm_ = pred_tbm.copy()
boltz_pred_ = boltz_pred.copy()
pred_rnapro_ = pred_rnapro.copy()
drfold_pred_ = drfold_pred.copy()
protenix_pred_ = protenix_pred.copy()

In [ ]:
# ① pred_tbm側に target_id 列を作る（_より前を抽出）
pred_tbm_["target_id"] = pred_tbm_["ID"].str.split("_").str[0]

# ② test_seq から必要な列だけ抽出
test_seqs["sequence_len"] = test_seqs["sequence"].apply(lambda x : len(x))
seq_len_df = test_seqs[["target_id", "sequence_len"]]

# ③ merge
pred_tbm_ = pred_tbm_.merge(seq_len_df, on="target_id", how="left")

# 4 boltz_pred側にtarget_id 列を作る（_より前を抽出）
boltz_pred_["target_id"] = boltz_pred_["ID"].str.split("_").str[0]

# 5 seaquences から必要な列だけ抽出
# sequences["sequence_len"] = sequences["sequence"].apply(lambda x : len(x))
seq_len_df = sequences[["target_id", "sequence_len"]]

# 6 boltz_predにmerge
boltz_pred_ = boltz_pred_.merge(seq_len_df, on="target_id", how="left")

In [ ]:
# merge用にprotenix_predの準備
test_df = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")
test_df["sequence_len"] = test_df["sequence"].str.len()
seq_len_df = test_df[["target_id", "sequence_len"]]
protenix_pred_["target_id"] = protenix_pred_["ID"].str.split("_").str[0]
protenix_pred_ = protenix_pred_.merge(seq_len_df, on="target_id", how="left")

In [ ]:
# merge用にpred_rnaproの準備
# test_df = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")
# test_df["sequence_len"] = test_df["sequence"].str.len()
# seq_len_df = test_df[["target_id", "sequence_len"]]
# pred_rnapro_["target_id"] = pred_rnapro_["ID"].str.split("_").str[0]
# pred_rnapro_ = pred_rnapro_.merge(seq_len_df, on="target_id", how="left")

In [ ]:
pred_tbm_

In [ ]:
# filterd_pred_tbm = pred_tbm_[pred_tbm_['sequence_len'] >= 100]
filterd_pred_tbm = pred_tbm_[pred_tbm_['sequence_len'] >= 250]
# filterd_pred_tbm = pred_tbm_[pred_tbm_['sequence_len'] >= 350]
# filterd_boltz_pred = boltz_pred_[boltz_pred_['sequence_len'] < 100]
filterd_boltz_pred = boltz_pred_[boltz_pred_['sequence_len'] < 250]
# filterd_boltz_pred = boltz_pred_[boltz_pred_['sequence_len'] < 350]

merged_pred = pd.concat([filterd_pred_tbm, filterd_boltz_pred], axis=0)

In [ ]:
# RNAProを結合
# 将来的にはRNAProの信頼度を高いものから左に並べたい

# target_cols = ["x_3", "y_3", "z_3", "x_4", "y_4", "z_4", "x_5", "y_5", "z_5"]
# target_cols = ["x_3", "y_3", "z_3", "x_4", "y_4", "z_4"]
target_cols = ["x_1", "y_1", "z_1", "x_2", "y_2", "z_2"]
replace_cols = ["x_3", "y_3", "z_3", "x_4", "y_4", "z_4"]
# target_cols = ["x_1", "y_1", "z_1", "x_2", "y_2", "z_2"]
# replace_cols = ["x_4", "y_4", "z_4", "x_5", "y_5", "z_5"]

# 1. 一旦 ID 列をインデックスに変更（既存の ID 列名を 'ID_col' と仮定）
#    ※ 既に ID がインデックスならこの工程は不要です
merged_pred = merged_pred.set_index("ID") 

# 2. loc を使って置換（右辺に .values を忘れずに）
# filterd_pred_rnapro = pred_rnapro_[(pred_rnapro_['sequence_len'] >= 400) &  (pred_rnapro_['sequence_len'] < 1000)]
merged_pred.loc[pred_rnapro_["ID"], replace_cols] = pred_rnapro_[target_cols].values
# merged_pred.loc[pred_rnapro_["ID"], replace_cols] = filterd_pred_rnapro[target_cols].values

merged_pred.reset_index(drop=False, inplace=True)

In [ ]:
# DRfoldを結合

# target_cols = ["x_5", "y_5", "z_5"]
target_cols = ["x_1", "y_1", "z_1"]
replace_cols = ["x_5", "y_5", "z_5"]

# 1. 一旦 ID 列をインデックスに変更（既存の ID 列名を 'ID_col' と仮定）
#    ※ 既に ID がインデックスならこの工程は不要です
merged_pred = merged_pred.set_index("ID") 

# 2. loc を使って置換（右辺に .values を忘れずに）
merged_pred.loc[drfold_pred_["ID"], replace_cols] = drfold_pred_[target_cols].values

merged_pred.reset_index(drop=False, inplace=True)

In [ ]:
# Boltzを結合
# target_cols = ["x_2", "y_2", "z_2", "x_5", "y_5", "z_5"]
target_cols = ["x_1", "y_1", "z_1", "x_2", "y_2", "z_2"]
replace_cols = ["x_2", "y_2", "z_2", "x_5", "y_5", "z_5"]

# 1. 一旦 ID 列をインデックスに変更（既存の ID 列名を 'ID_col' と仮定）
#    ※ 既に ID がインデックスならこの工程は不要です
merged_pred = merged_pred.set_index("ID") 

# 2. loc を使って置換（右辺に .values を忘れずに）
# filterd_boltz_pred = boltz_pred_[(boltz_pred_['sequence_len'] >= 350) & (boltz_pred_['sequence_len'] < 1000)]
filterd_boltz_pred = boltz_pred_[(boltz_pred_['sequence_len'] >= 250) & (boltz_pred_['sequence_len'] < 1000)]
merged_pred.loc[filterd_boltz_pred["ID"], replace_cols] = filterd_boltz_pred[target_cols].values

merged_pred.reset_index(drop=False, inplace=True)

In [ ]:
protenix_pred_[target_cols]

In [ ]:
# Protenixを結合
# target_cols = ["x_4", "y_4", "z_4"]
target_cols = ["x_1", "y_1", "z_1"]
replace_cols = ["x_4", "y_4", "z_4"]
# target_cols = ["x_1", "y_1", "z_1", "x_2", "y_2", "z_2"]
# replace_cols = ["x_4", "y_4", "z_4", "x_5", "y_5", "z_5"]

# 1. 一旦 ID 列をインデックスに変更（既存の ID 列名を 'ID_col' と仮定）
#    ※ 既に ID がインデックスならこの工程は不要です
merged_pred = merged_pred.set_index("ID") 

# filterd_protenix_pred = protenix_pred_[(protenix_pred_['sequence_len'] >= 350) & (protenix_pred_['sequence_len'] < 1000)]
# filterd_protenix_pred = protenix_pred_[protenix_pred_['sequence_len'] < 350]
filterd_protenix_pred = protenix_pred_[protenix_pred_['sequence_len'] < 250]
# filterd_protenix_pred = protenix_pred_[protenix_pred_['sequence_len'] < 400]
merged_pred.loc[filterd_protenix_pred["ID"], replace_cols] = filterd_protenix_pred[target_cols].values
# merged_pred.loc[protenix_pred_["ID"], replace_cols] = protenix_pred_[target_cols].values

merged_pred.reset_index(drop=False, inplace=True)

In [ ]:
target_cols = ["x_1", "y_1", "z_1", "x_2", "y_2", "z_2"]
replace_cols = ["x_4", "y_4", "z_4", "x_5", "y_5", "z_5"]

# 1. 一旦 ID 列をインデックスに変更（既存の ID 列名を 'ID_col' と仮定）
#    ※ 既に ID がインデックスならこの工程は不要です
merged_pred = merged_pred.set_index("ID") 

filterd_protenix_pred = protenix_pred_[protenix_pred_['sequence_len'] >= 1000]
merged_pred.loc[filterd_protenix_pred["ID"], replace_cols] = filterd_protenix_pred[target_cols].values

merged_pred.reset_index(drop=False, inplace=True)

In [ ]:
# RNAProを結合

# target_cols = ["x_3", "y_3", "z_3", "x_4", "y_4", "z_4", "x_5", "y_5", "z_5"]
# target_cols = ["x_3", "y_3", "z_3", "x_4", "y_4", "z_4"]
# target_cols = ["x_1", "y_1", "z_1", "x_2", "y_2", "z_2"]
# replace_cols = ["x_3", "y_3", "z_3", "x_4", "y_4", "z_4"]
# target_cols = ["x_1", "y_1", "z_1"]
# replace_cols = ["x_3", "y_3", "z_3"]

# 1. 一旦 ID 列をインデックスに変更（既存の ID 列名を 'ID_col' と仮定）
#    ※ 既に ID がインデックスならこの工程は不要です
# merged_pred = merged_pred.set_index("ID") 

# 2. loc を使って置換（右辺に .values を忘れずに）
# filterd_pred_rnapro = pred_rnapro_[(pred_rnapro_['sequence_len'] >= 250) &  (pred_rnapro_['sequence_len'] < 400)]
# merged_pred.loc[pred_rnapro_["ID"], replace_cols] = pred_rnapro_[target_cols].values
# merged_pred.loc[pred_rnapro_["ID"], replace_cols] = filterd_pred_rnapro[target_cols].values

# merged_pred.reset_index(drop=False, inplace=True)

In [ ]:
# merged_pred.query('target_id == "9LEL"')

In [ ]:
# merged_pred.iloc[protenix_pred_["ID"]]

In [ ]:
sub = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/sample_submission.csv")["ID"]
sub = pd.merge(sub, merged_pred, how="left", on="ID").iloc[:, :18]
sub["resid"] = sub["resid"].astype(int)
sub

In [ ]:
output_path = '/kaggle/working/submission.csv'
sub.to_csv(output_path, index=False)

# Evaluation

In [ ]:
PATH = "/kaggle/input/stanford-rna-3d-folding-2/"

solution = pd.read_csv(os.path.join(PATH, 'validation_labels.csv'))
solution.head()

In [ ]:
# solution = pd.merge(pred_rnapro["ID"], solution, how='left', on='ID')

In [ ]:
# def parse_tmscore_output(output):
#     # Extract TM-score based on length of reference structure (second)
#     tm_score_match = re.findall(r'TM-score=\s+([\d.]+)', output)[1]
#     if not tm_score_match:
#         raise ValueError('No TM score found')
#     return float(tm_score_match)


# def write_target_line(
#     atom_name, atom_serial, residue_name, chain_id, residue_num, x_coord, y_coord, z_coord, occupancy=1.0, b_factor=0.0, atom_type='P'
# ) -> str:
#     """
#     Writes a single line of PDB format based on provided atom information.

#     Args:
#         atom_name (str): Name of the atom (e.g., "N", "CA").
#         atom_serial (int): Atom serial number.
#         residue_name (str): Residue name (e.g., "ALA").
#         chain_id (str): Chain identifier.
#         residue_num (int): Residue number.
#         x_coord (float): X coordinate.
#         y_coord (float): Y coordinate.
#         z_coord (float): Z coordinate.
#         occupancy (float, optional): Occupancy value (default: 1.0).
#         b_factor (float, optional): B-factor value (default: 0.0).

#     Returns:
#         str: A single line of PDB string.
#     """
#     return f'ATOM  {atom_serial:>5d}  {atom_name:<5s} {residue_name:<3s} {residue_num:>3d}    {x_coord:>8.3f}{y_coord:>8.3f}{z_coord:>8.3f}{occupancy:>6.2f}{b_factor:>6.2f}           {atom_type}\n'


# def write2pdb(df: pd.DataFrame, xyz_id: str, target_path: str) -> int:
#     resolved_cnt = 0
#     with open(target_path, 'w') as target_file:
#         for _, row in df.iterrows():
#             x_coord = row[f'x_{xyz_id}']
#             y_coord = row[f'y_{xyz_id}']
#             z_coord = row[f'z_{xyz_id}']

#             if x_coord > -1e17 and y_coord > -1e17 and z_coord > -1e17:
#                 # if True:
#                 resolved_cnt += 1
#                 target_line = write_target_line(
#                     atom_name="C1'",
#                     atom_serial=int(row['resid']),
#                     residue_name=row['resname'],
#                     chain_id='0',
#                     residue_num=int(row['resid']),
#                     x_coord=x_coord,
#                     y_coord=y_coord,
#                     z_coord=z_coord,
#                     atom_type='C',
#                 )
#                 target_file.write(target_line)
#     return resolved_cnt


# def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
#     """
#     Computes the TM-score between predicted and native RNA structures using USalign.

#     This function evaluates the structural similarity of RNA predictions to native structures
#     by computing the TM-score. It uses USalign, a structural alignment tool, to compare
#     the predicted structures with the native structures.

#     Workflow:
#     1. Copies the USalign binary to the working directory and grants execution permissions.
#     2. Extracts the `target_id` from the `ID` column of both the solution and submission DataFrames.
#     3. Iterates over each unique `target_id`, grouping the native and predicted structures.
#     4. Writes PDB files for native and predicted structures.
#     5. Runs USalign on each predicted-native pair and extracts the TM-score.
#     6. Computes the highest TM-score per target and returns aggregated results.

#     Args:
#         solution (pd.DataFrame): A DataFrame containing the native RNA structures.
#         submission (pd.DataFrame): A DataFrame containing the predicted RNA structures.
#         row_id_column_name (str): The name of the column containing unique row identifiers.

#     Returns:
#         float: the average highest TM-scores.
#     """
    
#     os.system('cp /kaggle/input/datasets/metric/usalign/USalign /kaggle/working')
#     os.system('sudo chmod u+x /kaggle/working//USalign')

#     # Extract target_id from ID (target_resid)
#     solution['target_id'] = solution['ID'].apply(lambda x: x.split('_')[0])
#     submission['target_id'] = submission['ID'].apply(lambda x: x.split('_')[0])

#     results = {}
#     # Iterate through each target_id and generate PDB files for both clean and corrupted data
#     for target_id, group_native in tqdm(solution.groupby('target_id'), desc='TM-scoring'):
#         group_predicted = submission[submission['target_id'] == target_id]
#         native_pdb = 'native.pdb'
#         predicted_pdb = 'predicted.pdb'

#         target_id_scores = []
#         for pred_cnt in range(1, 6):
#             prediction_scores = []
#             for native_cnt in range(1, 41):
#                 # Write solution PDB
#                 resolved_cnt = write2pdb(group_native, native_cnt, native_pdb)

#                 # Write predicted PDB
#                 _ = write2pdb(group_predicted, pred_cnt, predicted_pdb)

#                 if resolved_cnt > 0:
#                     command = f'/kaggle/working/USalign {predicted_pdb} {native_pdb} -atom " C1\'"'
#                     usalign_output = os.popen(command).read()
#                     prediction_scores.append(parse_tmscore_output(usalign_output))

#             target_id_scores.append(max(prediction_scores))
#         results[target_id] = max(target_id_scores)

#     return results

In [ ]:
# import os
# import re
# import subprocess
# from multiprocessing import Pool, cpu_count
# import numpy as np
# import pandas as pd
# from tqdm import tqdm


# def parse_tmscore_output(output):
#     m = re.findall(r'TM-score=\s+([\d.]+)', output)
#     if len(m) < 2:
#         return 0.0
#     return float(m[1])


# def write2pdb_fast(df, xyz_id, target_path):
#     x = df[f'x_{xyz_id}'].to_numpy()
#     y = df[f'y_{xyz_id}'].to_numpy()
#     z = df[f'z_{xyz_id}'].to_numpy()
#     resid = df['resid'].to_numpy()
#     resname = df['resname'].to_numpy()

#     mask = (x > -1e17) & (y > -1e17) & (z > -1e17)

#     if not np.any(mask):
#         return 0

#     x = x[mask]
#     y = y[mask]
#     z = z[mask]
#     resid = resid[mask]
#     resname = resname[mask]

#     with open(target_path, 'w') as f:
#         for i in range(len(x)):
#             line = (
#                 f"ATOM  {int(resid[i]):>5d}  C1'  {resname[i]:<3s} "
#                 f"{int(resid[i]):>3d}    "
#                 f"{x[i]:>8.3f}{y[i]:>8.3f}{z[i]:>8.3f}"
#                 f"{1.0:>6.2f}{0.0:>6.2f}           C\n"
#             )
#             f.write(line)

#     return len(x)


# def run_usalign(args):
#     pred_pdb, native_pdb = args
#     cmd = [
#         "/kaggle/working/USalign",
#         pred_pdb,
#         native_pdb,
#         "-atom",
#         " C1'"
#     ]
#     try:
#         result = subprocess.run(
#             cmd,
#             capture_output=True,
#             text=True,
#             timeout=10  # ← 重要
#         )
#         return parse_tmscore_output(result.stdout)
#     except subprocess.TimeoutExpired:
#         print(f"Timeout: {pred_pdb} vs {native_pdb}")
#         return 0.0


# def score(solution: pd.DataFrame, submission: pd.DataFrame):
#     os.system('cp /kaggle/input/datasets/metric/usalign/USalign /kaggle/working/')
#     os.system('chmod u+x /kaggle/working/USalign')

#     solution['target_id'] = solution['ID'].str.split('_').str[0]
#     submission['target_id'] = submission['ID'].str.split('_').str[0]

#     results = {}
#     # n_workers = cpu_count()

#     n_workers = min(4, cpu_count())

#     for target_id, group_native in tqdm(solution.groupby('target_id')):
    
#         print(f"Processing {target_id}")
    
#         group_pred = submission[submission['target_id'] == target_id]
    
#         native_paths = []
#         for n in range(1, 41):
#             path = f"native_{target_id}_{n}.pdb"
#             resolved = write2pdb_fast(group_native, n, path)
#             if resolved > 5:   # ← 重要：最低原子数制限
#                 native_paths.append(path)
    
#         pred_paths = []
#         for p in range(1, 6):
#             path = f"pred_{target_id}_{p}.pdb"
#             write2pdb_fast(group_pred, p, path)
#             pred_paths.append(path)
    
#         tasks = [(pred, native) for pred in pred_paths for native in native_paths]

#         # print(f"Target: {target_id}, tasks={len(tasks)}")
    
#         if len(tasks) == 0:
#             results[target_id] = 0.0
#             continue
    
#         with Pool(n_workers, maxtasksperchild=20) as pool:
#             scores = pool.map(run_usalign, tasks)
    
#         results[target_id] = max(scores)

#     return results


In [ ]:
import os
import re
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing


# -----------------------------
# TM-score parser
# -----------------------------
def parse_tmscore_output(output):
    m = re.findall(r"TM-score=\s+([\d.]+)", output)
    if len(m) < 2:
        return 0.0
    return float(m[1])


# -----------------------------
# Fast PDB writer
# -----------------------------
def write2pdb_fast(df, xyz_id, target_path):
    x = df[f"x_{xyz_id}"].to_numpy()
    y = df[f"y_{xyz_id}"].to_numpy()
    z = df[f"z_{xyz_id}"].to_numpy()
    resid = df["resid"].to_numpy()
    resname = df["resname"].to_numpy()

    mask = (x > -1e17) & (y > -1e17) & (z > -1e17)
    if not np.any(mask):
        return 0

    x, y, z = x[mask], y[mask], z[mask]
    resid, resname = resid[mask], resname[mask]

    with open(target_path, "w") as f:
        for i in range(len(x)):
            line = (
                f"ATOM  {i+1:5d}  C1' {resname[i]:>3s} A"
                f"{int(resid[i]):4d}    "
                f"{x[i]:8.3f}{y[i]:8.3f}{z[i]:8.3f}"
                f"{1.00:6.2f}{0.00:6.2f}           C\n"
            )
            f.write(line)

    return len(x)


# -----------------------------
# USalign runner (fast mode)
# -----------------------------
def run_usalign(pred_pdb, native_pdb):
    cmd = [
        "/kaggle/working/USalign",
        pred_pdb,
        native_pdb,
        "-atom",
        " C1'",
        "-fast",
        "-TMscore",
        "5",
    ]

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=30,   # ← 長鎖用
        )
        return parse_tmscore_output(result.stdout)
    except subprocess.TimeoutExpired:
        return 0.0


# -----------------------------
# Main score function
# -----------------------------
def score(solution: pd.DataFrame, submission: pd.DataFrame):

    os.system("cp /kaggle/input/datasets/metric/usalign/USalign /kaggle/working/")
    os.system("chmod u+x /kaggle/working/USalign")

    solution["target_id"] = solution["ID"].str.split("_").str[0]
    submission["target_id"] = submission["ID"].str.split("_").str[0]

    results = {}
    max_workers = min(8, multiprocessing.cpu_count())

    for target_id, group_native in tqdm(solution.groupby("target_id")):

        group_pred = submission[submission["target_id"] == target_id]

        # ------------------
        # write PDBs once
        # ------------------
        native_paths = []
        for n in range(1, 41):
            path = f"native_{target_id}_{n}.pdb"
            resolved = write2pdb_fast(group_native, n, path)
            if resolved > 10:
                native_paths.append(path)

        pred_paths = []
        for p in range(1, 6):
            path = f"pred_{target_id}_{p}.pdb"
            write2pdb_fast(group_pred, p, path)
            pred_paths.append(path)

        if len(native_paths) == 0:
            results[target_id] = 0.0
            continue

        tasks = [(pred, native) for pred in pred_paths for native in native_paths]

        scores = []

        # ------------------
        # ThreadPool (重要)
        # ------------------
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [
                executor.submit(run_usalign, pred, native)
                for pred, native in tasks
            ]

            for f in as_completed(futures):
                scores.append(f.result())
        # print(scores)
        results[target_id] = max(scores) if scores else 0.0

    return results


In [ ]:
# pred_tbm_ = pred_tbm.copy()
# pred_rnapro_ = pred_rnapro.copy()

In [ ]:
# pred_tbm_[["x_1", "y_1", "z_1", "x_3", "y_3", "z_3"]] = 0
# pred_tbm_ = pred_tbm_.drop("target_id", axis=1)
# pred_tbm_

In [ ]:
# pred_rnapro_

In [ ]:
# pred_rnapro_["target_id"] = pred_rnapro_["ID"].str.split("_").str[0]
# pred_rnapro_[["x_1", "y_1", "z_1", "x_2", "y_2", "z_2"]] = 0
# pred_rnapro_ = pred_rnapro_.drop("target_id", axis=1)
# pred_rnapro_
# pred_rnapro_[["x_3", "y_3", "z_3", "x_4", "y_4", "z_4", "x_5", "y_5", "z_5"]]

In [ ]:
# pred_tbm_.iloc[:, 9:] = 0
# pred_tbm_

In [ ]:
from tqdm import tqdm
import re

scores = score(solution, sub) # sub pred_tbm_ submission pred_rnapro_ pred_tbm
result = {'target_id': [],
          'length': [],
          'tm_score': []}

In [ ]:
for idx, rna in test_seqs.iterrows(): # test_seqs sequences test_df test_df.iloc[:2] df test_sequences valid_df
    
    target_id = rna.target_id
    length = len(rna.sequence)
    tm_score = scores[target_id]
    
    result['target_id'].append(target_id)
    result['length'].append(length)
    result['tm_score'].append(tm_score)

result_df = pd.DataFrame(result)
result_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_results(df, desc, axs=None):
    if axs is None:
        fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15, 5), gridspec_kw={'width_ratios': [1, 2]})
    else:
        fig = None

    df['color'] = df['length'].apply(lambda x: colors[0] if x > 400 else colors[-1])

    sns.scatterplot(data=df, x='length', y='tm_score', hue='color', palette={colors[0]: colors[0], colors[-1]: colors[-1]}, ax=axs[0], legend=False)
    axs[0].set_title('TM-score per length')
    axs[0].set_xlabel('RNA length')
    axs[0].set_ylabel('TM-score')
    axs[0].set_ylim(0., 1.)

    mean_tm = df['tm_score'].mean()
    axs[0].axhline(mean_tm, color=colors[1], linestyle='--', label=f'Mean = {mean_tm:.2f}')
    axs[0].legend()

    sns.barplot(data=df, x='target_id', y='tm_score', palette=df['color'].to_list(), ax=axs[1])
    axs[1].set_title('TM-score per target')
    axs[1].set_xlabel('Target id')
    axs[1].set_ylabel('TM-score')
    axs[1].set_ylim(0., 1.)
    axs[1].tick_params(axis='x', labelrotation=45)
    axs[1].axhline(mean_tm, color=colors[1], linestyle='--', label=f'Mean = {mean_tm:.2f}')
    axs[1].legend()

In [ ]:
colors = ['#c1121f', '#ffb703', '#003049']
print('\n----- Color -----\n')
sns.palplot(sns.color_palette(colors))

In [ ]:
plot_results(result_df, desc='Results')

In [ ]:
def log_resluts(result_df):
    long_score = result_df.loc[result_df.length > 400, 'tm_score'].mean()
    short_score = result_df.loc[result_df.length <= 400, 'tm_score'].mean()
    all_score = result_df.tm_score.mean()

    print(f'[TM-score] > 400: {long_score:.4f}')
    print(f'[TM-score] <= 400: {short_score:.4f}')
    print(f'[TM-score] all: {all_score:.4f}')

log_resluts(result_df)

In [ ]:
# [TM-score] > 400: 0.0611
# [TM-score] <= 400: 0.4631
# [TM-score] all: 0.4200